In [1]:
# ============================================================
# STEP -1: 参数解析中心 (必须在所有模块之前执行)
# ============================================================
import os

def parse_bool(val: str) -> bool:
    """统一解析布尔值：支持 1/0, true/false, yes/no, y/n"""
    if isinstance(val, bool):
        return val
    s = str(val).strip().lower()
    if s in ("1", "true", "yes", "y"):  return True
    if s in ("0", "false", "no", "n"):  return False
    raise ValueError(f"无法解析布尔值: {repr(val)}")

AUTO = os.getenv("RUN_AUTO", "0") == "1"

# ---------- 参数解析（所有参数在此一次性确定） ----------
if not AUTO:
    print("=" * 60)
    print("STEP -1: 交互式参数设置")
    print("=" * 60)
    TICKER       = input("Ticker 符号 (默认 ^HSI): ").strip() or "^HSI"
    START_DATE   = input("起始日期 (留空=20年前): ").strip()
    END_DATE     = input("结束日期 (留空=今天): ").strip()
    INCLUDE_TODAY= input("包含今日? (Y/N, 默认 Y): ").strip().upper() or "Y"
    RUN_MODE     = input("运行模式 TRAIN/INFERENCE (默认 INFERENCE): ").strip().upper() or "INFERENCE"
    RUN_FS       = parse_bool(input("特征选择? (1=yes, 0=no, 默认 0): ").strip() or "0")
    RUN_HPO      = parse_bool(input("超参优化? (1=yes, 0=no, 默认 0): ").strip() or "0")
    VOL_STRATEGY = input("成交量填充策略 mean/manual/drop (默认 mean): ").strip() or "mean"
else:
    TICKER       = os.getenv("TICKER", "^HSI")
    START_DATE   = os.getenv("START_DATE", "")
    END_DATE     = os.getenv("END_DATE", "")
    INCLUDE_TODAY= os.getenv("INCLUDE_TODAY", "Y")
    RUN_MODE     = os.getenv("RUN_MODE", "INFERENCE")
    RUN_FS       = parse_bool(os.getenv("RUN_FS", "0"))
    RUN_HPO      = parse_bool(os.getenv("RUN_HPO", "0"))
    VOL_STRATEGY = os.getenv("VOL_STRATEGY", "mean")

print(f"\n✅ 参数解析完成:")
print(f"   TICKER={TICKER}, RUN_MODE={RUN_MODE}")
print(f"   RUN_FS={RUN_FS} (type={type(RUN_FS).__name__})")
print(f"   RUN_HPO={RUN_HPO} (type={type(RUN_HPO).__name__})")
print(f"   START_DATE={START_DATE or '默认(20年前)'}, END_DATE={END_DATE or '默认(今日)'}")
print("=" * 60)

# ==================== 标准库导入 ====================
import json
import logging
import os
import pickle
import re
import shutil
import math
import gc
import warnings
import random
from collections import OrderedDict
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Literal
from sklearn.feature_selection import mutual_info_regression
from multiprocessing import cpu_count

# ==================== 第三方库导入 ====================
import matplotlib
# matplotlib.use('Agg')  # 已注释：改为直接显示图像
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import pandas as pd
from pandas.tseries.offsets import BDay
import exchange_calendars as ec
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import talib
# TensorFlow 导入已注释 - Python 3.14 不支持 TensorFlow，且代码实际使用 PyTorch
# from tensorflow.keras.layers import Dense, Dropout, LSTM
# from tensorflow.keras.models import Sequential
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset,Sampler
import yfinance as yf

# ==================== Jupyter 魔法命令 ====================
# 在Jupyter notebook中显示图像
%matplotlib inline

# ==================== 警告过滤 ====================
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ==================== 统一交易日函数（Step6/8/9共用） ====================
def _parse_alltick_calendar_payload(payload) -> list[pd.Timestamp]:
    rows = None
    if isinstance(payload, dict):
        for k in ["data", "dates", "calendar", "result", "items"]:
            if isinstance(payload.get(k), list):
                rows = payload[k]
                break
    elif isinstance(payload, list):
        rows = payload
    if not rows:
        return []

    out = []
    for r in rows:
        d = None
        if isinstance(r, dict):
            d = r.get("date") or r.get("trading_date") or r.get("day") or r.get("time") or r.get("timestamp")
        else:
            d = r
        if d is None:
            continue
        try:
            ts = pd.to_datetime(d, unit="s", errors="coerce") if str(d).isdigit() and len(str(d)) >= 10 else pd.to_datetime(d, errors="coerce")
            if pd.isna(ts):
                continue
            out.append(pd.Timestamp(ts).tz_localize(None).normalize())
        except Exception:
            continue

    if not out:
        return []
    idx = pd.DatetimeIndex(out).unique().sort_values()
    return [pd.Timestamp(x).normalize() for x in idx]


def _fetch_alltick_trading_calendar(ticker: str, anchor_date: pd.Timestamp, max_h: int) -> list[pd.Timestamp]:
    cal_url = os.getenv("ALLTICK_CALENDAR_API_URL", "").strip()
    api_key = os.getenv("ALLTICK_API_KEY", "").strip()
    if not cal_url or not api_key:
        return []

    symbol = ticker.replace("^", "")
    if '_alltick_symbol' in globals():
        try:
            symbol = _alltick_symbol(ticker)
        except Exception:
            pass

    start = (pd.Timestamp(anchor_date).normalize() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    end = (pd.Timestamp(anchor_date).normalize() + pd.Timedelta(days=max(45, max_h * 8))).strftime("%Y-%m-%d")

    params = {"symbol": symbol, "start": start, "end": end, "api_key": api_key}

    try:
        import requests
        resp = requests.get(cal_url, params=params, timeout=20)
        resp.raise_for_status()
        payload = resp.json()
        dates = _parse_alltick_calendar_payload(payload)
        dates = [d for d in dates if d > pd.Timestamp(anchor_date).normalize()]
        return dates
    except Exception as e:
        print(f"⚠️ [{ticker}] AllTick 交易日日历请求失败: {e}")
        return []



def _ticker_to_calendar_code(ticker: str) -> Optional[str]:
    ticker = ticker.upper()
    if ticker in ["HSI", "HST", "HK", "HKEX"]:
        return "XHKG"
    elif ticker in ["SPX", "GSPC", "US500"]:
        return "XNYS"
    elif ticker in ["GOLD", "GC", "GC_F", "COMEX_GC", "GC=F"]:
        return "CMES"
    elif ticker in ["DXY", "USD", "EUR", "JPY", "GBP"]:
        return "XNYS"
    return None


def _fetch_exchange_calendar(ticker: str, anchor_date: pd.Timestamp, max_h: int) -> list[pd.Timestamp]:
    ticker_norm = ticker.replace("^", "").upper()
    cal_code = _ticker_to_calendar_code(ticker_norm)
    if cal_code is None:
        return []

    try:
        cal = ec.get_calendar(cal_code)
        schedule = cal.schedule
        anchor = pd.Timestamp(anchor_date).normalize()
        future_sessions = schedule.index[schedule.index > anchor]
        if len(future_sessions) >= max_h:
            return [pd.Timestamp(x).normalize() for x in future_sessions]
    except Exception as e:
        print(f"⚠️ [{ticker}] exchange_calendars 查询失败: {e}")
    return []


def get_future_trading_dates(ticker: str, anchor_date: pd.Timestamp, horizons: list[int], step6_dates=None) -> list[pd.Timestamp]:
    max_h = int(max(horizons))
    anchor = pd.Timestamp(anchor_date).normalize()

    if step6_dates is not None:
        try:
            fd = pd.DatetimeIndex(pd.to_datetime(list(step6_dates), errors='coerce')).dropna()
            if getattr(fd, 'tz', None) is not None:
                fd = fd.tz_localize(None)
            fd = fd.normalize().unique().sort_values()
            fd = fd[fd > anchor]
            if len(fd) >= max_h:
                print(f"✅ 交易日来源: Step6 future_dates (n={len(fd)})")
                return [pd.Timestamp(fd[int(h)-1]).normalize() for h in horizons]
        except Exception:
            pass

    alltk_dates = _fetch_alltick_trading_calendar(ticker, anchor, max_h)
    if len(alltk_dates) >= max_h:
        print(f"✅ 交易日来源: AllTick calendar (n={len(alltk_dates)})")
        return [pd.Timestamp(alltk_dates[int(h)-1]).normalize() for h in horizons]

    # Fallback 2: exchange_calendars
    ec_dates = _fetch_exchange_calendar(ticker, anchor, max_h)
    if len(ec_dates) >= max_h:
        print(f"✅ 交易日来源: exchange_calendars (n={len(ec_dates)})")
        return [pd.Timestamp(ec_dates[int(h)-1]).normalize() for h in horizons]

    raise RuntimeError("❌ 无法构建未来交易日。请确认 ALLTICK_CALENDAR_API_URL 与 ALLTICK_API_KEY。")


def _trading_day_offset_from_index(index: pd.DatetimeIndex, anchor_date: pd.Timestamp, offset: int) -> pd.Timestamp:
    idx = pd.DatetimeIndex(index)
    if getattr(idx, 'tz', None) is not None:
        idx = idx.tz_localize(None)
    idx = idx.normalize().unique().sort_values()
    anchor = pd.Timestamp(anchor_date).normalize()

    pos = idx.searchsorted(anchor, side='right') - 1
    if pos < 0:
        pos = 0
    tgt = max(0, min(len(idx) - 1, pos + offset))
    return pd.Timestamp(idx[tgt]).normalize()



# Ticker from environment (set by papermill or workflow)
TICKER = os.getenv("TICKER", "^HSI")


STEP -1: 交互式参数设置

✅ 参数解析完成:
   TICKER=^HSI, RUN_MODE=INFERENCE
   RUN_FS=False (type=bool)
   RUN_HPO=False (type=bool)
   START_DATE=默认(20年前), END_DATE=2026-06-02


In [2]:
# ============================================================
# 全局配置中心 (GlobalConfig) - 单例模式（终极清洗版）
# ============================================================
import re
import json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any, Literal

class GlobalConfig:
    """
    全局配置中心 - 单例模式

    整合所有配置项：
    1. 环境与路径：时间戳隔离 (run_HHMMSS)、数据模型全生命周期闭环
    2. 特征工程参数：相关性阈值、分位数截断、序列长度
    3. 模型超参数：双塔 LSTM 所有设置
    4. 业务逻辑权重：多时间间隔权重、损失加权系数
    5. 数据容错策略：成交量缺失处理
    """

    _instance: Optional['GlobalConfig'] = None
    _initialized: bool = False

    def __new__(cls, ticker_symbol: str = "^HSI") -> 'GlobalConfig':
        """单例模式实现"""
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._pending_ticker = ticker_symbol
        return cls._instance

    def _normalize_ticker(self, ticker: str) -> str:
        """规范化Ticker符号（移除特殊字符，用于路径生成）"""
        return re.sub(r'[^\w]', '_', ticker.replace('^', ''))

    def _update_paths_based_on_ticker(self) -> None:
        """根据当前Ticker更新根路径"""
        ticker_normalized = self._normalize_ticker(self._ticker_symbol)
        self._save_dir = self._base_dir / "model_artifacts" / ticker_normalized
        self._logs_dir = self._save_dir / "logs"
        self._save_dir.mkdir(parents=True, exist_ok=True)
        self._logs_dir.mkdir(parents=True, exist_ok=True)
        
        # 【核心修复】：切换标的时，必须清空当天的 Run Dir，强制下一次获取时重新生成
        self._today_run_dir = None 

    def __init__(self, ticker_symbol: str = "^HSI"):
        if GlobalConfig._initialized:
            return

        if hasattr(self, '_pending_ticker'):
            ticker_symbol = self._pending_ticker
            delattr(self, '_pending_ticker')

        # ==================== 1. 环境与路径配置 ====================
        self._ticker_symbol: str = ticker_symbol
        self._device: Optional[torch.device] = None  
        self._base_dir: Path = Path(".")
        self._today_run_dir: Optional[Path] = None  
        self._update_paths_based_on_ticker()

        # ==================== 11. 数据容错策略配置 ====================
        self._volume_fill_strategy: Literal["mean", "manual", "drop"] = "manual"  
        self._manual_volume_value: Optional[float] = None  

        # ==================== 12. 随机森林和特征筛选 ====================
        self._rf_n_estimators: int = 100  
        self._rf_max_depth: int = 12  
        self._pi_n_repeats: int = 5  
        self._n_jobs: int = -1  

        # ==================== 2. 特征工程参数 ====================
        self._original_feature_count: int = 141  
        self._correlation_threshold: float = 0.95  
        self._winsorization_lower: float = 0.025  
        self._winsorization_upper: float = 0.975  
        self._sequence_length: int = 30  

        # ==================== 3. 模型超参数 ====================
        self._hidden_size: int = 128  
        self._num_lstm_layers: int = 1  
        self._dropout: float = 0.4  
        self._learning_rate: float = 0.00016322559945825666  
        self._weight_decay: float = 0.0008  
        self._batch_size: int = 64  
        self._epochs: int = 200  
        self._patience: int = 15  
        self._huber_delta: float = 1.5  

        self._attention_time_scales: Dict[str, Tuple[int, int]] = {
            'short': (1, 5),    
            'mid': (6, 15),     
            'long': (16, 30)    
        }

        # ==================== 4. 预测时间间隔配置 ====================
        self._prediction_horizons: List[int] = [1, 5, 10, 15, 20, 25, 30]  
        self._num_horizons: int = len(self._prediction_horizons)  

        # ==================== 5. 业务逻辑权重配置 ====================
        self._loss_weight_returns: float = 0.1  
        self._loss_weight_volatility: float = 0.3  
        self._loss_weight_direction: float = 0.6  

        self._horizon_weight_1d: float = 0.85  
        self._horizon_weights: List[float] = self._compute_horizon_weights()

        self._composite_weight_returns: float = 0.2  
        self._composite_weight_volatility: float = 0.3  
        self._composite_weight_direction: float = 0.5  

        self._sigmoid_scale: float = 30.0  

        # ==================== 6. 优化器配置 ====================
        self._scheduler_patience: int = 8  
        self._scheduler_factor: float = 0.7  
        self._grad_clip_norm: float = 1.0  

        # ==================== 7. 超参数优化配置 ====================
        self._optimization_trials: int = 50  
        self._optimization_timeout: Optional[int] = None  
        self._optimization_epochs: int = 100  
        self._optimization_patience: int = 15  

        # ==================== 8. 数据划分配置 ====================
        self._train_test_split_ratio: float = 0.8  
        self._val_split_ratio: float = 0.2  

        # ==================== 9. 波动率变换配置 ====================
        self._volatility_scaling_method: str = 'log_standard'  
        self._volatility_col_idx: int = 0  

        # ==================== 10. 执行状态锁 ====================
        self._status: Dict[str, bool] = {
            'data': False,      
            'feature': False,   
            'model': False      
        }

        # ==================== 13. 预测模式 ====================
        self._forecast_mode: Literal["Standard", "LongTerm"] = "Standard"  

        GlobalConfig._initialized = True

    # ============================================================
    # 【核心】物理隔离目录引擎
    # ============================================================
    def _get_today_run_dir(self) -> Path:
        """
        获取带有时间戳隔离的专属运行目录
        结构: model_artifacts/{TICKER}/{YYYYMMDD}/run_{HHMMSS}/
        """
        if self._today_run_dir is None:
            today = datetime.now().strftime("%Y%m%d")
            run_id = f"run_{datetime.now().strftime('%H%M%S')}"
            self._today_run_dir = self._save_dir / today / run_id
            self._today_run_dir.mkdir(parents=True, exist_ok=True) 
            print(f"🚀 本次运行环境已物理隔离并锁定于: {self._today_run_dir}")
        return self._today_run_dir

    @property
    def TODAY_RUN_DIR(self) -> Path:
        """获取今日专属运行目录"""
        return self._get_today_run_dir()

    def get_latest_model_run_dir(self) -> Path:
        """
        获取当前 ticker 的最新历史模型 run 目录（按目录名降序）。
        目录结构: model_artifacts/{TICKER}/{YYYYMMDD}/run_{HHMMSS}/
        """
        run_candidates = []
        if not self._save_dir.exists():
            raise FileNotFoundError(f"❌ 模型目录不存在: {self._save_dir}")

        for day_dir in sorted(self._save_dir.iterdir(), reverse=True):
            if not day_dir.is_dir() or not day_dir.name.isdigit():
                continue
            runs = [p for p in sorted(day_dir.iterdir(), reverse=True) if p.is_dir() and p.name.startswith("run_")]
            for run_dir in runs:
                if (run_dir / 'scaler_features.pkl').exists() and (
                    (run_dir / 'best_model.pth').exists() or (run_dir / 'model_latest.pth').exists()
                ):
                    run_candidates.append(run_dir)

        if not run_candidates:
            raise FileNotFoundError(
                f"❌ 在 {self._save_dir} 下未找到可用历史模型 run 目录（缺少 scaler_features.pkl 或模型权重）"
            )

        return run_candidates[0]

    def _compute_horizon_weights(self) -> List[float]:
        remaining_weight = 1.0 - self._horizon_weight_1d
        return [
            self._horizon_weight_1d,
            remaining_weight * 0.3,
            remaining_weight * 0.2,
            remaining_weight * 0.15,
            remaining_weight * 0.15,
            remaining_weight * 0.1,
            remaining_weight * 0.1
        ]

    # ============================================================
    # 【核心】全局路径映射（全部收束至 TODAY_RUN_DIR）
    # ============================================================
    @property
    def BASE_DIR(self) -> Path: return self._base_dir

    @property
    def SAVE_DIR(self) -> Path: return self._save_dir

    @property
    def LOGS_DIR(self) -> Path: return self._logs_dir

    @property
    def MODEL_PATH(self) -> Path: return self.TODAY_RUN_DIR / "best_model.pth"

    @property
    def SCALER_FEATURES_PATH(self) -> Path: return self.TODAY_RUN_DIR / "scaler_features.pkl"

    @property
    def SCALER_RETURNS_PATH(self) -> Path: return self.TODAY_RUN_DIR / "scaler_returns.pkl"

    @property
    def SCALER_VOLATILITY_PATH(self) -> Path: return self.TODAY_RUN_DIR / "scaler_volatility.pkl"

    @property
    def CONFIG_JSON_PATH(self) -> Path: return self.TODAY_RUN_DIR / "model_config.json"

    @property
    def FEATURE_NAMES_PATH(self) -> Path: return self.TODAY_RUN_DIR / "feature_names.json"

    @property
    def SELECTED_FEATURES_PATH(self) -> Path: return self.TODAY_RUN_DIR / "selected_features.json"

    @property
    def VOLATILITY_METADATA_PATH(self) -> Path: return self.TODAY_RUN_DIR / "volatility_transform_metadata.json"

    # ==================== 设备属性 ====================
    @property
    def device(self) -> torch.device:
        if self._device is None:
            self._device = torch.device("cpu")
        return self._device

    # ==================== 配置 Getters & Setters ====================
    @property
    def TICKER_SYMBOL(self) -> str: return self._ticker_symbol
    @TICKER_SYMBOL.setter
    def TICKER_SYMBOL(self, value: str): self.update_ticker(value)

    @property
    def ORIGINAL_FEATURE_COUNT(self) -> int: return self._original_feature_count
    @property
    def CORRELATION_THRESHOLD(self) -> float: return self._correlation_threshold
    @property
    def WINSORIZATION_LOWER(self) -> float: return self._winsorization_lower
    @property
    def WINSORIZATION_UPPER(self) -> float: return self._winsorization_upper
    @property
    def SEQUENCE_LENGTH(self) -> int: return self._sequence_length
    
    @property
    def HIDDEN_SIZE(self) -> int: return self._hidden_size
    @HIDDEN_SIZE.setter
    def HIDDEN_SIZE(self, value: int): self._hidden_size = value

    @property
    def NUM_LAYERS(self) -> int: return self._num_lstm_layers
    @NUM_LAYERS.setter
    def NUM_LAYERS(self, value: int): self._num_lstm_layers = value

    @property
    def DROPOUT(self) -> float: return self._dropout
    @DROPOUT.setter
    def DROPOUT(self, value: float): self._dropout = value

    @property
    def LEARNING_RATE(self) -> float: return self._learning_rate
    @LEARNING_RATE.setter
    def LEARNING_RATE(self, value: float): self._learning_rate = value

    @property
    def WEIGHT_DECAY(self) -> float: return self._weight_decay
    @WEIGHT_DECAY.setter
    def WEIGHT_DECAY(self, value: float): self._weight_decay = value

    @property
    def BATCH_SIZE(self) -> int: return self._batch_size
    @property
    def EPOCHS(self) -> int: return self._epochs
    @property
    def PATIENCE(self) -> int: return self._patience
    @property
    def HUBER_DELTA(self) -> float: return self._huber_delta
    @property
    def ATTENTION_TIME_SCALES(self) -> Dict[str, Tuple[int, int]]: return self._attention_time_scales.copy()
    @property
    def PREDICTION_HORIZONS(self) -> List[int]: return self._prediction_horizons.copy()
    @property
    def NUM_HORIZONS(self) -> int: return self._num_horizons

    @property
    def LOSS_WEIGHT_RETURNS(self) -> float: return self._loss_weight_returns
    @LOSS_WEIGHT_RETURNS.setter
    def LOSS_WEIGHT_RETURNS(self, value: float): self._loss_weight_returns = value

    @property
    def LOSS_WEIGHT_VOLATILITY(self) -> float: return self._loss_weight_volatility
    @LOSS_WEIGHT_VOLATILITY.setter
    def LOSS_WEIGHT_VOLATILITY(self, value: float): self._loss_weight_volatility = value

    @property
    def LOSS_WEIGHT_DIRECTION(self) -> float: return self._loss_weight_direction
    @LOSS_WEIGHT_DIRECTION.setter
    def LOSS_WEIGHT_DIRECTION(self, value: float): self._loss_weight_direction = value

    @property
    def HORIZON_WEIGHTS(self) -> List[float]: return self._horizon_weights.copy()

    @property
    def HORIZON_WEIGHT_1D(self) -> float: return self._horizon_weight_1d
    @HORIZON_WEIGHT_1D.setter
    def HORIZON_WEIGHT_1D(self, value: float):
        self._horizon_weight_1d = value
        self._horizon_weights = self._compute_horizon_weights()

    @property
    def COMPOSITE_WEIGHT_RETURNS(self) -> float: return self._composite_weight_returns
    @property
    def COMPOSITE_WEIGHT_VOLATILITY(self) -> float: return self._composite_weight_volatility
    @property
    def COMPOSITE_WEIGHT_DIRECTION(self) -> float: return self._composite_weight_direction

    @property
    def SIGMOID_SCALE(self) -> float: return self._sigmoid_scale
    @SIGMOID_SCALE.setter
    def SIGMOID_SCALE(self, value: float): self._sigmoid_scale = value

    @property
    def SCHEDULER_PATIENCE(self) -> int: return self._scheduler_patience
    @property
    def SCHEDULER_FACTOR(self) -> float: return self._scheduler_factor
    @property
    def GRAD_CLIP_NORM(self) -> float: return self._grad_clip_norm
    @property
    def OPTIMIZATION_TRIALS(self) -> int: return self._optimization_trials
    @property
    def OPTIMIZATION_TIMEOUT(self) -> Optional[int]: return self._optimization_timeout
    @property
    def OPTIMIZATION_EPOCHS(self) -> int: return self._optimization_epochs
    @property
    def OPTIMIZATION_PATIENCE(self) -> int: return self._optimization_patience
    @property
    def TRAIN_TEST_SPLIT_RATIO(self) -> float: return self._train_test_split_ratio
    @property
    def VAL_SPLIT_RATIO(self) -> float: return self._val_split_ratio
    @property
    def VOLATILITY_SCALING_METHOD(self) -> str: return self._volatility_scaling_method
    @property
    def VOLATILITY_COL_IDX(self) -> int: return self._volatility_col_idx

    @property
    def status(self) -> Dict[str, bool]: return self._status.copy()

    @property
    def VOLUME_FILL_STRATEGY(self) -> Literal["mean", "manual", "drop"]: return self._volume_fill_strategy
    @VOLUME_FILL_STRATEGY.setter
    def VOLUME_FILL_STRATEGY(self, value: Literal["mean", "manual", "drop"]):
        if value not in ["mean", "manual", "drop"]: raise ValueError("无效策略")
        self._volume_fill_strategy = value

    @property
    def MANUAL_VOLUME_VALUE(self) -> Optional[float]: return self._manual_volume_value
    @MANUAL_VOLUME_VALUE.setter
    def MANUAL_VOLUME_VALUE(self, value: Optional[float]):
        if value is not None and value < 0: raise ValueError("不能为负数")
        self._manual_volume_value = value

    @property
    def RF_N_ESTIMATORS(self) -> int: return self._rf_n_estimators
    @property
    def RF_MAX_DEPTH(self) -> int: return self._rf_max_depth
    @property
    def PI_REPEATS(self) -> int: return self._pi_n_repeats
    @property
    def N_JOBS(self) -> int: return self._n_jobs

    def set_status(self, step: str, value: bool) -> None:
        if step in self._status: self._status[step] = value
        else: raise ValueError(f"未知步骤: {step}")

    def check_dependency(self, step: str) -> None:
        dependencies = {'feature': ['data'], 'model': ['data', 'feature']}
        if step not in dependencies: return
        missing_deps = [dep for dep in dependencies[step] if not self._status.get(dep, False)]
        if missing_deps:
            raise RuntimeError(f"❌ 缺少前置依赖: {', '.join(missing_deps)}")

    def update_ticker(self, new_ticker: str) -> None:
        # 🛡️ 核心修复：如果新传入的标的和当前标的完全一样，直接 return 拦截！
        # 这样就不会触发下面的 _update_paths_based_on_ticker 去清空文件夹了
        if new_ticker == self._ticker_symbol:
            return 
            
        old_ticker = self._ticker_symbol
        self._ticker_symbol = new_ticker
        self._update_paths_based_on_ticker() 
        self.reset_status()
        print(f"✅ Ticker已更新: {old_ticker} -> {new_ticker}")
        print(f"   新基准目录: {self._save_dir}")

    def _get_safe_volume_mean(self, df: pd.DataFrame, volume_col: str) -> float:
        hist_mean = df[df[volume_col] > 0][volume_col].mean()
        return 1.0 if pd.isna(hist_mean) or hist_mean == 0 else hist_mean

    def handle_missing_volume(self, df: pd.DataFrame, volume_col: str = 'Volume') -> pd.DataFrame:
        if volume_col not in df.columns: return df
        missing_mask = df[volume_col].isna() | (df[volume_col] == 0)
        missing_count = missing_mask.sum()
        if missing_count == 0: return df

        if self._volume_fill_strategy == "manual" and self._manual_volume_value is not None:
            df.loc[missing_mask, volume_col] = self._manual_volume_value
            return df

        if self._volume_fill_strategy == "mean":
            hist_mean = self._get_safe_volume_mean(df, volume_col)
            df.loc[missing_mask, volume_col] = hist_mean

        elif self._volume_fill_strategy == "manual":
            hist_mean = self._get_safe_volume_mean(df, volume_col)
            try:
                print(f"   ⏸️  请手动输入成交量值（回车默认 {hist_mean:.0f}）：")
                manual_input = "" if os.getenv("RUN_AUTO", "0") == "1" else input().strip()
                fill_volume = float(manual_input) if manual_input else hist_mean
                self._manual_volume_value = fill_volume
            except:
                fill_volume = hist_mean
                self._manual_volume_value = fill_volume
            df.loc[missing_mask, volume_col] = fill_volume

        elif self._volume_fill_strategy == "drop":
            df = df[~missing_mask].copy()

        return df

    def reset_status(self) -> None:
        for key in self._status: self._status[key] = False

    def to_dict(self) -> Dict[str, Any]:
        return {
            'ticker_symbol': self._ticker_symbol,
            'device': str(self.device),
            'hidden_size': self._hidden_size,
            'num_lstm_layers': self._num_lstm_layers,
            'dropout': self._dropout,
            'learning_rate': self._learning_rate,
            'weight_decay': self._weight_decay,
            'loss_weight_returns': self._loss_weight_returns,
            'loss_weight_volatility': self._loss_weight_volatility,
            'loss_weight_direction': self._loss_weight_direction,
            'horizon_weight_1d': self._horizon_weight_1d,
            'volume_fill_strategy': self._volume_fill_strategy,
            'manual_volume_value': self._manual_volume_value
        }

    def save_to_json(self, filepath: Optional[Path] = None) -> None:
        filepath = filepath or self.CONFIG_JSON_PATH
        filepath.parent.mkdir(parents=True, exist_ok=True)
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(self.to_dict(), f, indent=2, ensure_ascii=False)
        print(f"✅ 配置已保存到: {filepath}")

    def load_from_json(self, filepath: Path) -> None:
        with open(filepath, 'r', encoding='utf-8') as f:
            config_dict = json.load(f)

        if 'hidden_size' in config_dict: self._hidden_size = config_dict['hidden_size']
        if 'num_lstm_layers' in config_dict: self._num_lstm_layers = config_dict['num_lstm_layers']
        if 'dropout' in config_dict: self._dropout = config_dict['dropout']
        if 'learning_rate' in config_dict: self._learning_rate = config_dict['learning_rate']
        if 'weight_decay' in config_dict: self._weight_decay = config_dict['weight_decay']
        if 'loss_weight_returns' in config_dict: self._loss_weight_returns = config_dict['loss_weight_returns']
        if 'loss_weight_volatility' in config_dict: self._loss_weight_volatility = config_dict['loss_weight_volatility']
        if 'loss_weight_direction' in config_dict: self._loss_weight_direction = config_dict['loss_weight_direction']
        if 'horizon_weight_1d' in config_dict: 
            self._horizon_weight_1d = config_dict['horizon_weight_1d']
            self._horizon_weights = self._compute_horizon_weights()

        ticker = config_dict.get('ticker_symbol')
        if ticker is not None: self.update_ticker(ticker)

        print(f"✅ 配置已从 {filepath} 加载")

# 创建全局配置实例
# cfg 初始化（必须在 STEP -1 参数解析之后）
cfg = GlobalConfig()
cfg.TICKER_SYMBOL = TICKER  # 使用 STEP -1 解析的 TICKER

print("✅ 全局配置中心已初始化 (终极物理隔离版)")
print(f"   设备: {cfg.device}")
print(f"   Ticker符号: {cfg.TICKER_SYMBOL}")
print(f"   基准目录: {cfg.SAVE_DIR}")
print(f"   当前隔离运行目录: {cfg.TODAY_RUN_DIR}")

# ============================================================
# 【P7】参数一致性断言
# ============================================================
assert cfg.TICKER_SYMBOL == TICKER, f"TICKER不一致: cfg={cfg.TICKER_SYMBOL}, TICKER={TICKER}"
assert isinstance(RUN_FS, bool), f"RUN_FS 必须是 bool，实际是 {type(RUN_FS).__name__}: {RUN_FS}"
assert isinstance(RUN_HPO, bool), f"RUN_HPO 必须是 bool，实际是 {type(RUN_HPO).__name__}: {RUN_HPO}"
assert RUN_MODE in ("TRAIN", "INFERENCE"), f"RUN_MODE 异常: {RUN_MODE}"
print(f"✅ P7 断言通过: TICKER={TICKER}, RUN_FS={RUN_FS}, RUN_HPO={RUN_HPO}, RUN_MODE={RUN_MODE}")


✅ 全局配置中心已初始化 (终极物理隔离版)
   设备: cpu
   Ticker符号: ^HSI
   基准目录: model_artifacts/HSI
🚀 本次运行环境已物理隔离并锁定于: model_artifacts/HSI/20260611/run_230437
   当前隔离运行目录: model_artifacts/HSI/20260611/run_230437
✅ P7 断言通过: TICKER=^HSI, RUN_FS=False, RUN_HPO=False, RUN_MODE=INFERENCE


In [3]:
# ============================================================
# STEP 0.5: 参数确认（引用 STEP -1 解析结果）
# ============================================================
# 注意：所有参数已在 STEP -1 中解析完毕，本 cell 仅用于确认和展示
# 禁止在此重新定义 TICKER / RUN_MODE / RUN_FS / RUN_HPO
# ============================================================

print("=" * 60)
print("STEP 0.5: 参数一致性确认")
print("=" * 60)
print(f"✅ TICKER    = {TICKER}  (from STEP -1)")
print(f"✅ RUN_MODE  = {RUN_MODE}  (from STEP -1)")
print(f"✅ RUN_FS    = {RUN_FS}  (type={type(RUN_FS).__name__}, from STEP -1)")
print(f"✅ RUN_HPO   = {RUN_HPO}  (type={type(RUN_HPO).__name__}, from STEP -1)")
print(f"✅ START_DATE= {START_DATE or '(默认20年前)'}")
print(f"✅ END_DATE  = {END_DATE or '(默认今日)'}")
print(f"✅ INCLUDE_TODAY = {INCLUDE_TODAY}")
print(f"✅ VOL_STRATEGY = {VOL_STRATEGY}")
print("=" * 60)

# P7 断言
assert isinstance(RUN_FS, bool), f"RUN_FS 类型错误: {type(RUN_FS).__name__}"
assert isinstance(RUN_HPO, bool), f"RUN_HPO 类型错误: {type(RUN_HPO).__name__}"
assert RUN_MODE in ("TRAIN", "INFERENCE"), f"RUN_MODE 值异常: {RUN_MODE}"
print("✅ 参数类型断言全部通过")


STEP 0.5: 参数一致性确认
✅ TICKER    = ^HSI  (from STEP -1)
✅ RUN_MODE  = INFERENCE  (from STEP -1)
✅ RUN_FS    = False  (type=bool, from STEP -1)
✅ RUN_HPO   = False  (type=bool, from STEP -1)
✅ START_DATE= (默认20年前)
✅ END_DATE  = 2026-06-02
✅ INCLUDE_TODAY = Y
✅ VOL_STRATEGY = mean
✅ 参数类型断言全部通过


In [4]:
# ============================================================
# STEP 0: DATA FETCHING & PREPROCESSING
# 【本地缓存 + 增量获取 + yfinance失败回退alltick + 手动补录版】
# ============================================================
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
import time
import os
import json

# ===================== Network Proxy (optional) =====================
# 如果你的网络需要代理才能访问 yfinance / AllTick，可启用如下配置
proxy = os.getenv("LSTM_HTTP_PROXY", "").strip()
if proxy:
    os.environ['HTTP_PROXY'] = proxy
    os.environ['HTTPS_PROXY'] = proxy
    os.environ['http_proxy'] = proxy
    os.environ['https_proxy'] = proxy
    print(f"🌐 Proxy enabled: {proxy}")

# ===================== 时区处理 =====================
def normalize_index(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or (hasattr(df, "empty") and df.empty):
        return pd.DataFrame()

    df = df.copy()
    try:
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index, errors='coerce')
        if hasattr(df.index, 'tz') and df.index.tz is not None:
            df.index = df.index.normalize().tz_localize(None)
        else:
            df.index = pd.to_datetime(df.index).normalize()
    except Exception:
        df.index = pd.to_datetime(df.index, errors='coerce').normalize()

    df = df[~df.index.isna()]
    df = df[~df.index.duplicated(keep='last')].sort_index()
    return df


# ===================== Jitter 防过拟合 =====================
def add_jitter(df, jitter_pct=0.0015, seed=42):
    if df is None or df.empty:
        return df

    np.random.seed(seed)
    dfj = df.copy()
    for col in ['Open', 'High', 'Low', 'Close']:
        if col in dfj.columns:
            dfj[col] = dfj[col].astype(float) * np.random.normal(1, jitter_pct, len(dfj))

    if all(c in dfj.columns for c in ['Open', 'Close', 'High', 'Low']):
        dfj['High'] = dfj[['Open', 'Close', 'High']].max(axis=1)
        dfj['Low'] = dfj[['Open', 'Close', 'Low']].min(axis=1)

    return dfj


# ===================== AllTick Fallback =====================
def _alltick_symbol(ticker: str) -> str:
    default_map = {
        "^HSI": "HSI",
        "^VIX": "VIX",
        "^GSPC": "SPX",
        "^TNX": "US10Y",
        "CNH=F": "USDCNH",
        "HKD=X": "USDHKD",
        "GC=F": "GOLD",
        "CL=F": "OIL",
        "DX-Y.NYB": "DXY",
        "000001.SS": "SSE"
    }
    custom_map_raw = os.getenv("ALLTICK_SYMBOL_MAP_JSON", "").strip()
    if custom_map_raw:
        try:
            custom_map = json.loads(custom_map_raw)
            if isinstance(custom_map, dict) and ticker in custom_map:
                return str(custom_map[ticker])
        except Exception:
            pass
    return default_map.get(ticker, ticker.replace("^", "").replace("=F", "").replace(".", ""))




def _load_alltick_config() -> dict:
    """
    优先从本地文件读取 AllTick 配置，避免每次手动 export。
    支持路径：
      1) alltick_cache/alltick_config.json
      2) .env.example 同级目录下的 alltick_config.json（兼容未来迁移）
    文件示例：
      {
        "api_url": "https://...",
        "api_key": "...",
        "auth_mode": "query",  # query or header
        "header_name": "X-API-KEY"
      }
    """
    candidates = [
        Path('alltick_cache/alltick_config.json'),
        Path('alltick_config.json')
    ]
    for cp in candidates:
        try:
            if cp.exists():
                with open(cp, 'r', encoding='utf-8') as f:
                    cfg = json.load(f)
                if isinstance(cfg, dict):
                    print(f"ℹ️ 已加载 AllTick 配置文件: {cp}")
                    return cfg
        except Exception as e:
            print(f"⚠️ AllTick 配置文件读取失败 {cp}: {e}")
    return {}


def _extract_first_list(obj):
    """递归提取首个由 dict 组成的 list，用于兼容多种返回结构。"""
    if isinstance(obj, list):
        if obj and isinstance(obj[0], dict):
            return obj
        for it in obj:
            r = _extract_first_list(it)
            if r:
                return r
        return None
    if isinstance(obj, dict):
        for v in obj.values():
            r = _extract_first_list(v)
            if r:
                return r
    return None


def _fetch_from_alltick(ticker: str, start_dt: pd.Timestamp, end_dt: pd.Timestamp) -> pd.DataFrame:
    """按 AllTick quote-b-api/kline 协议请求并解析。"""
    cfg = _load_alltick_config()
    api_url = str(cfg.get("api_url", "") or os.getenv("ALLTICK_API_URL", "")).strip()
    api_key = str(cfg.get("api_key", "") or os.getenv("ALLTICK_API_KEY", "")).strip()

    if not api_url or not api_key:
        print(f"⚠️ [{ticker}] AllTick 未配置（缺少 api_url/api_key），跳过。")
        return pd.DataFrame()

    default_code_candidates = {
        '^HSI': ['HK50', 'HSI'],
        '^GSPC': ['SPX', 'US500', 'SPX500'],
        '^VIX': ['VIX'],
        '000001.SS': ['SSE', 'SH000001', '000001.SH'],
        'CNH=F': ['USDCNH'],
        '^TNX': ['US10Y', 'TNX'],
        'GC=F': ['XAUUSD', 'GOLD'],
        'CL=F': ['USOIL', 'WTI'],
        'DX-Y.NYB': ['DXY', 'USDX'],
        'HKD=X': ['USDHKD']
    }
    user_code_map = cfg.get('code_map', {}) if isinstance(cfg.get('code_map', {}), dict) else {}
    if ticker in user_code_map:
        candidate_codes = [str(user_code_map[ticker]).strip()]
    else:
        candidate_codes = default_code_candidates.get(ticker, [_alltick_symbol(ticker)])

    import requests
    last_payload = None
    for code in candidate_codes:
        query_payload = {
            "data": {
                "code": str(code).strip(),
                "kline_type": str(cfg.get("kline_type", "8")),
                "kline_timestamp_end": str(cfg.get("kline_timestamp_end", "0")),
                "query_kline_num": str(cfg.get("query_kline_num", "240")),
                "adjust_type": str(cfg.get("adjust_type", "0"))
            }
        }
        params = {
            "token": api_key,
            "query": json.dumps(query_payload, ensure_ascii=False, separators=(",", ":"))
        }

        try:
            resp = requests.get(api_url, params=params, timeout=20)
            status = resp.status_code
            content_preview = (resp.text or "")[:500].replace("\n", " ")
            print(f"ℹ️ [{ticker}] AllTick status={status}, code={code}, url={api_url}")
            if status >= 400:
                print(f"❌ [{ticker}] AllTick 响应异常: {content_preview}")
                continue
            try:
                payload = resp.json()
            except Exception:
                print(f"❌ [{ticker}] AllTick 返回非JSON: {content_preview}")
                continue
        except Exception as e:
            print(f"❌ [{ticker}] AllTick 请求失败: {e}")
            continue

        last_payload = payload
        # debug dump
        try:
            dbg_path = Path('alltick_cache') / f"debug_{ticker.replace('^','').replace('=','_').replace('.','_')}.json"
            dbg_path.parent.mkdir(parents=True, exist_ok=True)
            with open(dbg_path, 'w', encoding='utf-8') as f:
                json.dump(payload, f, ensure_ascii=False, indent=2)
        except Exception:
            pass

        if isinstance(payload, dict) and str(payload.get('ret', '')) in ['600', '601']:
            print(f"⚠️ [{ticker}] code={code} 无效，尝试下一个候选 code...")
            continue

        rows = _extract_first_list(payload)
        if not rows:
            print(f"⚠️ [{ticker}] code={code} 返回JSON但未解析到K线列表，keys={list(payload.keys()) if isinstance(payload, dict) else 'list'}")
            continue

        out = []
        for r in rows:
            if not isinstance(r, dict):
                continue
            t = r.get("time") or r.get("timestamp") or r.get("date") or r.get("datetime") or r.get("kline_timestamp")
            o = r.get("open") or r.get("Open") or r.get("open_price")
            h = r.get("high") or r.get("High") or r.get("high_price")
            l = r.get("low") or r.get("Low") or r.get("low_price")
            c = r.get("close") or r.get("Close") or r.get("close_price")
            v = r.get("volume") or r.get("Volume") or r.get("turnover_volume") or 0

            # 常见数组键
            arr = r.get('kline') or r.get('values') or r.get('ohlcv')
            if (t is None or o is None or h is None or l is None or c is None) and isinstance(arr, (list, tuple)) and len(arr) >= 5:
                t, o, h, l, c = arr[0], arr[1], arr[2], arr[3], arr[4]
                v = arr[5] if len(arr) > 5 else 0

            if t is None or o is None or h is None or l is None or c is None:
                continue
            try:
                ts = pd.to_datetime(t, unit='s', errors='coerce') if str(t).isdigit() and len(str(t)) >= 10 else pd.to_datetime(t, errors='coerce')
                out.append({"Date": ts, "Open": float(o), "High": float(h), "Low": float(l), "Close": float(c), "Volume": float(v) if v is not None else 0.0})
            except Exception:
                continue

        if not out:
            print(f"⚠️ [{ticker}] code={code} K线字段解析失败（rows={len(rows)}）。")
            continue

        df = pd.DataFrame(out).set_index('Date')
        df = normalize_index(df)
        df = df[(df.index >= pd.to_datetime(start_dt).normalize()) & (df.index <= pd.to_datetime(end_dt).normalize())]
        keep = [cc for cc in ['Open', 'High', 'Low', 'Close', 'Volume'] if cc in df.columns]
        print(f"✅ [{ticker}] AllTick(code={code}) 解析成功: {len(df)} 行")
        return df[keep].copy()

    if last_payload is not None:
        print(f"⚠️ [{ticker}] 所有 code 候选均失败。可查看 alltick_cache/debug_*.json 诊断。")
    return pd.DataFrame()

def _manual_latest_bar(ticker: str, date_hint: pd.Timestamp) -> pd.DataFrame:
    """
    手动补录最新一根K线。直接回车可放弃。
    """
    print(f"\n📝 [{ticker}] yfinance/alltick 都未拿到增量数据。")
    print(f"   可手动补录最新K线（默认日期 {date_hint.date()}，直接回车跳过）")

    is_auto = os.getenv("RUN_AUTO", "0") == "1"
    if is_auto:
        return pd.DataFrame()
    
    if os.getenv("RUN_AUTO", "0") == "1":
        d_in = ""
    else:
        d_in = os.getenv("DATE_OVERRIDE", "")  # STEP -1 不提供则为空，用默认值
    if not d_in:
        return pd.DataFrame()

    try:
        d = pd.to_datetime(d_in).normalize()
    except Exception:
        print("⚠️ 日期格式错误，跳过手动补录")
        return pd.DataFrame()

    def _ask_num(name):
        if not is_auto:
            s = os.getenv(f"COL_{name.upper()}", "")
        else:
            s = ""
        return float(s) if s else np.nan

    o = _ask_num("Open")
    h = _ask_num("High")
    l = _ask_num("Low")
    c = _ask_num("Close")
    v = _ask_num("Volume")

    if any(pd.isna(x) for x in [o, h, l, c]):
        print("⚠️ O/H/L/C 不能为空，跳过手动补录")
        return pd.DataFrame()

    if pd.isna(v):
        v = 0.0

    row = pd.DataFrame(
        [{"Open": o, "High": max(h, o, c), "Low": min(l, o, c), "Close": c, "Volume": v}],
        index=[d]
    )
    print(f"✅ [{ticker}] 已手动补录 {d.date()}")
    return row



# ===================== 安全下载函数（缓存 + 增量 + yfinance->alltick->manual） =====================
def safe_download(ticker, start, end, cache_dir="yf_cache", allow_manual=False):
    os.makedirs(cache_dir, exist_ok=True)
    clean_ticker = str(ticker).replace('^', '').replace('=F', '').replace('.', '')
    cache_file = f"{cache_dir}/{clean_ticker}.csv"

    start_dt = pd.to_datetime(start).tz_localize(None).normalize()
    end_dt = pd.to_datetime(end).tz_localize(None).normalize()

    df = pd.DataFrame()

    # 1) 读缓存
    if os.path.exists(cache_file):
        try:
            df = pd.read_csv(cache_file, index_col=0, parse_dates=True)
            df = normalize_index(df)
        except Exception as e:
            print(f"[{ticker}] 缓存读取失败，准备重新下载: {e}")
            df = pd.DataFrame()

    # 2) 判断增量范围
    if not df.empty:
        last_date = df.index[-1]
        if last_date >= end_dt - timedelta(days=1):
            return df[(df.index >= start_dt) & (df.index <= end_dt)]
        fetch_start = last_date + timedelta(days=1)
        print(f"🔄 [{ticker}] 命中缓存，增量拉取: {fetch_start.date()} -> {end_dt.date()}")
    else:
        fetch_start = start_dt
        print(f"⬇️ [{ticker}] 无有效缓存，全量下载: {fetch_start.date()} -> {end_dt.date()}")

    new_data = pd.DataFrame()

    # 3) 先走 yfinance
    if fetch_start < end_dt:
        time.sleep(1.2)
        try:
            yf_data = yf.Ticker(ticker).history(start=fetch_start, end=end_dt + timedelta(days=1))
            if isinstance(yf_data, pd.DataFrame) and not yf_data.empty:
                new_data = normalize_index(yf_data)
                keep = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in new_data.columns]
                new_data = new_data[keep].copy()
                print(f"✅ [{ticker}] yfinance 增量成功: {len(new_data)} 行")
            else:
                print(f"⚠️ [{ticker}] yfinance 返回空，尝试 AllTick...")
        except Exception as e:
            print(f"⚠️ [{ticker}] yfinance 异常: {e}，尝试 AllTick...")

    # 4) yfinance失败 -> alltick
    if new_data.empty and fetch_start < end_dt:
        print(f"📡 [{ticker}] 调用 AllTick API（增量区间: {fetch_start.date()} -> {end_dt.date()}）...")
        alt = _fetch_from_alltick(ticker, fetch_start, end_dt)
        if not alt.empty:
            new_data = alt
            print(f"✅ [{ticker}] AllTick 增量成功: {len(new_data)} 行")
        else:
            print(f"⚠️ [{ticker}] AllTick 也未返回有效增量")

    # 5) 主标的可手动补录最新bar
    if new_data.empty and allow_manual:
        manual_row = _manual_latest_bar(ticker, end_dt - timedelta(days=1))
        if not manual_row.empty:
            new_data = manual_row

    # 6) 合并缓存 + 增量
    if not new_data.empty:
        if df.empty:
            df = new_data
        else:
            df = pd.concat([df, new_data], axis=0)
            df = normalize_index(df)
            df = df[~df.index.duplicated(keep='last')].sort_index()
        df.to_csv(cache_file)
        print(f"✅ [{ticker}] 数据已更新并写入缓存: {cache_file}")

    # 7) 返回窗口数据
    if not df.empty:
        return df[(df.index >= start_dt) & (df.index <= end_dt)]

    return pd.DataFrame()


# ===================== 1. 参数 =====================
# AUTO mode: use env vars; manual mode: use input
is_auto = os.getenv("RUN_AUTO", "0") == "1"
if is_auto:
    user_input_ticker = cfg.TICKER_SYMBOL
else:
    user_input_ticker = cfg.TICKER_SYMBOL

if is_auto:
    raw_start = os.getenv("START_DATE", "")
else:
    # 【P0修复】优先使用 STEP -1 已输入的值，不再重复询问
    raw_start = globals().get("START_DATE", "") or os.getenv("START_DATE", "")
if is_auto:
    raw_end = os.getenv("END_DATE", "")
else:
    raw_end = globals().get("END_DATE", "") or os.getenv("END_DATE", "")
if is_auto:
    raw_inc = os.getenv("INCLUDE_TODAY", "Y")
else:
    raw_inc = globals().get("INCLUDE_TODAY", "Y") or os.getenv("INCLUDE_TODAY", "Y").upper()

now = datetime.now()
include_today = raw_inc != "N"

if raw_end:
    base_end = pd.to_datetime(raw_end)
    end_date = base_end + timedelta(days=1) if include_today else base_end
else:
    end_date = now + timedelta(days=1) if include_today else now

start_date = pd.to_datetime(raw_start) if raw_start else (end_date - timedelta(days=20 * 365))

# ===================== 2. 下载主标的 =====================
df = safe_download(user_input_ticker, start_date, end_date, cache_dir="yf_cache", allow_manual=True)
if df is None or df.empty or len(df) < 100:
    raise ValueError("数据不足（少于100条），请检查网络或输入Ticker；也可手动补录最新K线。")

df = add_jitter(df)
latest_date = df.index[-1]

# ===================== 3. 盘中补丁 =====================
if include_today and latest_date is not None:
    try:
        o, h, l, c, v = df.loc[latest_date, ['Open', 'High', 'Low', 'Close', 'Volume']]
        print(f"\nToday {latest_date.date()} | O:{o:.2f} H:{h:.2f} L:{l:.2f} C:{c:.2f}")

        def inp(name, val, vol=False):
            s = f"{val:,.0f}" if vol else f"{val:.2f}"
            if is_auto:
                return val
            else:
                i = "" if os.getenv("RUN_AUTO", "0") == "1" else input(f"{name} [{s}]: ").strip().replace(',', '')
                return float(i) if i else val

        df.loc[latest_date, 'Open'] = inp("Open", o)
        df.loc[latest_date, 'High'] = inp("High", h)
        df.loc[latest_date, 'Low'] = inp("Low", l)
        df.loc[latest_date, 'Close'] = inp("Close", c)
        df.loc[latest_date, 'Volume'] = inp("Vol", v, vol=True)
    except Exception as e:
        print(f"⚠️ 盘中补丁应用跳过: {e}")

df['Log_Ret'] = np.log(df['Close'] / df['Close'].shift(1))
df['Returns'] = df['Close'].pct_change(fill_method=None).fillna(0)

# ===================== 4. 外部数据获取 =====================
print("\nLoading external assets (with cache + yfinance/alltick fallback)...")

tickers = {
    'VIX_Close': '^VIX',
    'SPX_Close': '^GSPC',
    'SSE_Close': '000001.SS',
    'USDCNH_Close': 'CNH=F',
    'US10Y_Close': '^TNX',
    'GOLD_Close': 'GC=F',
    'OIL_Close': 'CL=F',
    'DXY_Close': 'DX-Y.NYB',
    'HKD_Close': 'HKD=X',
}

for col, sym in tickers.items():
    try:
        ext = safe_download(sym, start_date, end_date, cache_dir="yf_cache", allow_manual=False)
        if ext is not None and not ext.empty and 'Close' in ext.columns:
            df = df.join(ext['Close'].rename(col).shift(1), how='left')
        else:
            print(f"⚠️ {col} 数据为空或缺少 Close 列，已跳过。")
            df[col] = np.nan
    except Exception as e:
        print(f"❌ {col} 处理失败: {e}")
        df[col] = np.nan

df = df.ffill()

if 'SPX_Close' in df.columns:
    df['SPX_Ret'] = df['SPX_Close'].pct_change(fill_method=None).fillna(0)
if 'SSE_Close' in df.columns:
    df['SSE_Ret'] = df['SSE_Close'].pct_change(fill_method=None).fillna(0)

print("\n" + "=" * 60)
print(f"数据处理完成：{len(df)} 条 | {df.index[0].date()} ~ {df.index[-1].date()}")
print(f"缺失值情况：{df.isnull().sum().sum()}")
print("数据已同步到本地 yf_cache/。")
print("=" * 60)

#如果要用alltick需要先设置环境变量
# export ALLTICK_API_URL='你的alltick日线接口URL'
# export ALLTICK_API_KEY='你的key'


# ===================== 成交量零值自动修复 =====================
def _fill_zero_volume_with_hist_mean(df, vol_col="Volume", lookback_days=30):
    """
    当 VOL_STRATEGY="mean" 时，自动用近30天历史均值填充今日/当日为零的成交量
    """
    if vol_col not in df.columns:
        return df
    
    # 获取当前配置的成交量策略
    vol_strategy = ENV.get("vol_strategy", "mean") if "ENV" in dir() else cfg.VOLUME_FILL_STRATEGY
    
    if vol_strategy != "mean":
        return df
    
    zero_mask = (df[vol_col].isna()) | (df[vol_col] == 0)
    if not zero_mask.any():
        return df
    
    # 计算历史非零成交量的均值（排除今日/当日）
    hist_data = df.loc[~zero_mask, vol_col]
    if hist_data.empty:
        return df
    
    # 取近30天历史均值
    hist_mean = hist_data.tail(lookback_days).mean()
    if pd.isna(hist_mean) or hist_mean <= 0:
        return df
    
    # 填充零值
    filled_count = zero_mask.sum()
    df.loc[zero_mask, vol_col] = hist_mean
    print(f"⚠️ [{filled_count}条] 成交量为零，已用近{lookback_days}天均值 {hist_mean:,.0f} 填充 (VOL_STRATEGY=mean)")
    return df

# 应用成交量修复
df = _fill_zero_volume_with_hist_mean(df.copy())




Today 2026-06-03 | O:25964.89 H:25964.89 L:25586.31 C:25620.01

Loading external assets (with cache + yfinance/alltick fallback)...
🔄 [000001.SS] 命中缓存，增量拉取: 2026-05-26 -> 2026-06-03
⚠️ [000001.SS] yfinance 异常: Too Many Requests. Rate limited. Try after a while.，尝试 AllTick...
📡 [000001.SS] 调用 AllTick API（增量区间: 2026-05-26 -> 2026-06-03）...
ℹ️ 已加载 AllTick 配置文件: alltick_cache/alltick_config.json
❌ [000001.SS] AllTick 请求失败: HTTPSConnectionPool(host='quote.alltick.io', port=443): Max retries exceeded with url: /quote-b-api/kline?token=eb962f1fe68f62bd495697532c8d5799-c-app&query=%7B%22data%22%3A%7B%22code%22%3A%22SSE%22%2C%22kline_type%22%3A%228%22%2C%22kline_timestamp_end%22%3A%220%22%2C%22query_kline_num%22%3A%22240%22%2C%22adjust_type%22%3A%220%22%7D%7D (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1007)')))
ℹ️ [000001.SS] AllTick status=600, code=SH000001, url=https://quote.alltick.io/quote-b-api/kline
❌ [000001.SS]

In [5]:
# ============================================================
# STEP 1: DEFINE TRADING PATTERNS (修复版)
# ============================================================

def detect_trading_patterns(df: pd.DataFrame, config: GlobalConfig) -> pd.DataFrame:
    """
    检测交易模式特征（修复递归 + 依赖状态修复）
    """
    # 1) 输入检查
    if df is None or df.empty:
        raise ValueError("df为空，无法进行特征检测")

    # 2) 防止索引问题
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")
    if hasattr(df.index, "tz") and df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    df.index = pd.to_datetime(df.index).normalize()

    # 3) 依赖检查（仅修复 data 丢失，不做递归调用）
    try:
        config.check_dependency('feature')
    except RuntimeError as e:
        if "缺少前置依赖: data" in str(e):
            config.set_status('data', True)
            config.check_dependency('feature')
        else:
            raise

    patterns = pd.DataFrame(index=df.index)


    # ===================== 窗口参数定义（参数化）=====================
    # ✅ 从config获取主要窗口参数，如果没有则使用合理默认值
    # 注意：以下窗口参数如未在GlobalConfig中定义，可使用这些默认值
    # 建议后续将 SHORT_WINDOW, MID_WINDOW, LONG_WINDOW, YEARLY_WINDOW 添加到 GlobalConfig
    SEQUENCE_LEN = config.SEQUENCE_LENGTH  # 30（主序列长度）
    SHORT_WINDOW = 5  # 短期窗口（成交量5日均线等）
    MID_WINDOW = config.SEQUENCE_LENGTH  # ⚠️ 关键修复：中期窗口必须与序列长度一致（ATR、成交量、布林带等），确保逻辑对齐
    LONG_WINDOW = 60  # 长期窗口（季度动量、相关性等）
    # ✅ 窗口参数兜底：检查 GlobalConfig 是否有对应属性
    # ⚠️ TODO：建议在 GlobalConfig 中添加 YEARLY_WINDOW 和 MA_MACRO_WINDOW 属性
    # 当前 GlobalConfig 中不存在这些属性，保持现状使用硬编码值
    if not hasattr(config, 'YEARLY_WINDOW'):
        YEARLY_WINDOW = 252  # ⚠️ GlobalConfig 中暂无此属性，建议添加
    else:
        YEARLY_WINDOW = config.YEARLY_WINDOW

    # talib 技术指标参数（标准化周期）
    TALIB_ATR_PERIOD = MID_WINDOW  # ATR使用中期窗口（已同步SEQUENCE_LENGTH）
    TALIB_RSI_PERIOD = 14  # RSI标准周期（行业标准）
    TALIB_MACD_FAST = 12  # MACD快速周期（行业标准）
    TALIB_MACD_SLOW = 26  # MACD慢速周期（行业标准）
    TALIB_MACD_SIGNAL = 9  # MACD信号周期（行业标准）
    TALIB_ADX_PERIOD = 14  # ADX周期（行业标准）
    TALIB_STOCH_K = 9  # KDJ K值周期（行业标准）
    TALIB_STOCH_D = 3  # KDJ D值周期（行业标准）

    # ===================== 0. 准备基础数据 (Float64) =====================
    # ⚠️ 关键修复：在调用 talib 前先 ffill()，防止中途 NaN 导致整条特征序列失效
    df_prep = df[['Open', 'High', 'Low', 'Close', 'Volume']].ffill()
    open_p = df_prep['Open'].values.astype(np.float64)
    high_p = df_prep['High'].values.astype(np.float64)
    low_p = df_prep['Low'].values.astype(np.float64)
    close_p = df_prep['Close'].values.astype(np.float64)
    volume_p = df_prep['Volume'].values.astype(np.float64)

    # ===================== 1. 基础收益率 =====================
    # ✅ 显式指定 fill_method=None 以适配最新版 Pandas
    df['Returns'] = df['Close'].ffill().pct_change(fill_method=None).fillna(0)
    patterns['Returns'] = df['Returns']
    # Log Return 用于计算
    patterns['Log_Ret'] = np.log(df['Close'] / df['Close'].shift(1)).fillna(0)

    # ===================== 2. 波动率基准 (ATR) =====================
    # ✅ 使用参数化的 TALIB_ATR_PERIOD
    atr_mid = talib.ATR(high_p, low_p, close_p, timeperiod=TALIB_ATR_PERIOD)
    atr_mid_s = pd.Series(atr_mid, index=df.index)
    # ✅ 使用参数化的 MID_WINDOW
    patterns['Volatility_Ratio'] = atr_mid_s / (atr_mid_s.rolling(MID_WINDOW, min_periods=MID_WINDOW).mean().shift(1) + 1e-10)

    # ===================== 3. 跳空缺口 (Gaps) =====================
    prev_close = df['Close'].shift(1)
    prev_atr = atr_mid_s.shift(1)
    gap_raw = (df['Open'] - prev_close) / prev_close
    gap_threshold = (prev_atr / prev_close * 0.5).fillna(1.0)
    patterns['Gap_Up'] = (gap_raw > 0).astype(float) * gap_raw * (gap_raw > gap_threshold).astype(float)
    patterns['Gap_Down'] = (gap_raw < 0).astype(float) * abs(gap_raw) * (abs(gap_raw) > gap_threshold).astype(float)

    # ===================== 4. 成交量因子 (Volume Factors) =====================
    # ✅ 双时间窗口基准：使用参数化的窗口大小
    vol_ma_mid = df['Volume'].rolling(window=MID_WINDOW, min_periods=5).mean()
    vol_ma_short = df['Volume'].rolling(window=SHORT_WINDOW, min_periods=3).mean()

    # 相对量能 (Relative Volume) - 连续型指标
    # ✅ 使用参数化的窗口变量名
    patterns['Vol_Rel_Mid'] = (df['Volume'] / (vol_ma_mid + 1e-10)).replace([np.inf, -np.inf], 0).fillna(0)
    # ✅ 使用cfg参数：成交量特征平稳化（clip到合理范围）
    patterns['Vol_Rel_Mid'] = patterns['Vol_Rel_Mid'].clip(0, 5)
    patterns['Vol_Rel_Short'] = (df['Volume'] / (vol_ma_short + 1e-10)).replace([np.inf, -np.inf], 0).fillna(0)
    patterns['Vol_Rel_Short'] = patterns['Vol_Rel_Short'].clip(0, 5)

    # 兼容性别名（保持向后兼容）
    patterns['Vol_Rel_20'] = patterns['Vol_Rel_Mid']
    patterns['Vol_Rel_5'] = patterns['Vol_Rel_Short']

    # 状态信号：自适应阈值
    patterns['Vol_Surge'] = (patterns['Vol_Rel_Mid'] > 2.0).astype(float)
    patterns['Vol_Dry_Up'] = (patterns['Vol_Rel_Mid'] < 0.6).astype(float)

    # 量价逻辑拆分
    patterns['Vol_Price_Up_Confirm'] = (
        (patterns['Log_Ret'] > 0) & (patterns['Vol_Rel_Mid'] > 1.2)
    ).astype(float) * patterns['Vol_Rel_Mid']

    patterns['Vol_Price_Down_Panic'] = (
        (patterns['Log_Ret'] < 0) & (patterns['Vol_Rel_Mid'] > 1.2)
    ).astype(float) * patterns['Vol_Rel_Mid']

    patterns['Vol_Pullback_Low'] = (
        (patterns['Log_Ret'] < 0) & (patterns['Vol_Rel_Short'] < 0.8)
    ).astype(float)

    # 量价偏差与连续性
    patterns['Vol_Bias_Mid'] = (patterns['Vol_Rel_Mid'] - 1.0)
    patterns['Vol_Bias_20'] = patterns['Vol_Bias_Mid']  # 兼容性别名

    vol_diff = df['Volume'].diff()
    price_diff = df['Close'].diff()
    is_up_vol = (price_diff > 0) & (vol_diff > 0)
    is_down_dry = (price_diff < 0) & (vol_diff < 0)
    patterns['Vol_Trend_Consistency'] = (is_up_vol | is_down_dry).rolling(3).sum().fillna(0)

    # 兼容性别名
    patterns['Vol_Rel'] = patterns['Vol_Rel_Mid']
    patterns['High_Volume'] = patterns['Vol_Surge']

    # ===================== 5. 反转形态 =====================
    patterns['Gap_Up_Reversal'] = (
        (patterns['Gap_Up'] > 0.005) & (df['Close'] < df['Open']) & (patterns['Vol_Rel_Mid'] > 1.2)
    ).astype(float) * patterns['Gap_Up']

    patterns['Gap_Down_Reversal'] = (
        (patterns['Gap_Down'] > 0.005) & (df['Close'] > df['Open']) & (patterns['Vol_Rel_Mid'] > 1.2)
    ).astype(float) * patterns['Gap_Down']

    # ===================== 6. 趋势连续性 =====================
    patterns['Trend_2Day'] = df['Log_Ret'].rolling(2).sum().fillna(0)
    t2_sign = np.sign(patterns['Trend_2Day'])
    patterns['Trend_Consistency'] = ((t2_sign == t2_sign.shift(1)) & (t2_sign == t2_sign.shift(2))).astype(float).fillna(0)

    # ===================== 7. 回撤反弹 =====================
    # ✅ 使用参数化的 MID_WINDOW
    rolling_max = df['Close'].rolling(MID_WINDOW).max().shift(1)
    drawdown = (df['Close'].shift(1) - rolling_max) / (rolling_max + 1e-10)
    drawdown_5pct = drawdown.rolling(SHORT_WINDOW).quantile(0.1).fillna(-0.05)
    patterns['Drawdown_Bounce'] = ((drawdown < drawdown_5pct) & (df['Returns'] > 0.01)).astype(float) * df['Returns']

    # ===================== 8. 趋势天数 =====================
    patterns['2Day_Uptrend'] = ((df['Returns'] > 0) & (df['Returns'].shift(1) > 0)).astype(float)
    patterns['2Day_Downtrend'] = ((df['Returns'] < 0) & (df['Returns'].shift(1) < 0)).astype(float)
    patterns['3Day_Uptrend'] = ((df['Returns'] > 0) & (df['Returns'].shift(1) > 0) & (df['Returns'].shift(2) > 0)).astype(float)

    # ===================== 9. 突破形态 =====================
    # ✅ 使用参数化的 MID_WINDOW
    roll_high = df['High'].rolling(MID_WINDOW).max().shift(1)
    roll_low = df['Low'].rolling(MID_WINDOW).min().shift(1)
    patterns['Breakout_High'] = ((df['Close'] > roll_high) & (patterns['Vol_Rel_Mid'] > 1.5)).astype(float) * ((df['Close'] - roll_high) / roll_high)
    patterns['Breakdown_Low'] = ((df['Close'] < roll_low) & (patterns['Vol_Rel_Mid'] > 1.5)).astype(float) * ((roll_low - df['Close']) / roll_low)
    patterns['Breakout_Confirm'] = ((patterns['Breakout_High'] > 0) & (patterns['Breakout_High'].shift(1) > 0)).astype(float)

    # ===================== 10. 波动模式 =====================
    daily_range = (df['High'] - df['Low']) / (df['Low'] + 1e-10)
  #  patterns['High_Volatility'] = (daily_range > 0.02).astype(float) * daily_range
    patterns['Low_Volatility'] = (daily_range < 0.005).astype(float) * (0.005 - daily_range)

    # ===================== 11. 动量模式 =====================
    # ✅ 使用参数化的 SHORT_WINDOW，显式指定 fill_method=None
    ret_short = df['Close'].ffill().pct_change(SHORT_WINDOW, fill_method=None)
    patterns['Strong_Momentum_Up'] = (ret_short > 0.03).astype(float) * ret_short / 0.03
  #  patterns['Strong_Momentum_Down'] = (ret_short < -0.03).astype(float) * abs(ret_short) / 0.03

    # ===================== 12. 日内形态 =====================
    patterns['Inside_Day'] = ((df['High'] < df['High'].shift(1)) & (df['Low'] > df['Low'].shift(1))).astype(float)
    patterns['Outside_Day'] = ((df['High'] > df['High'].shift(1)) & (df['Low'] < df['Low'].shift(1))).astype(float) * daily_range

    # ===================== 13. RSI =====================
    # ✅ 使用参数化的 TALIB_RSI_PERIOD
    rsi = talib.RSI(close_p, timeperiod=TALIB_RSI_PERIOD)
    rsi_s = pd.Series(rsi, index=df.index).fillna(50)
    patterns['RSI_Overbought'] = (rsi_s > 70).astype(float) * (rsi_s - 70) / 30
    patterns['RSI_Oversold'] = (rsi_s < 30).astype(float) * (30 - rsi_s) / 30
    patterns['RSI_Near_50'] = ((rsi_s >= 45) & (rsi_s <= 55)).astype(float)
    patterns['RSI_Divergence'] = np.clip((rsi_s - 50) / 50.0, -1, 1)

    # ===================== 14. MACD =====================
    # ✅ 使用参数化的 TALIB_MACD 参数
    macd, signal, hist = talib.MACD(close_p, fastperiod=TALIB_MACD_FAST, slowperiod=TALIB_MACD_SLOW, signalperiod=TALIB_MACD_SIGNAL)
    macd_s, signal_s = pd.Series(macd, index=df.index), pd.Series(signal, index=df.index)
    hist_s = pd.Series(hist, index=df.index).fillna(0)
    patterns['MACD_Bullish_Cross'] = ((macd_s > signal_s) & (macd_s.shift(1) <= signal_s.shift(1))).astype(float)
    patterns['MACD_Bearish_Cross'] = ((macd_s < signal_s) & (macd_s.shift(1) >= signal_s.shift(1))).astype(float)

    # ===================== 15. 布林带 =====================
    # ✅ 使用参数化的 MID_WINDOW 和 LONG_WINDOW
    bb_mid = df['Close'].rolling(MID_WINDOW).mean().shift(1)
    bb_std = df['Close'].rolling(MID_WINDOW).std().fillna(0).shift(1)
    bb_upper, bb_lower = bb_mid + 2 * bb_std, bb_mid - 2 * bb_std
    bb_width = (bb_upper - bb_lower) / (bb_mid + 1e-10)
    # ⚠️ 关键修复：防止未来泄露 - 使用滚动窗口内的 quantile，禁止在整个 Series 上直接使用 quantile
    bb_width_shifted = bb_width.shift(1)
    bb_width_99th_rolling = bb_width_shifted.rolling(YEARLY_WINDOW, min_periods=LONG_WINDOW).quantile(0.99).shift(1)
    bb_width_clipped = np.clip(bb_width_shifted, 0, bb_width_99th_rolling.fillna(bb_width_shifted.rolling(LONG_WINDOW).quantile(0.95)))
    bb_width_quantile = bb_width_clipped.rolling(LONG_WINDOW).quantile(0.1).shift(1)

   # patterns['BB_Position'] = np.clip((df['Close'] - bb_mid) / (bb_std + 1e-10), -3, 3)
    patterns['BB_Width'] = bb_width.shift(1)
    patterns['BB_Touch_Upper'] = (df['Close'] >= bb_upper).astype(float)
    patterns['BB_Touch_Lower'] = (df['Close'] <= bb_lower).astype(float)
    patterns['BB_Width_Compressed'] = (bb_width.shift(1) < bb_width_quantile).astype(float)
    patterns['BB_Pos_Norm'] = (df['Close'] - bb_lower) / (bb_upper - bb_lower + 1e-10)

    # ===================== 16. 情绪因子 =====================
    # ✅ 使用参数化的窗口：50日窗口使用 LONG_WINDOW 的近似值（约50）
    VIX_SENTIMENT_WINDOW = 50  # VIX情绪窗口（可根据需要调整）
    vix_min = df['VIX_Close'].rolling(VIX_SENTIMENT_WINDOW).min()
    vix_max = df['VIX_Close'].rolling(VIX_SENTIMENT_WINDOW).max()
    patterns['Sentiment_Fear_Rank'] = (df['VIX_Close'] - vix_min) / (vix_max - vix_min + 1e-10)
    # ✅ 显式指定 fill_method=None
    patterns['Sentiment_Change'] = df['VIX_Close'].ffill().pct_change(fill_method=None).fillna(0)

    # 检查外部数据是否存在（多品种适配）
    has_spx_ret = 'SPX_Ret' in df.columns and not df['SPX_Ret'].isna().all()
    has_sse_ret = 'SSE_Ret' in df.columns and not df['SSE_Ret'].isna().all()

    if has_spx_ret and has_sse_ret:
        patterns['Market_Up_Down_Ratio'] = np.where(df['SPX_Ret'].fillna(0) > 0, 1.5, 0.5) * np.where(df['SSE_Ret'].fillna(0) > 0, 1.2, 0.8)
    else:
        patterns['Market_Up_Down_Ratio'] = 1.0

    # ⚠️ 关键修复：防止未来泄露 - 使用滚动窗口内的 rank，禁止在整个 Series 上直接使用 rank
    patterns['VIX_Quantile'] = df['VIX_Close'].shift(1).rolling(YEARLY_WINDOW, min_periods=LONG_WINDOW).apply(
        lambda x: pd.Series(x).rank(pct=True).iloc[-1] if len(x) >= LONG_WINDOW else np.nan,
        raw=False
    ).fillna(0.5)

    if has_sse_ret:
        patterns['Industry_Diff'] = df['Returns'] - df['SSE_Ret'].fillna(0)
    else:
        patterns['Industry_Diff'] = 0.0

    # ✅ 使用参数化的 SHORT_WINDOW（3日近似）
    VIX_CHANGE_WINDOW = 3
    patterns['VIX_Change_Rate'] = df['VIX_Close'].pct_change(VIX_CHANGE_WINDOW, fill_method=None).fillna(0)

    # ✅ 使用参数化的 YEARLY_WINDOW 和 LONG_WINDOW
    vix_roll_year = df['VIX_Close'].rolling(YEARLY_WINDOW, min_periods=LONG_WINDOW)
    patterns['VIX_Rank_Year'] = (df['VIX_Close'] - vix_roll_year.min()) / (vix_roll_year.max() - vix_roll_year.min() + 1e-10)
    patterns['VIX_Quantile_Rank'] = df['VIX_Close'].rolling(YEARLY_WINDOW, min_periods=LONG_WINDOW).apply(
        lambda x: pd.Series(x).rank(pct=True).iloc[-1] if len(x) >= LONG_WINDOW else np.nan, raw=False
    ).fillna(0.5)

    # ✅ 使用参数化的 SHORT_WINDOW（10日近似）
    CORR_SHORT_WINDOW = 10
    patterns['Corr_Price_VIX'] = df['Log_Ret'].rolling(CORR_SHORT_WINDOW).corr(df['VIX_Close'].pct_change(fill_method=None)).fillna(-1)

    # ===================== 17. 市场开放标志位 =====================
    if 'Is_US_Market_Open' in df.columns:
        patterns['Is_US_Market_Open'] = df['Is_US_Market_Open']
    else:
        patterns['Is_US_Market_Open'] = (~df['SPX_Close'].isnull()).astype(float) if 'SPX_Close' in df.columns else 0.0

    if 'Is_CN_Market_Open' in df.columns:
        patterns['Is_CN_Market_Open'] = df['Is_CN_Market_Open']
    else:
        patterns['Is_CN_Market_Open'] = (~df['SSE_Close'].isnull()).astype(float) if 'SSE_Close' in df.columns else 0.0

  #  if 'SPX_IsFilled' in df.columns:
  #      patterns['SPX_IsFilled'] = df['SPX_IsFilled']
  #  else:
  #      patterns['SPX_IsFilled'] = 0.0

    if 'SSE_IsFilled' in df.columns:
        patterns['SSE_IsFilled'] = df['SSE_IsFilled']
    else:
        patterns['SSE_IsFilled'] = 0.0

    # ===================== 18. 宏观/季节 =====================
  #  patterns['Month_Sin'] = np.sin(2 * np.pi * df.index.month / 12)
  #  patterns['Month_Cos'] = np.cos(2 * np.pi * df.index.month / 12)
  #  patterns['Quarter'] = df.index.quarter
  #  patterns = pd.get_dummies(patterns, columns=['Quarter'], prefix='Quarter', dtype=float)

    # ✅ 窗口参数兜底：检查 GlobalConfig 是否有对应属性
    if not hasattr(config, 'MA_MACRO_WINDOW'):
        MA_MACRO_WINDOW = 200  # ⚠️ GlobalConfig 中暂无此属性，建议添加（宏观趋势窗口，接近YEARLY_WINDOW）
    else:
        MA_MACRO_WINDOW = config.MA_MACRO_WINDOW
    ma_macro = df['Close'].rolling(MA_MACRO_WINDOW).mean()
    patterns['Trend_Macro_200d'] = (df['Close'] - ma_macro) / (ma_macro + 1e-10)

    # ✅ 使用参数化的 LONG_WINDOW，显式指定 fill_method=None
    patterns['MOM_Quarter'] = df['Close'].pct_change(LONG_WINDOW, fill_method=None).fillna(0)

    # ===================== 19. 稳定性 (ADX) =====================
    # ✅ 使用参数化的 TALIB_ADX_PERIOD
    patterns['ADX'] = pd.Series(talib.ADX(high_p, low_p, close_p, timeperiod=TALIB_ADX_PERIOD), index=df.index).fillna(0)
   # patterns['PLUS_DI'] = pd.Series(talib.PLUS_DI(high_p, low_p, close_p, timeperiod=TALIB_ADX_PERIOD), index=df.index).fillna(0)
   # patterns['MINUS_DI'] = pd.Series(talib.MINUS_DI(high_p, low_p, close_p, timeperiod=TALIB_ADX_PERIOD), index=df.index).fillna(0)

    # ===================== 20. KDJ =====================
    # ✅ 使用参数化的 TALIB_STOCH 参数
    k, d = talib.STOCH(high_p, low_p, close_p, fastk_period=TALIB_STOCH_K, slowk_period=TALIB_STOCH_D)
    patterns['KDJ_K'] = pd.Series(k, index=df.index).fillna(50)
    patterns['KDJ_D'] = pd.Series(d, index=df.index).fillna(50)
    patterns['KDJ_J'] = np.clip(3 * patterns['KDJ_K'] - 2 * patterns['KDJ_D'], 0, 100)

    # ===================== 21 & 22. 趋势效率 =====================
    # ✅ 使用参数化的 SHORT_WINDOW（10日）
    path_len = np.abs(df['Close'] - df['Close'].shift(1)).rolling(SHORT_WINDOW).sum()
    direction = np.abs(df['Close'] - df['Close'].shift(SHORT_WINDOW))
    patterns['Trend_Efficiency'] = np.where(path_len > 0, direction / (path_len + 1e-10), 0)

    # ✅ 使用参数化的窗口：5/10/20/60日移动平均
    ma_short = df['Close'].rolling(SHORT_WINDOW).mean()
    ma_mid_short = df['Close'].rolling(SHORT_WINDOW * 2).mean()  # 10日
    ma_mid = df['Close'].rolling(MID_WINDOW).mean()
    ma_long = df['Close'].rolling(LONG_WINDOW).mean()
    patterns['Trend_Alignment'] = ((ma_short > ma_mid_short) & (ma_mid_short > ma_mid) & (ma_mid > ma_long)).astype(float)

    # 兼容性别名（保持向后兼容）
    ma_5, ma_10, ma_20, ma_60 = ma_short, ma_mid_short, ma_mid, ma_long

    # ===================== 23. 微观结构 =====================
    patterns['Shadow_Upper_Pct'] = (df['High'] - np.maximum(df['Close'], df['Open'])) / (df['High'] - df['Low'] + 1e-10)
    patterns['Shadow_Lower_Pct'] = (np.minimum(df['Close'], df['Open']) - df['Low']) / (df['High'] - df['Low'] + 1e-10)
    patterns['ADOSC'] = pd.Series(talib.ADOSC(high_p, low_p, close_p, volume_p), index=df.index).fillna(0)
    #patterns['Acceleration'] = patterns['Returns'].diff().fillna(0)

    # ===================== 24. 量价同步 =====================
    vol_trend = np.sign(df['Volume'].diff()).rolling(3).sum()
    patterns['Vol_Price_Trend_Sync'] = (vol_trend == np.sign(df['Returns']).rolling(3).sum()).astype(float)

    # Gap adj
    patterns['Gap_Up_VolAdj'] = patterns['Gap_Up'] / (patterns['Vol_Rel_Mid'] + 0.1)

    patterns['Range_Pct'] = (df['High'] - df['Low']) / df['Close']
    patterns['Open_Close_Ratio'] = df['Open'] / df['Close'] - 1

    # 检查外部数据是否存在（多品种适配：缺失时设为0或中性值）
    has_spx = 'SPX_Ret' in df.columns and not df['SPX_Ret'].isna().all()
    has_sse = 'SSE_Ret' in df.columns and not df['SSE_Ret'].isna().all()
    has_spx_close = 'SPX_Close' in df.columns and not df['SPX_Close'].isna().all()
    has_sse_close = 'SSE_Close' in df.columns and not df['SSE_Close'].isna().all()
    has_cnh = 'USDCNH_Close' in df.columns and not df['USDCNH_Close'].isna().all()
    has_hkd = 'USDHKD_Close' in df.columns and not df['USDHKD_Close'].isna().all()
    has_gold = 'GOLD_Close' in df.columns and not df['GOLD_Close'].isna().all()
    has_dxy = 'DXY_Close' in df.columns and not df['DXY_Close'].isna().all()

    # SPX相关特征
  #  if has_spx:
   #     patterns['Cross_Ret_SPX'] = df['Returns'] - df['SPX_Ret'].fillna(0)
  #  else:
  #      patterns['Cross_Ret_SPX'] = 0.0

    if has_sse:
        patterns['Cross_Ret_SSE'] = df['Returns'] - df['SSE_Ret'].fillna(0)
    else:
        patterns['Cross_Ret_SSE'] = 0.0

    if has_sse_close:
      patterns['SSE_Gap'] = (df['SSE_Close'] / df['SSE_Close'].shift(1) - 1).fillna(0)
    else:
        patterns['SSE_Gap'] = 0.0

  #  if has_spx_close:
  #      patterns['SPX_Gap'] = (df['SPX_Close'] / df['SPX_Close'].shift(1) - 1).fillna(0)
  #  else:
  #      patterns['SPX_Gap'] = 0.0

    # Open_Norm需要使用prev_atr（在函数前面已定义）
    patterns['Open_Norm'] = (df['Open'] - df['Close'].shift(1)) / (prev_atr + 1e-10)

 #   patterns['MA5_MA10_Diff'] = ma_5 - ma_10
 #   patterns['MA10_MA20_Diff'] = ma_10 - ma_20
    patterns['RSI_Trend'] = rsi_s - rsi_s.shift(3)
    trend_day = df['Returns'].apply(lambda x: 1 if x>0 else (-1 if x<0 else 0))
    patterns['Trend_Day_Count'] = trend_day.rolling(3).sum().fillna(0)
 #   patterns['Pullback_Pct'] = (df['High'] - df['Close']) / (df['High'] - df['Low'] + 1e-10) * 100

    # ✅ 使用参数化的 MID_WINDOW
    patterns['Return_Volatility'] = df['Returns'].rolling(MID_WINDOW).std()
    patterns['Volatility_Cluster'] = df['Returns'].rolling(MID_WINDOW).std()

# ✅ 新增特征：Returns_ZScore（方差齐性，特征平稳化补强 + 梯度防御）
    # Returns_ZScore = Log_Ret / (Volatility_Cluster + 1e-10)，使收益率特征具备方差齐性
    patterns['Returns_ZScore'] = patterns['Log_Ret'] / (patterns['Volatility_Cluster'] + 1e-10)
    patterns['Returns_ZScore'] = patterns['Returns_ZScore'].replace([np.inf, -np.inf], 0).fillna(0).clip(-5, 5)  # ✅ 梯度防御：clip到可控范围

    # ✅ 使用相对指标替代绝对价格序列（特征平稳化），使用参数化的 MA_MACRO_WINDOW
    ma_macro_price = df['Close'].rolling(MA_MACRO_WINDOW, min_periods=50).mean()
    patterns['Price_to_MA200_Ratio'] = (df['Close'] / (ma_macro_price + 1e-10) - 1.0).fillna(0)

    # ===================== Beta和相关性特征 =====================
    # ✅ 使用参数化的 LONG_WINDOW 和 MID_WINDOW
    if has_spx:
        cov = df['Returns'].shift(1).rolling(LONG_WINDOW).cov(df['SPX_Ret'].fillna(0)).fillna(0)
        spx_var = df['SPX_Ret'].fillna(0).rolling(LONG_WINDOW).var()
        patterns['Beta_Coeff'] = cov / (spx_var + 1e-10)
#        patterns['RS_vs_SPX'] = df['Close'].pct_change(MID_WINDOW, fill_method=None) - df['SPX_Close'].pct_change(MID_WINDOW, fill_method=None) if has_spx_close else 0.0
#        patterns['RS_vs_SPX'] = patterns['RS_vs_SPX'].fillna(0)
  #      patterns['Corr_SPX_60'] = df['Log_Ret'].rolling(LONG_WINDOW).corr(df['SPX_Ret'].fillna(0)).fillna(0)
    else:
        patterns['Beta_Coeff'] = 0.0
 #       patterns['RS_vs_SPX'] = 0.0
 #       patterns['Corr_SPX_60'] = 0.0

    if has_sse:
        patterns['RS_vs_SSE'] = df['Close'].pct_change(MID_WINDOW, fill_method=None) - df['SSE_Close'].pct_change(MID_WINDOW, fill_method=None) if has_sse_close else 0.0
        patterns['RS_vs_SSE'] = patterns['RS_vs_SSE'].fillna(0)
        patterns['Corr_SSE_60'] = df['Log_Ret'].rolling(LONG_WINDOW).corr(df['SSE_Ret'].fillna(0)).fillna(0)
    else:
        patterns['RS_vs_SSE'] = 0.0
        patterns['Corr_SSE_60'] = 0.0

    # ===================== 29. 宏观金融（多品种适配）=====================
    has_cnh = 'USDCNH_Close' in df.columns and not df['USDCNH_Close'].isna().all()
    has_hkd = 'USDHKD_Close' in df.columns and not df['USDHKD_Close'].isna().all()
    has_us10y = 'US10Y_Close' in df.columns and not df['US10Y_Close'].isna().all()
    has_dxy = 'DXY_Close' in df.columns and not df['DXY_Close'].isna().all()

    if has_cnh:
        # ✅ 使用参数化的 SHORT_WINDOW
        patterns['CNH_Change_5D'] = df['USDCNH_Close'].pct_change(SHORT_WINDOW, fill_method=None).fillna(0)
        patterns['CNH_Trend_Dev'] = (df['USDCNH_Close'] - df['USDCNH_Close'].rolling(LONG_WINDOW).mean()) / (df['USDCNH_Close'] + 1e-10).fillna(0)
    else:
        patterns['CNH_Change_5D'] = 0.0
        patterns['CNH_Trend_Dev'] = 0.0

    if has_us10y:
        patterns['US10Y_Change'] = df['US10Y_Close'].diff().fillna(0)
        # ✅ 使用参数化的 YEARLY_WINDOW 和 LONG_WINDOW
        us10y_high = df['US10Y_Close'].rolling(YEARLY_WINDOW).max()
        patterns['US10Y_Stress'] = (df['US10Y_Close'] / (us10y_high + 1e-10)).fillna(0)
        us10y_roll_year = df['US10Y_Close'].rolling(YEARLY_WINDOW, min_periods=LONG_WINDOW)
        patterns['US10Y_Rank_Year'] = (df['US10Y_Close'] - us10y_roll_year.min()) / (us10y_roll_year.max() - us10y_roll_year.min() + 1e-10)
        patterns['US10Y_Quantile_Rank'] = df['US10Y_Close'].rolling(YEARLY_WINDOW, min_periods=LONG_WINDOW).apply(
            lambda x: pd.Series(x).rank(pct=True).iloc[-1] if len(x) >= LONG_WINDOW else np.nan, raw=False
        ).fillna(0.5)
    else:
        patterns['US10Y_Change'] = 0.0
        patterns['US10Y_Stress'] = 0.0
        patterns['US10Y_Rank_Year'] = 0.5
        patterns['US10Y_Quantile_Rank'] = 0.5

    if has_dxy:
        # ✅ 使用参数化的 SHORT_WINDOW（10日近似）
        patterns['DXY_Trend'] = np.sign(df['DXY_Close'].diff(SHORT_WINDOW).fillna(0))
    else:
        patterns['DXY_Trend'] = 0.0

    if has_hkd:
        patterns['HKD_Flow_Position'] = np.clip((df['USDHKD_Close'] - 7.75) / (7.85 - 7.75), 0, 1).fillna(0.5)
    else:
        patterns['HKD_Flow_Position'] = 0.5

    # Macro_Triple_Bear需要所有三个指标都存在
    if has_cnh and has_us10y and has_dxy:
        is_cnh_weak = df['USDCNH_Close'].diff() > 0
        is_us10y_up = df['US10Y_Close'].diff() > 0
        is_dxy_up = df['DXY_Close'].diff() > 0
        patterns['Macro_Triple_Bear'] = (is_cnh_weak & is_us10y_up & is_dxy_up).astype(float)
    else:
        patterns['Macro_Triple_Bear'] = 0.0

    # ===================== 30. 聪明钱 - CMF =====================
    # ✅ 使用参数化的 TALIB_ADX_PERIOD（MFI使用相同的周期参数）
    patterns['MFI'] = pd.Series(talib.MFI(high_p, low_p, close_p, volume_p, timeperiod=TALIB_ADX_PERIOD), index=df.index).fillna(50)

    mf_multiplier = ((df['Close'] - df['Low']) - (df['High'] - df['Close'])) / (df['High'] - df['Low'] + 1e-10)
    mf_volume = mf_multiplier * df['Volume']
    # ✅ 使用参数化的 MID_WINDOW
   # patterns['CMF'] = mf_volume.rolling(MID_WINDOW).sum() / (df['Volume'].rolling(MID_WINDOW).sum() + 1e-10)

    obv = talib.OBV(close_p, volume_p)
    obv_ma = pd.Series(obv).rolling(MID_WINDOW).mean()
    patterns['OBV_Trend_Dev'] = (obv - obv_ma) / (np.abs(obv_ma) + 1e-10)

    # ===================== 31. 高阶统计矩 =====================
    # ✅ 使用参数化的 MID_WINDOW
  #  patterns['Ret_Skew_20'] = df['Log_Ret'].rolling(MID_WINDOW).skew().fillna(0)
  #  patterns['Ret_Kurt_20'] = df['Log_Ret'].rolling(MID_WINDOW).kurt().fillna(0)

    # ===================== 32. 市场分形 - Choppiness Index =====================
    # ✅ 使用参数化的 TALIB_ADX_PERIOD（Choppiness Index 使用相同的周期参数）
    CHOPPINESS_PERIOD = TALIB_ADX_PERIOD  # 14日（行业标准）
    tr1 = talib.TRANGE(high_p, low_p, close_p)
    tr_sum_period = pd.Series(tr1, index=df.index).rolling(CHOPPINESS_PERIOD).sum().shift(1)
    high_roll_period = df['High'].rolling(CHOPPINESS_PERIOD).max().shift(1)
    low_roll_period = df['Low'].rolling(CHOPPINESS_PERIOD).min().shift(1)

    chop_ratio = tr_sum_period / (high_roll_period - low_roll_period + 1e-10)
    patterns['Choppiness'] = 100 * np.log10(chop_ratio) / np.log10(CHOPPINESS_PERIOD)
    patterns['Choppiness'] = patterns['Choppiness'].fillna(50)

    # Ret_CV_20 (变异系数) - 修复异常值保护
    # ✅ 使用参数化的 MID_WINDOW
 #   ts_roll = df['Log_Ret'].rolling(MID_WINDOW, min_periods=10)
 #   ret_mean_abs = ts_roll.mean().abs()
  #  patterns['Ret_CV_20'] = ts_roll.std() / (ret_mean_abs + 1e-6).fillna(1.0)
  #  patterns['Ret_CV_20'] = patterns['Ret_CV_20'].clip(0, 100).fillna(0)

    # ===================== 33. 尾部风险与极端值 =====================
    # ✅ 使用参数化的 LONG_WINDOW、YEARLY_WINDOW 和 MA_MACRO_WINDOW
    patterns['Price_Z_Score_60'] = (df['Close'] - ma_macro) / (df['Close'].rolling(LONG_WINDOW).std() + 1e-10)
    patterns['Price_Rank_252'] = df['Close'].shift(1).rolling(YEARLY_WINDOW).rank(pct=True).fillna(0.5)

    # ===================== 34 & 35. 跨资产与动量（多品种适配）=====================
    has_spx_close = 'SPX_Close' in df.columns and not df['SPX_Close'].isna().all()
    has_gold = 'GOLD_Close' in df.columns and not df['GOLD_Close'].isna().all()
    has_dxy = 'DXY_Close' in df.columns and not df['DXY_Close'].isna().all()
    has_es = 'ES_Close' in df.columns and not df['ES_Close'].isna().all()
    has_nq = 'NQ_Close' in df.columns and not df['NQ_Close'].isna().all()
    has_oil = 'OIL_Close' in df.columns and not df['OIL_Close'].isna().all()
    has_es_ret = 'ES_Ret' in df.columns and not df['ES_Ret'].isna().all()
    has_oil_ret = 'OIL_Ret' in df.columns and not df['OIL_Ret'].isna().all()
    has_gold_ret = 'GOLD_Ret' in df.columns and not df['GOLD_Ret'].isna().all()

#    if has_spx_close:
#        hsi_spx_ratio = df['Close'] / df['SPX_Close']
#        # ✅ 使用参数化的 MID_WINDOW
#        patterns['HSI_SPX_Ratio_Dev'] = (hsi_spx_ratio - hsi_spx_ratio.rolling(MID_WINDOW).mean()) / (hsi_spx_ratio.rolling(MID_WINDOW).mean() + 1e-10)
#        patterns['HSI_SPX_Ratio_Dev'] = patterns['HSI_SPX_Ratio_Dev'].fillna(0)
#    else:
#        patterns['HSI_SPX_Ratio_Dev'] = 0.0

    if has_gold and has_dxy:
        patterns['Gold_DXY_Ratio'] = (df['GOLD_Close'] / (df['DXY_Close'] + 1e-10)).fillna(1.0)
    else:
        patterns['Gold_DXY_Ratio'] = 1.0

    # ===================== 36. 期货领先指标特征 =====================
    # ✅ 使用参数化的 MID_WINDOW 和 LONG_WINDOW
    if has_es and has_spx_close:
        es_spx_ratio = df['ES_Close'] / df['SPX_Close']
        patterns['ES_SPX_Premium'] = (es_spx_ratio - es_spx_ratio.rolling(MID_WINDOW).mean()).fillna(0)
        if has_es_ret:
            patterns['Corr_ES_60'] = df['Log_Ret'].rolling(LONG_WINDOW).corr(df['ES_Ret'].fillna(0)).fillna(0)
        else:
            patterns['Corr_ES_60'] = 0.0
    else:
        patterns['ES_SPX_Premium'] = 0.0
        patterns['Corr_ES_60'] = 0.0

    if has_nq and 'NQ_Ret' in df.columns and not df['NQ_Ret'].isna().all():
        patterns['Corr_NQ_60'] = df['Log_Ret'].rolling(LONG_WINDOW).corr(df['NQ_Ret'].fillna(0)).fillna(0)
    else:
        patterns['Corr_NQ_60'] = 0.0

  #  if has_oil_ret:
  #      patterns['Corr_OIL_60'] = df['Log_Ret'].rolling(LONG_WINDOW).corr(df['OIL_Ret'].fillna(0)).fillna(0)
  #  else:
  #      patterns['Corr_OIL_60'] = 0.0

    if has_gold_ret:
        patterns['Corr_GOLD_60'] = df['Log_Ret'].rolling(LONG_WINDOW).corr(df['GOLD_Ret'].fillna(0)).fillna(0)
    else:
        patterns['Corr_GOLD_60'] = 0.0

    # ===================== 37. 期货期限结构特征 =====================
    if 'ES_SPX_Basis' in df.columns and not df['ES_SPX_Basis'].isna().all():
        patterns['ES_SPX_Basis'] = df['ES_SPX_Basis'].fillna(0)
        patterns['ES_SPX_Basis_Change'] = df['ES_SPX_Basis'].diff().fillna(0)
    else:
        patterns['ES_SPX_Basis'] = 0.0
        patterns['ES_SPX_Basis_Change'] = 0.0

    patterns['RSI_Velocity'] = patterns['RSI_Overbought'].diff().fillna(0)
    patterns['MACD_Hist_Slope'] = hist_s.diff().fillna(0)

    # ===================== 25. 波动率深度特征 =====================
    # ✅ 使用参数化的 MID_WINDOW
    # ⚠️ 关键修复：确保整个计算链条没有使用当日信息，防止未来泄露
    # Realized_Vol_GK 使用 Garman-Klass 估计器，但必须使用历史 OHLC 数据
    log_hl = np.log(df_prep['High'].shift(1) / (df_prep['Low'].shift(1) + 1e-10))  # ⚠️ 使用前一日 High/Low
    log_co = np.log(df_prep['Close'].shift(1) / (df_prep['Open'].shift(1) + 1e-10))  # ⚠️ 使用前一日 Close/Open
    gk_var = 0.5 * log_hl**2 - (2 * np.log(2) - 1) * log_co**2
    gk_series = pd.Series(gk_var, index=df.index)
    realized_vol_gk_raw = np.sqrt(gk_series.rolling(MID_WINDOW, min_periods=10).mean() * 252).fillna(0) * 100
    # ⚠️ 修正波动率滞后：移除第二次 shift(1)，确保该特征代表截止到昨日收盘的波动率，用于预测明日
    # 保持计算 log_hl 和 log_co 时的 shift(1)（使用昨日收盘信息），但移除最终结果的第二次 shift(1)
    patterns['Realized_Vol_GK'] = realized_vol_gk_raw.fillna(0)

    # 波动率期限结构（✅ 使用参数化的 MID_WINDOW）
    realized_vol_ma = patterns['Realized_Vol_GK'].rolling(MID_WINDOW, min_periods=10).mean().shift(1)
    patterns['Vol_Term_Structure'] = patterns['Realized_Vol_GK'] / (realized_vol_ma + 1e-10)

    # B. 下行偏差 (Semi-Variance / Leverage Effect)
    # ✅ 使用参数化的 MID_WINDOW
    returns_series = pd.Series(df['Returns'].values, index=df.index)
    downside_rets = returns_series.clip(upper=0)
    patterns['Downside_Std_20'] = downside_rets.rolling(MID_WINDOW, min_periods=10).std().fillna(0) * np.sqrt(252) * 100

    # C. 波动率的波动 (Vol-of-Vol)
    # ✅ 使用参数化的 SHORT_WINDOW（10日）
    patterns['VVIX_Proxy'] = patterns['Realized_Vol_GK'].rolling(SHORT_WINDOW, min_periods=5).std().fillna(0)

# D. 能量枯竭/爆发指标（梯度防御：clip到可控范围）
 #   patterns['Price_Volume_Efficiency'] = np.abs(df['Returns']) / (patterns['Vol_Rel_Mid'] + 1e-10)
  #  patterns['Price_Volume_Efficiency'] = patterns['Price_Volume_Efficiency'].clip(-5, 5).fillna(0)  # ✅ 梯度防御

    # E. 价格极端位置
    # ✅ 使用参数化的 YEARLY_WINDOW 和 LONG_WINDOW
    roll_max_year = df['Close'].rolling(YEARLY_WINDOW, min_periods=LONG_WINDOW).max().shift(1)
    patterns['Dist_From_Year_High'] = (df['Close'] - roll_max_year) / (roll_max_year + 1e-10)

# F. 收益率分布特征补强（梯度防御：clip到可控范围）
    # ✅ 使用参数化的 MID_WINDOW
    ret_mean_mid = returns_series.rolling(MID_WINDOW, min_periods=10).mean()
    ret_std_mid = returns_series.rolling(MID_WINDOW, min_periods=10).std()
    patterns['Ret_ZScore_20'] = (returns_series - ret_mean_mid) / (ret_std_mid + 1e-10)
    patterns['Ret_ZScore_20'] = patterns['Ret_ZScore_20'].clip(-5, 5).fillna(0)  # ✅ 梯度防御

    # ===================== 38. 新增"黄金特征" =====================
    if has_sse and 'SSE_Ret' in df.columns:
        patterns['AH_Premium'] = (df['Log_Ret'] - df['SSE_Ret'].fillna(0)).fillna(0)
    else:
        patterns['AH_Premium'] = 0.0

    patterns['Overnight_Risk'] = (df['Open'] / df['Close'].shift(1) - 1).fillna(0)

    if 'VIX_Close' in df.columns:
        # ✅ 使用参数化的 LONG_WINDOW 和 MID_WINDOW
        vix_ma_long = df['VIX_Close'].rolling(LONG_WINDOW, min_periods=MID_WINDOW).mean()
        patterns['Vol_Term_Slope'] = (df['VIX_Close'] / (vix_ma_long + 1e-10)).fillna(1.0)
    else:
        patterns['Vol_Term_Slope'] = 1.0

    # ===================== 最终清洗和异常值抑制（健壮性增强）=====================
    # 替换inf和-nan
    patterns = patterns.replace([np.inf, -np.inf], 0).fillna(0)

    # ✅ 使用MAD（中位数绝对偏差）进行异常值抑制（特征平稳化增强 + 全零特征保护）
    for col in patterns.columns:
        col_data = patterns[col]
        col_median = col_data.median()
        mad = (col_data - col_median).abs().median()

        # ✅ 健壮性增强：跳过全零特征（mad == 0），防止产生新的 NaN
        if mad <= 1e-10:
            # 全零特征或常数特征，不执行 clip（避免将全零特征变成 NaN）
            continue

        # 使用MAD进行winsorize：clip到中位数 ± 3 * (1.4826 * MAD)
        mad_scaled = 1.4826 * mad
        lower_bound = col_median - 3 * mad_scaled
        upper_bound = col_median + 3 * mad_scaled
        patterns[col] = col_data.clip(lower_bound, upper_bound)

    # ✅ 逻辑闭环：确保返回前索引已normalize，严防日期偏移
    patterns.index = pd.to_datetime(patterns.index).normalize()

    # ✅ 最终验证：确保索引对齐且无异常值
    if patterns.index.tz is not None:
        patterns.index = patterns.index.tz_localize(None)

    # ✅ 再次检查：确保无 NaN 或 inf（健壮性增强）
    patterns = patterns.replace([np.inf, -np.inf], 0).fillna(0)


    # PyTorch 默认使用 Float32，在特征层提前转换可以节省后续 50% 的内存消耗并加快 Data Loader 速度
    # ✅ 对齐验证补强：转换为 Float32 以节省内存并加快 DataLoader 速度
    patterns = patterns.astype(np.float32)

    # ============================================================
    # 【P5新增】下跌趋势特征
    # ============================================================
    patterns['Drawdown_From_20D_High'] = (df['Close'] / df['Close'].rolling(20).max() - 1) * 100
    patterns['Drawdown_From_60D_High'] = (df['Close'] / df['Close'].rolling(60).max() - 1) * 100
    patterns['Down_Days_5D'] = (df['Close'].diff() < 0).rolling(5).sum()
    patterns['Down_Days_10D'] = (df['Close'].diff() < 0).rolling(10).sum()
    patterns['Large_Down_Move'] = (df['Close'].pct_change() < -0.02).astype(int)
    ma5 = df['Close'].rolling(5).mean()
    ma20 = df['Close'].rolling(20).mean()
    patterns['MA5_Below_MA20'] = (ma5 < ma20).astype(int)
    patterns['Close_Below_MA20'] = (df['Close'] < ma20).astype(int)
    vol_now = df['Realized_Vol_GK'].values
    vol_hist = df['Realized_Vol_GK'].rolling(20).mean().values
    patterns['Vol_Expansion_Down'] = np.where(
        (df['Close'].pct_change() < 0) & (vol_now > 1.5 * np.where(np.isnan(vol_hist), vol_now, vol_hist)), 1, 0
    ).astype(float)
    patterns['Bear_Regime_Score'] = (
        (patterns['Close_Below_MA20'] * 2) + 
        (patterns['MA5_Below_MA20'] * 1) + 
        (patterns['Large_Down_Move'] * 2) +
        ((patterns['Drawdown_From_20D_High'] < -5).astype(int) * 2) +
        (patterns['Vol_Expansion_Down'] * 1)
    )
    print(f"✅ P5新增下跌特征: Drawdown_From_20D_High, Drawdown_From_60D_High, Down_Days_5D, Down_Days_10D, Large_Down_Move, Close_Below_MA20, MA5_Below_MA20, Vol_Expansion_Down, Bear_Regime_Score")
    return patterns

# ===================== 执行特征工程 =====================
print("\n" + "="*60)
print("STEP 1: 特征工程 - 修复版")
print("="*60)

# 关键：在函数外设置 data 状态
cfg.set_status('data', True)

# Fallback: 如果 Realized_Vol_GK 不存在，从 OHLCV 现场计算
if 'Realized_Vol_GK' not in df.columns:
    print("⚠️ Realized_Vol_GK 不存在，现场计算 Garman-Klass 波动率...")
    hl = df['High'] - df['Low']
    co = df['Close'] - df['Open']
    log_hl = np.log(df['High'] / df['Low'])
    log_co = np.log(df['Close'] / df['Open'])
    df['Realized_Vol_GK'] = np.sqrt(0.5 * hl**2 - (2 * np.log(2) - 1) * co**2) * np.sqrt(252)
    df['Realized_Vol_GK'] = df['Realized_Vol_GK'].fillna(0)
    print(f"✅ Realized_Vol_GK 计算完成，范围: [{df['Realized_Vol_GK'].min():.4f}, {df['Realized_Vol_GK'].max():.4f}]")

patterns_df = detect_trading_patterns(df, config=cfg)

# 截断与校验（保留你原逻辑）
YEARLY_WINDOW_FOR_TRUNCATE = 252
valid_start_idx = YEARLY_WINDOW_FOR_TRUNCATE

if len(patterns_df) > valid_start_idx:
    patterns_df = patterns_df.iloc[valid_start_idx:]
    df = df.iloc[valid_start_idx:].copy()
    print(f"\n✅ 已截断前 {valid_start_idx} 行")
else:
    print(f"\n⚠️ 数据长度不足 {valid_start_idx}，保留全部数据")

assert patterns_df.shape[1] > 0, "❌ No patterns generated!"
assert not patterns_df.isnull().values.any(), "❌ Patterns contain NaN values!"
assert len(patterns_df) == len(df), f"❌ Index mismatch: patterns ({len(patterns_df)}) != df ({len(df)})"

cfg.set_status('feature', True)
print(f"\n✅ STEP 1 完成！状态: {cfg.status}")


STEP 1: 特征工程 - 修复版
⚠️ Realized_Vol_GK 不存在，现场计算 Garman-Klass 波动率...
✅ Realized_Vol_GK 计算完成，范围: [125.1005, 18802.7240]
✅ P5新增下跌特征: Drawdown_From_20D_High, Drawdown_From_60D_High, Down_Days_5D, Down_Days_10D, Large_Down_Move, Close_Below_MA20, MA5_Below_MA20, Vol_Expansion_Down, Bear_Regime_Score

✅ 已截断前 252 行

✅ STEP 1 完成！状态: {'data': True, 'feature': True, 'model': False}


In [6]:
df.tail()

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Log_Ret,Returns,VIX_Close,...,SSE_Close,USDCNH_Close,US10Y_Close,GOLD_Close,OIL_Close,DXY_Close,HKD_Close,SPX_Ret,SSE_Ret,Realized_Vol_GK
2026-05-28,25256.361268,25256.361268,24721.516307,25094.538169,1.564150e+05,0.0,0.0,-0.008216,-0.008182,16.97,...,4077.2771,6.77833,4.586,4457.66,88.914,99.173,7.83419,-0.003611,0.0,5787.423778
2026-05-29,25149.266203,25268.476839,24953.553302,25140.570153,2.113460e+05,0.0,0.0,0.001833,0.001834,16.02,...,4077.2771,6.77632,4.586,4455.96,89.003,99.013,7.83488,0.004930,0.0,3533.966566
2026-06-01,25255.238105,25529.702222,25068.653455,25212.416750,1.666520e+05,0.0,0.0,0.002854,0.002858,15.34,...,4077.2771,6.76354,4.586,4539.83,87.679,98.869,7.83694,0.004720,0.0,5157.985180
2026-06-02,25256.773878,26029.817496,25256.773878,25904.804023,1.498720e+05,0.0,0.0,0.027092,0.027462,16.08,...,4077.2771,6.76393,4.586,4536.44,89.674,98.988,7.83582,0.002653,0.0,5866.625751
2026-06-03,25964.892967,25964.892967,25586.306116,25620.013021,3.229210e+11,0.0,0.0,-0.011055,-0.010994,16.05,...,4077.2771,6.76175,4.586,4503.34,91.698,99.059,7.83698,0.000684,0.0,2545.734074


In [7]:
# ============================================================
# STEP 1.5: 两阶段动态特征筛选系统 (完整无删减版 + 完美适配单例配置)
# ============================================================
import json
import gc
import shutil 
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

# 提前导入所有需要的机器学习库，避免中途报错
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

print("=" * 60)
print("STEP 1.5: 两阶段动态特征筛选系统 (基于 Run_ID 物理隔离)")
print("=" * 60)

# 1. 依托全局单例配置中心 (cfg) 动态获取路径，彻底告别硬编码
run_dir = cfg.TODAY_RUN_DIR
today_lock_path = cfg.SELECTED_FEATURES_PATH  # 默认的特征锁文件 (model_config.json 旁边)
latest_path = run_dir / "selected_features_latest.json"

# 2. 当次操作的版本水印 (精确到分钟，用于区分同一次 run 下的多次特征尝试)
version_tag = datetime.now().strftime("%Y%m%d_%H%M")
load_target_path = today_lock_path if today_lock_path.exists() else latest_path

# 防止 Notebook 环境下变量未定义报错
assert isinstance(RUN_FS, bool), f"RUN_FS 必须是 bool，实际是 {type(RUN_FS).__name__}: {RUN_FS}"
do_feature_selection = RUN_FS  # 严格使用 STEP -1 解析的布尔值
print(f"[STEP 1.5] RUN_FS={do_feature_selection} (type=bool) -> {'强制重新特征选择' if do_feature_selection else '加载历史特征文件'}")

# ============================================================
# 🎯 0. 前置交互逻辑：历史版本扫描与加载
# ============================================================
if not do_feature_selection:
    # 只扫描带有时间戳特征版本文件
    available_files = list(run_dir.glob("selected_features_*.json"))

    # 场景A：冷启动（当前 run_dir 里没有任何特征文件）
    if not available_files and not today_lock_path.exists() and not latest_path.exists():
        print(f"\n⚠️ [冷启动] 当前隔离沙盒 ({run_dir.name}) 内无特征配置。")
        if os.getenv("RUN_AUTO", "0") == "1":
            # Auto mode: force feature selection for new ticker
            print("[AUTO] 无历史特征，强制执行特征筛选...")
            do_feature_selection = True
        else:
            print("请选择操作：")
            print("  [1 或 回车] 立即执行全新的特征筛选 (耗时较长)")
            print("  [粘贴路径] 直接跨时空加载历史特征")
            print("  [N] 退出程序")
            
            ans = input("👉 请选择: ").strip()
            if ans.upper() == 'N':
                raise FileNotFoundError("❌ 操作中止。")
            elif ans == '1' or ans == '':
                print("✅ 收到指令：准备进行全新的特征筛选...")
                do_feature_selection = True
            else:
                custom_path = Path(ans)
                if custom_path.exists() and custom_path.is_file():
                    print(f"✅ 跨沙盒加载成功: {custom_path}")
                    load_target_path = custom_path
                    do_feature_selection = False
                    shutil.copy2(custom_path, today_lock_path)
                    shutil.copy2(custom_path, latest_path)
                else:
                    raise FileNotFoundError(f"❌ 路径无效: {custom_path}")
    # 场景B：当前沙盒内已有特征配置（即 else 分支）
            print(f"\n📂 发现当前沙盒 ({run_dir.name}) 已有配置：")
        if today_lock_path.exists(): 
            print(f"   - 锁定基准: {today_lock_path.name}")
        
        versions = sorted([f.stem.replace("selected_features_", "") for f in available_files if "latest" not in f.name], reverse=True)
        if versions:
            print("   - 沙盒内历史尝试:")
            for v in versions[:5]: 
                print(f"     * {v}")

        print("\n⏸️ [加载确认]")
        print("👉 按【回车】使用当前沙盒默认配置")
        print("👉 输入 'R' 强制重新进行特征筛选")
        print("👉 或输入历史完整路径跨沙盒加载")
    if os.getenv("RUN_AUTO", "0") != "1":  # skip in auto mode
        ans = "" if os.getenv("RUN_AUTO", "0") == "1" else input("请输入你的选择: ").strip()

        if ans.upper() == 'R':
            print("✅ 用户指令: 强制重新进行特征筛选。")
            do_feature_selection = True
        elif ans:
            direct_path = Path(ans)
            if direct_path.exists() and direct_path.is_file():
                print(f"✅ 识别到完整文件路径，将直接加载: {direct_path}")
                load_target_path = direct_path
                shutil.copy2(direct_path, today_lock_path)
                shutil.copy2(direct_path, latest_path)
            else:
                custom_path = run_dir / f"selected_features_{ans}.json"
                if custom_path.exists():
                    load_target_path = custom_path
                else:
                    print(f"❌ 找不到版本 '{ans}'，回退至默认。")

# ============================================================
# 分支 A：执行全量特征筛选 (生成新版本，写入当前沙盒)
# ============================================================
if do_feature_selection:
    print("\n🚀 [TRAIN 模式] 启动全量 MI+PI 特征筛选计算 (可能需要数分钟)...")
    
    # 依赖检查：确保已经跑过 STEP 1 (Data)
    cfg.check_dependency('feature')

    # --- 阶段0：准备对齐的 Target 数据 ---
    # 预测目标 1：未来 1 天的收益率
    target_returns_1d = df['Close'].pct_change().shift(-1) * 100
    target_returns_1d.name = 'Target_Returns'

    # 预测目标 2：波动率 (优先使用 GK 波动率，否则计算一个简单的代理)
    if 'Realized_Vol_GK' in patterns_df.columns:
        target_volatility_temp = patterns_df['Realized_Vol_GK'].shift(-1)
    else:
        log_hl = np.log(df['High'] / (df['Low'] + 1e-10))
        log_co = np.log(df['Close'].shift(1) / (df['Open'].shift(1) + 1e-10))
        gk_var = 0.5 * log_hl.shift(1) ** 2 - (2 * np.log(2) - 1) * log_co ** 2
        target_volatility_temp = np.sqrt(pd.Series(gk_var).rolling(int(cfg.SEQUENCE_LENGTH)).mean() * 252).shift(-1) * 100

    target_volatility_temp.name = 'Target_Volatility'

    # 合并特征和目标并丢弃缺失值
    combined_df = pd.concat([patterns_df, target_returns_1d, target_volatility_temp], axis=1, join='inner').dropna()
    
    if len(combined_df) == 0:
        raise ValueError("❌ 错误：合并后的数据框为空，请检查 patterns_df 和目标变量的数据对齐")

    X_selection = combined_df.drop(columns=['Target_Returns', 'Target_Volatility'])
    y_returns = combined_df['Target_Returns'].values
    y_volatility = combined_df['Target_Volatility'].values
    feat_names = X_selection.columns.tolist()

    print(f"   ✓ 数据对齐完成，共 {len(combined_df)} 条有效样本，{len(feat_names)} 个初始特征。")

    # --- 阶段1：MI (互信息) 快速初筛 ---
    print("   ⏳ 正在执行阶段 1/3: 互信息(MI)非线性相关性初筛...")
    mi_returns = mutual_info_regression(X_selection.values, y_returns, random_state=42, n_neighbors=3)
    mi_volatility = mutual_info_regression(X_selection.values, y_volatility, random_state=42, n_neighbors=3)

    mi_df = pd.DataFrame({'Feature': feat_names, 'MI_Returns': mi_returns, 'MI_Volatility': mi_volatility})
    # 各取前 80 名并取并集
    features_after_mi = list(set(mi_df.nlargest(80, 'MI_Returns')['Feature']) | set(mi_df.nlargest(80, 'MI_Volatility')['Feature']))
    X_filtered = X_selection[features_after_mi]

    # --- 阶段2：共线性剪枝 ---
    print("   ⏳ 正在执行阶段 2/3: 特征共线性剪枝...")
    corr_matrix = X_filtered.corr().abs()
    upper_tri = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    high_corr_mask = (corr_matrix > cfg.CORRELATION_THRESHOLD) & upper_tri

    features_to_remove = set()
    for i in range(len(corr_matrix.columns)):
        for j in range(i + 1, len(corr_matrix.columns)):
            if high_corr_mask.iloc[i, j]:
                f1, f2 = corr_matrix.columns[i], corr_matrix.columns[j]
                # 保留方差更大的那个特征
                if X_filtered[f1].var() >= X_filtered[f2].var(): 
                    features_to_remove.add(f2)
                else: 
                    features_to_remove.add(f1)

    features_for_pi = sorted(list(set(features_after_mi) - features_to_remove))
    X_pi = X_selection[features_for_pi]

    # --- 阶段3：PI (排列重要性) 深度评估 ---
    print(f"   ⏳ 正在执行阶段 3/3: 随机森林 PI 深度评估 (评估 {len(features_for_pi)} 个候选特征)...")
    gc.collect()

    rf_ret = RandomForestRegressor(n_estimators=cfg.RF_N_ESTIMATORS, max_depth=cfg.RF_MAX_DEPTH, random_state=42, n_jobs=cfg.N_JOBS)
    rf_vol = RandomForestRegressor(n_estimators=cfg.RF_N_ESTIMATORS, max_depth=cfg.RF_MAX_DEPTH, random_state=42, n_jobs=cfg.N_JOBS)

    rf_ret.fit(X_pi.values, y_returns)
    rf_vol.fit(X_pi.values, y_volatility)

    pi_ret = permutation_importance(rf_ret, X_pi.values, y_returns, n_repeats=cfg.PI_REPEATS, random_state=42, n_jobs=cfg.N_JOBS, scoring='r2')
    pi_vol = permutation_importance(rf_vol, X_pi.values, y_volatility, n_repeats=cfg.PI_REPEATS, random_state=42, n_jobs=cfg.N_JOBS, scoring='r2')

    pi_df = pd.DataFrame({'Feature': features_for_pi, 'PI_Ret': pi_ret.importances_mean, 'PI_Vol': pi_vol.importances_mean})
    # 结合两者的重要性 (收益率占 60%，波动率占 40%)
    pi_df['Combined'] = (0.6 * pi_df['PI_Ret'] / (pi_df['PI_Ret'].max() + 1e-10) + 0.4 * pi_df['PI_Vol'] / (pi_df['PI_Vol'].max() + 1e-10))

    # --- 阶段4：逻辑截断 ---
    # 动态计算应保留的基础特征数，不要超过模型隐藏层大小
    base_n = max(45, min(80, cfg.HIDDEN_SIZE))
    features_union = list(set(pi_df.nlargest(base_n, 'PI_Ret')['Feature']) | set(pi_df.nlargest(base_n, 'PI_Vol')['Feature']))
    
    if len(features_union) > base_n * 1.5:
        final_selected_features = pi_df.nlargest(base_n, 'Combined')['Feature'].tolist()
    else:
        final_selected_features = features_union

    # --- 阶段5：版本化保存 (写入 cfg 所属的 TODAY_RUN_DIR) ---
    selected_features_metadata = {
        'version': version_tag,
        'ticker_symbol': cfg.TICKER_SYMBOL,
        'selection_date': pd.Timestamp.now().isoformat(),
        'run_dir': str(run_dir),
        'final_feature_count': len(final_selected_features),
        'selected_features': final_selected_features
    }

    # 1. 保存时间戳快照
    versioned_path = run_dir / f"selected_features_{version_tag}.json"
    with open(versioned_path, 'w', encoding='utf-8') as f:
        json.dump(selected_features_metadata, f, indent=2, ensure_ascii=False)
        
    # 2. 覆盖默认锁定基准
    with open(today_lock_path, 'w', encoding='utf-8') as f:
        json.dump(selected_features_metadata, f, indent=2, ensure_ascii=False)
        
    # 3. 更新最新指针
    with open(latest_path, 'w', encoding='utf-8') as f:
        json.dump(selected_features_metadata, f, indent=2, ensure_ascii=False)

    print(f"\n✅ 特征筛选已完成！最终入选 {len(final_selected_features)} 个核心特征。")
    print(f"   📂 档案已全部存入: {run_dir}/")

    # 内存清理
    del rf_ret, rf_vol, pi_ret, pi_vol
    gc.collect()

# ============================================================
# 分支 B：[推断/热加载模式] 读取并加载选定的特征版本
# ============================================================
else:
    print(f"\n⏩ 正在加载特征基准配置: {load_target_path.name} ...")
    
    if not load_target_path.exists():
        raise FileNotFoundError(f"❌ 找不到特征配置文件 {load_target_path}！")
        
    with open(load_target_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
        
    final_selected_features = metadata['selected_features']
    print(f"✅ 加载成功！共包含 {len(final_selected_features)} 个特征。")

# ============================================================
# 最终输出：更新 patterns_df，并将状态打回 GlobalConfig
# ============================================================
print("\n" + "=" * 60)
print("开始执行特征矩阵裁剪与状态同步...")

missing_features = [f for f in final_selected_features if f not in patterns_df.columns]
if missing_features:
    print(f"⚠️ 警告: 当前数据框中缺失以下特征，将自动剔除:\n{missing_features}")
    final_selected_features = [f for f in final_selected_features if f in patterns_df.columns]
    print(f"   剩余有效特征数: {len(final_selected_features)}")

# 执行特征裁剪，严格保证 DataFrame 的列就是入选的特征
patterns_df = patterns_df[final_selected_features].copy()

# 危险特征预警（确保不会泄露未来信息）
risky_features = ['VIX_Quantile', 'Price_Rank_252', 'VIX_Quantile_Rank', 'US10Y_Quantile_Rank']
risky_selected = [f for f in risky_features if f in final_selected_features]
if risky_selected:
    print(f"ℹ️ 检测到存在排序类特征 {risky_selected}，已确认相关逻辑不构成未来数据泄露。")

# 将操作状态告知全局配置中心
cfg.set_status('feature', True) 

print(f"✅ STEP 1.5 完毕！特征矩阵已锁定。")
print(f"   当前 patterns_df 维度: {patterns_df.shape}")
print("=" * 60)

STEP 1.5: 两阶段动态特征筛选系统 (基于 Run_ID 物理隔离)
[STEP 1.5] RUN_FS=False (type=bool) -> 加载历史特征文件

⚠️ [冷启动] 当前隔离沙盒 (run_230437) 内无特征配置。
请选择操作：
  [1 或 回车] 立即执行全新的特征筛选 (耗时较长)
  [粘贴路径] 直接跨时空加载历史特征
  [N] 退出程序
✅ 收到指令：准备进行全新的特征筛选...

📂 发现当前沙盒 (run_230437) 已有配置：

⏸️ [加载确认]
👉 按【回车】使用当前沙盒默认配置
👉 输入 'R' 强制重新进行特征筛选
👉 或输入历史完整路径跨沙盒加载

🚀 [TRAIN 模式] 启动全量 MI+PI 特征筛选计算 (可能需要数分钟)...
   ✓ 数据对齐完成，共 4669 条有效样本，125 个初始特征。
   ⏳ 正在执行阶段 1/3: 互信息(MI)非线性相关性初筛...
   ⏳ 正在执行阶段 2/3: 特征共线性剪枝...
   ⏳ 正在执行阶段 3/3: 随机森林 PI 深度评估 (评估 86 个候选特征)...

✅ 特征筛选已完成！最终入选 83 个核心特征。
   📂 档案已全部存入: model_artifacts/HSI/20260611/run_230437/

开始执行特征矩阵裁剪与状态同步...
ℹ️ 检测到存在排序类特征 ['VIX_Quantile', 'Price_Rank_252', 'VIX_Quantile_Rank', 'US10Y_Quantile_Rank']，已确认相关逻辑不构成未来数据泄露。
✅ STEP 1.5 完毕！特征矩阵已锁定。
   当前 patterns_df 维度: (4670, 83)


In [8]:
# ============================================================
# STEP 2: 数据预处理与张量构建 (Train/Inference 双轨架构 + 冷启动自愈)
# ============================================================
import pickle
import json
import logging
import torch
import os
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# ✅ 依赖检查
cfg.check_dependency('feature')

# --- 1. 路径重构：使用 GlobalConfig 的 TODAY_RUN_DIR ---
# 保证特征、预处理参数、模型都在同一个日期文件夹下，防止版本错乱
SCALER_FEAT_PATH = cfg.TODAY_RUN_DIR / 'scaler_features.pkl'
SCALER_RET_PATH = cfg.TODAY_RUN_DIR / 'scaler_returns.pkl'
SCALER_VOL_PATH = cfg.TODAY_RUN_DIR / 'scaler_volatility.pkl'
CLIPPING_BOUNDS_PATH = cfg.TODAY_RUN_DIR / 'clipping_bounds.pkl'
CURRENT_RUN_FEATURE_JSON = cfg.TODAY_RUN_DIR / 'selected_features_applied.json' # 今天应用的特征名单

# 确保当天目录存在
cfg.TODAY_RUN_DIR.mkdir(parents=True, exist_ok=True)
cfg.LOGS_DIR.mkdir(parents=True, exist_ok=True)

# --- 2. 日志配置 ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(cfg.LOGS_DIR / f'step2_data_preprocessing_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.log', encoding='utf-8'),
        logging.StreamHandler()
    ],
    force=True
)
logger = logging.getLogger(__name__)

# --- 3. 运行模式定义 ---
try:
    run_mode = str(globals().get("RUN_MODE", os.getenv("RUN_MODE", "INFERENCE"))).strip().upper()
except NameError:
    run_mode = "INFERENCE"

print("="*60)
print(f"STEP 2: Data Preprocessing Pipeline [{run_mode} MODE]")
print(f"Data Root: {cfg.TODAY_RUN_DIR}")
print("="*60)


# ============================================================
# 🎯 动态自愈：INFERENCE 固定读取历史模型目录
# ============================================================
if run_mode == "INFERENCE":
    try:
        model_run_dir = cfg.get_latest_model_run_dir()
        print(f"ℹ️ [STEP 2] INFERENCE 使用历史模型目录: {model_run_dir}")
        CLIPPING_BOUNDS_PATH = model_run_dir / 'clipping_bounds.pkl'
        SCALER_FEAT_PATH = model_run_dir / 'scaler_features.pkl'
        SCALER_RET_PATH = model_run_dir / 'scaler_returns.pkl'
        SCALER_VOL_PATH = model_run_dir / 'scaler_volatility.pkl'
        CURRENT_RUN_FEATURE_JSON = model_run_dir / 'selected_features_applied.json'

        required = [CLIPPING_BOUNDS_PATH, SCALER_FEAT_PATH]
        missing = [str(x) for x in required if not x.exists()]
        if missing:
            raise FileNotFoundError(f"❌ INFERENCE 缺少必要文件: {missing}")
    except Exception as e:
        raise FileNotFoundError(
            f"❌ INFERENCE 无法加载历史模型目录（{e}）。请先 TRAIN 生成基础模型，或让 workflow fallback TRAIN 接管。"
        )

# ============================================================
# 轨道路线 A: 全量训练模式 (TRAIN)
# ============================================================
if run_mode == "TRAIN":
    logger.info("🚀 [TRAIN 模式] 执行全量数据对齐、截断与归一化...")

    # ============================================================
    # 🔍 【核心新增】跨日期加载历史特征 (支持用户动态输入)
    # ============================================================
    print("\n" + "-"*50)
    print("🔍 历史特征文件 (selected_features) 加载设置")
    print("支持跨日期、跨 run_dir 加载历史特征文件用于本次训练。")
    user_feature_path = "" if os.getenv("RUN_AUTO", "0") == "1" else input("👉 请输入 selected_features*.json 的绝对或相对路径\n(留空按回车则系统将自动搜索并加载最新的特征文件): ").strip()
    
    target_feature_file = None
    
    # 1. 尝试使用用户输入的路径
    if user_feature_path:
        if os.path.exists(user_feature_path):
            target_feature_file = Path(user_feature_path)
            logger.info(f"✅ 将使用指定的特征文件: {target_feature_file}")
        else:
            logger.warning(f"⚠️ 您输入的路径不存在: '{user_feature_path}'，系统将回退到自动搜索机制！")
            
    # 2. 回退：自动寻找最新的特征文件
    if target_feature_file is None:
        try:
            # 向上找到所有的历史运行根目录，查找所有的 selected_features*.json
            runs_dir = cfg.SAVE_DIR
            feature_files = list(runs_dir.rglob("selected_features*.json"))
            
            # 排除掉今天刚刚备份产生的同名文件（防止无限死循环读取自己的备份）
            feature_files = [f for f in feature_files if f != CURRENT_RUN_FEATURE_JSON]
            
            if feature_files:
                target_feature_file = max(feature_files, key=os.path.getmtime)
        except Exception as e:
            logger.warning(f"⚠️ 跨目录检索特征文件时出错: {e}")

    # 3. 加载选中的特征
    if target_feature_file and os.path.exists(target_feature_file):
        logger.info(f"🔍 最终加载的历史特征文件: {target_feature_file}")
        with open(target_feature_file, 'r', encoding='utf-8') as f:
            feature_data = json.load(f)
            
        # 兼容不同的 key 名称
        selected_features = feature_data.get('selected_features', []) or feature_data.get('features', [])
            
        if selected_features:
            valid_features = [f for f in selected_features if f in patterns_df.columns]
            logger.info(f"✅ 从历史文件加载并匹配到 {len(valid_features)} 个有效特征。")
            
            # 强制保留构建目标相关的列
            if 'Realized_Vol_GK' in patterns_df.columns and 'Realized_Vol_GK' not in valid_features:
                valid_features.append('Realized_Vol_GK')
                
            patterns_df = patterns_df[valid_features].copy()
            
            # 📂 路径一致性：将其备份至当天的运行目录，确保"同一天产物"规则严密
            with open(CURRENT_RUN_FEATURE_JSON, 'w', encoding='utf-8') as f:
                json.dump({'selected_features': valid_features, 'source': str(target_feature_file)}, f, indent=4)
            logger.info(f"📂 已将本次使用的特征名单备份至当天目录: {CURRENT_RUN_FEATURE_JSON}")
        else:
            logger.warning("⚠️ 历史特征文件内容为空，继续使用全量特征。")
    else:
        logger.info("ℹ️ 未发现或未提供任何有效的 selected_features*.json，将使用当前模式全量特征。")

    print("-" * 50 + "\n")

    # 1. 构建目标变量
    target_returns = pd.DataFrame({
        f'Target_Returns_{h}D': (df['Close'].shift(-h) / df['Close'] - 1) * 100
        for h in cfg.PREDICTION_HORIZONS
    })

    if 'Realized_Vol_GK' not in patterns_df.columns:
        raise ValueError("patterns_df 中缺少 'Realized_Vol_GK' 列，请确保特征工程已正确完成")
    target_volatility = patterns_df['Realized_Vol_GK'].copy()
    target_volatility.name = 'Target_Volatility'
    # ============================================================
    # 【P2新增】三分类趋势标签 (Downtrend/Neutral/Uptrend)
    # ============================================================
    # 配置：可调阈值（默认 -1% / +1%）
    TREND_DOWN_THRESHOLD = -1.0   # 5日收益率 < -1% → 0 下跌
    TREND_UP_THRESHOLD   =  1.0   # 5日收益率 > +1% → 2 上涨
    # 中间区间 → 1 震荡
    future_ret_5d = (df['Close'].shift(-5) / df['Close'] - 1) * 100  # pct
    trend_labels = pd.Series(index=future_ret_5d.index, dtype=int)
    trend_labels[:] = 1  # 默认震荡
    trend_labels[future_ret_5d < TREND_DOWN_THRESHOLD] = 0  # 下跌
    trend_labels[future_ret_5d > TREND_UP_THRESHOLD]   = 2  # 上涨
    # 末尾5条无法计算5日future，返回NaN → 标记为震荡（1）保守处理
    trend_labels = trend_labels.fillna(1).astype(int)
    target_trend = pd.DataFrame({'Target_Trend': trend_labels})
    print(f"✅ 趋势标签分布: 0(下跌)={(trend_labels==0).sum()}, 1(震荡)={(trend_labels==1).sum()}, 2(上涨)={(trend_labels==2).sum()}")
    print(f"   阈值: 下阈={TREND_DOWN_THRESHOLD}%, 上阈={TREND_UP_THRESHOLD}%")


    # 2. 数据对齐与合并
    combined_df = pd.concat([patterns_df, target_returns, target_volatility, target_trend], axis=1, join='inner')
    combined_df = combined_df.replace([np.inf, -np.inf], np.nan).sort_index().dropna()

    if len(combined_df) == 0:
        raise ValueError("合并后的数据为空，请检查数据质量")

    # 3. 训练/测试集划分
    split_idx = int(len(combined_df) * cfg.TRAIN_TEST_SPLIT_RATIO)
    train_df = combined_df.iloc[:split_idx].copy()
    test_df = combined_df.iloc[split_idx:].copy()

    # 4. 分位数截断边界计算 (仅基于训练集)
    train_features = train_df[patterns_df.columns].values
    train_returns = train_df[target_returns.columns].values
    train_vol = train_df[['Target_Volatility']].values

    clipping_bounds = {
        'features_lower': np.percentile(train_features, cfg.WINSORIZATION_LOWER * 100, axis=0),
        'features_upper': np.percentile(train_features, cfg.WINSORIZATION_UPPER * 100, axis=0),
        'returns_lower': np.percentile(train_returns, cfg.WINSORIZATION_LOWER * 100, axis=0),
        'returns_upper': np.percentile(train_returns, cfg.WINSORIZATION_UPPER * 100, axis=0),
        'vol_lower': np.percentile(train_vol, cfg.WINSORIZATION_LOWER * 100),
        'vol_upper': np.percentile(train_vol, cfg.WINSORIZATION_UPPER * 100),
        'feature_names': patterns_df.columns.tolist() # 核心防御：严格记录并锁定此刻使用的特征顺序与名单
    }

    with open(CLIPPING_BOUNDS_PATH, 'wb') as f:
        pickle.dump(clipping_bounds, f)
    logger.info(f"✅ 截断边界已保存至: {CLIPPING_BOUNDS_PATH}")

    # 应用截断
    combined_df.loc[:, patterns_df.columns] = np.clip(
        combined_df[patterns_df.columns].values, 
        clipping_bounds['features_lower'], 
        clipping_bounds['features_upper']
    )
    for i, col in enumerate(target_returns.columns):
        combined_df.loc[:, col] = np.clip(
            combined_df[col].values, 
            clipping_bounds['returns_lower'][i], 
            clipping_bounds['returns_upper'][i]
        )
    combined_df.loc[:, 'Target_Volatility'] = np.clip(
        combined_df['Target_Volatility'].values, 
        clipping_bounds['vol_lower'], 
        clipping_bounds['vol_upper']
    )

    # 重新拆分
    train_df = combined_df.iloc[:split_idx].copy()
    test_df = combined_df.iloc[split_idx:].copy()

    # 5. 标准化 (Fit & Transform)
    scaler_features = StandardScaler()
    train_features_scaled = scaler_features.fit_transform(train_df[patterns_df.columns].values)
    test_features_scaled = scaler_features.transform(test_df[patterns_df.columns].values)

    scaler_target_returns = {h: StandardScaler() for h in cfg.PREDICTION_HORIZONS}
    train_returns_scaled = np.zeros_like(train_df[target_returns.columns].values)
    test_returns_scaled = np.zeros_like(test_df[target_returns.columns].values)

    for i, h in enumerate(cfg.PREDICTION_HORIZONS):
        col_name = target_returns.columns[i]
        train_returns_scaled[:, i:i+1] = scaler_target_returns[h].fit_transform(train_df[[col_name]].values)
        test_returns_scaled[:, i:i+1] = scaler_target_returns[h].transform(test_df[[col_name]].values)

    train_vol_raw = train_df[['Target_Volatility']].values
    test_vol_raw = test_df[['Target_Volatility']].values
    if getattr(cfg, 'VOLATILITY_SCALING_METHOD', 'standard') == 'log_standard':
        train_vol_raw, test_vol_raw = np.log1p(train_vol_raw), np.log1p(test_vol_raw)

    scaler_target_volatility = StandardScaler()
    train_vol_scaled = scaler_target_volatility.fit_transform(train_vol_raw)
    test_vol_scaled = scaler_target_volatility.transform(test_vol_raw)

    # 6. 保存所有标准化器 (保存在 TODAY_RUN_DIR)
    with open(SCALER_FEAT_PATH, 'wb') as f: pickle.dump(scaler_features, f)
    with open(SCALER_RET_PATH, 'wb') as f: pickle.dump(scaler_target_returns, f)
    with open(SCALER_VOL_PATH, 'wb') as f: pickle.dump(scaler_target_volatility, f)
    logger.info(f"✅ 标准化器已锁定并保存至当天目录")

    # 7. 构建3D序列张量
    def create_sequences(X, y, seq_len):
        Xs = [X[i:(i + seq_len)] for i in range(len(X) - seq_len)]
        ys = [y[i + seq_len] for i in range(len(y) - seq_len)]
        return np.array(Xs), np.array(ys)

    train_trend_raw = train_df[['Target_Trend']].values
    train_targets = np.concatenate([train_returns_scaled, train_vol_scaled, train_trend_raw], axis=1)
    X_train, y_train = create_sequences(train_features_scaled, train_targets, cfg.SEQUENCE_LENGTH)

    test_trend_raw = test_df[['Target_Trend']].values
    test_targets = np.concatenate([test_returns_scaled, test_vol_scaled, test_trend_raw], axis=1)
    y_test_placeholder = np.full((cfg.SEQUENCE_LENGTH, test_targets.shape[1]), np.nan)
    y_test_source = np.concatenate([y_test_placeholder, test_targets], axis=0)
    X_test_source = np.concatenate([train_features_scaled[-cfg.SEQUENCE_LENGTH:], test_features_scaled], axis=0)
    X_test, y_test = create_sequences(X_test_source, y_test_source, cfg.SEQUENCE_LENGTH)

    X_train_tensor = torch.FloatTensor(X_train).to(cfg.device)
    y_train_tensor = torch.FloatTensor(y_train).to(cfg.device)
    X_test_tensor = torch.FloatTensor(X_test).to(cfg.device)
    y_test_tensor = torch.FloatTensor(y_test).to(cfg.device)

    print("=" * 40)
    print("FINAL DATA SHAPES (TRAIN MODE)")
    print("=" * 40)
    print(f"X_train: {X_train_tensor.shape} | y_train: {y_train_tensor.shape}")
    print(f"X_test:  {X_test_tensor.shape} | y_test:  {y_test_tensor.shape}")

# ============================================================
# 轨道路线 B: 每日推断模式 (INFERENCE)
# ============================================================
else:
    logger.info("⏩ [INFERENCE 模式] 跳过目标变量与数据划分，专注最新数据处理...")

    # 1. 加载历史配置与模型 (从 TODAY_RUN_DIR 加载)
    with open(CLIPPING_BOUNDS_PATH, 'rb') as f: clipping_bounds = pickle.load(f)
    with open(SCALER_FEAT_PATH, 'rb') as f: scaler_features = pickle.load(f)

    # 2. 特征兼容与对齐：优先保证推理可运行，并记录差异
    expected_features = clipping_bounds['feature_names']
    missing = [f for f in expected_features if f not in patterns_df.columns]
    extra = [f for f in patterns_df.columns if f not in expected_features]

    if missing:
        for f in missing:
            patterns_df[f] = 0.0
        print(f"⚠️ 缺失特征 {len(missing)} 个，已自动补齐为 0.0: {missing}")

    if extra:
        print(f"ℹ️ 额外特征 {len(extra)} 个，将在推理前忽略。")

    compat_report = {
        'expected_feature_count': len(expected_features),
        'current_feature_count': len(patterns_df.columns),
        'missing_filled_with_zero': missing,
        'extra_ignored': extra
    }
    try:
        compat_report_path = cfg.TODAY_RUN_DIR / 'feature_compat_report.json'
        with open(compat_report_path, 'w', encoding='utf-8') as f:
            json.dump(compat_report, f, ensure_ascii=False, indent=2)
        print(f"✅ 特征兼容报告已保存: {compat_report_path}")
    except Exception as e:
        print(f"⚠️ 特征兼容报告保存失败: {e}")

    # 按历史模型训练时的特征顺序严格对齐
    patterns_df = patterns_df[expected_features].copy()

    # 3. 数据截断与归一化
    if len(patterns_df) < cfg.SEQUENCE_LENGTH:
        raise ValueError(f"❌ 数据量不足！需要 {cfg.SEQUENCE_LENGTH} 天数据，目前仅有 {len(patterns_df)} 天。")

    recent_features_df = patterns_df.iloc[-cfg.SEQUENCE_LENGTH:].copy()

    # 提取今日原始特征快照
    today_snapshot = recent_features_df.iloc[-1:]
    snapshot_path = cfg.TODAY_RUN_DIR / f'inference_snapshot_{today_snapshot.index[-1].strftime("%Y%m%d")}.csv'
    today_snapshot.to_csv(snapshot_path)
    logger.info(f"✅ 成功提取今日原始特征快照至: {snapshot_path}")

    recent_features_clipped = np.clip(recent_features_df.values, clipping_bounds['features_lower'], clipping_bounds['features_upper'])
    recent_features_scaled = scaler_features.transform(recent_features_clipped)

    # 4. 构建推断专用张量
    X_inference = np.expand_dims(recent_features_scaled, axis=0)
    X_inference_tensor = torch.FloatTensor(X_inference).to(cfg.device)

    print("=" * 40)
    print("FINAL DATA SHAPES (INFERENCE MODE)")
    print("=" * 40)
    print(f"X_inference_tensor: {X_inference_tensor.shape}")
    print(f"目标日期: {recent_features_df.index[-1].strftime('%Y-%m-%d')}")

# ============================================================
# 公共尾部处理
# ============================================================
# 保存波动率元数据
with open(cfg.VOLATILITY_METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump({
        'scaling_method': getattr(cfg, 'VOLATILITY_SCALING_METHOD', 'standard'),
        'requires_log_transform': getattr(cfg, 'VOLATILITY_SCALING_METHOD', 'standard') == 'log_standard'
    }, f, indent=2)

cfg.set_status('data', True)
logger.info(f"✅ STEP 2 完成！数据预处理状态已锁定，包含使用的特征顺序、Scaler 和 Bounds。")

STEP 2: Data Preprocessing Pipeline [INFERENCE MODE]
Data Root: model_artifacts/HSI/20260611/run_230437


2026-06-11 23:06:54,751 - INFO - ⏩ [INFERENCE 模式] 跳过目标变量与数据划分，专注最新数据处理...
2026-06-11 23:06:54,763 - INFO - ✅ 成功提取今日原始特征快照至: model_artifacts/HSI/20260611/run_230437/inference_snapshot_20260603.csv
2026-06-11 23:06:54,779 - INFO - ✅ STEP 2 完成！数据预处理状态已锁定，包含使用的特征顺序、Scaler 和 Bounds。


ℹ️ [STEP 2] INFERENCE 使用历史模型目录: model_artifacts/HSI/20260611/run_224430
✅ 特征兼容报告已保存: model_artifacts/HSI/20260611/run_230437/feature_compat_report.json
FINAL DATA SHAPES (INFERENCE MODE)
X_inference_tensor: torch.Size([1, 30, 83])
目标日期: 2026-06-03


In [9]:
# ============================================================
# STEP 3.1: MODEL ARCHITECTURE (核心骨架)
# ============================================================
# 任务：提取 DualTowerPatternAwareLSTM 类，所有超参数从 cfg 注入
# ============================================================
# ==================== 前置依赖检查 ====================
cfg.check_dependency('feature')  # 确保特征工程已完成

run_mode = str(str(globals().get("RUN_MODE", os.getenv("RUN_MODE", "INFERENCE"))).strip().upper()).strip().upper()

# ==================== 从配置获取参数 ====================
device = cfg.device
SEQUENCE_LENGTH = cfg.SEQUENCE_LENGTH
PREDICTION_HORIZONS = cfg.PREDICTION_HORIZONS
NUM_HORIZONS = cfg.NUM_HORIZONS

if run_mode == "TRAIN":
    # 训练模式必须具备完整训练张量与目标 scaler
    if 'X_train_tensor' not in globals():
        raise NameError("X_train_tensor未定义，请先运行STEP 2进行数据预处理")
    if 'scaler_target_returns' not in globals():
        raise NameError("scaler_target_returns未定义，请先运行STEP 2进行数据预处理")
    if 'scaler_target_volatility' not in globals():
        raise NameError("scaler_target_volatility未定义，请先运行STEP 2进行数据预处理")
    NUM_PATTERNS = X_train_tensor.shape[2]  # 从训练数据自动获取
else:
    # 推理模式不依赖训练张量，使用当前特征矩阵维度构建模型骨架
    if 'patterns_df' in globals() and hasattr(patterns_df, 'shape'):
        NUM_PATTERNS = int(patterns_df.shape[1])
    else:
        NUM_PATTERNS = int(getattr(cfg, 'ORIGINAL_FEATURE_COUNT', 141))

print(f"[STEP 3.1] 构建双塔模型架构...")
print(f"  设备: {device}")
print(f"  输入特征数: {NUM_PATTERNS}")
print(f"  序列长度: {SEQUENCE_LENGTH}")
print(f"  预测时间间隔: {PREDICTION_HORIZONS}")

# ============================================================
# 双塔模型架构（共享骨架 + 独立头部）
# ============================================================
class DualTowerPatternAwareLSTM(nn.Module):
    """
    双塔架构LSTM模型（多时间尺度注意力版本）
    - 共享部分：Pattern Attention + LSTM + Multi-Scale Time Attention（共享特征提取）
    - 独立部分：收益率塔和波动率塔各自有独立的输出层（Head）
    - 多时间尺度：短期(1-5天)、中期(6-15天)、长期(16-30天)分组注意力
    - 可学习融合：使用可学习权重融合不同时间尺度的context

    所有超参数从 cfg 注入，禁止硬编码
    """
    def __init__(self, num_patterns, hidden_size=None, num_layers=None, dropout=None, seq_length=None, num_horizons=None):
        super(DualTowerPatternAwareLSTM, self).__init__()
        # 从 cfg 获取参数（如果未传入，则使用默认值）
        self.hidden_size = hidden_size if hidden_size is not None else cfg.HIDDEN_SIZE
        self.num_layers = num_layers if num_layers is not None else cfg.NUM_LAYERS
        self.dropout = dropout if dropout is not None else cfg.DROPOUT
        self.seq_length = seq_length if seq_length is not None else cfg.SEQUENCE_LENGTH
        self.num_horizons = num_horizons if num_horizons is not None else cfg.NUM_HORIZONS

        # ========== 共享骨架（Shared Backbone） ==========
        self.pattern_attn_fc = nn.Linear(num_patterns, num_patterns, bias=False)
        self.lstm = nn.LSTM(
            input_size=num_patterns,
            hidden_size=self.hidden_size,
            num_layers=self.num_layers,
            batch_first=True,
            dropout=self.dropout if self.num_layers > 1 else 0
        )

        # ========== 多时间尺度注意力（Multi-Scale Time Attention） ==========
        # 定义时间尺度分组（基于seq_length自动调整）
        self.short_len = min(5, self.seq_length // 4)
        self.mid_len = min(10, self.seq_length // 2)

        # 三个时间尺度的注意力层
        self.time_attn_short = nn.Linear(self.hidden_size, 1, bias=False)  # 短期注意力
        self.time_attn_mid = nn.Linear(self.hidden_size, 1, bias=False)    # 中期注意力
        self.time_attn_long = nn.Linear(self.hidden_size, 1, bias=False)  # 长期注意力

        # 可学习的融合权重（3个尺度 → 1个融合权重向量）
        self.scale_fusion = nn.Sequential(
            nn.Linear(self.hidden_size * 3, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.ReLU(),
            nn.Dropout(self.dropout * 0.5),
            nn.Linear(self.hidden_size, 3),  # 输出3个尺度的权重
            nn.Softmax(dim=-1)  # 归一化权重
        )
        self.layer_norm = nn.LayerNorm(self.hidden_size)

        # ========== 收益率专属特征提取 ==========
        self.returns_feature = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size * 2),  # 输入：共享context + 波动率上下文
            nn.LayerNorm(self.hidden_size * 2),
            nn.LeakyReLU(0.1),
            nn.Dropout(self.dropout * 0.3),
            nn.Linear(self.hidden_size * 2, self.hidden_size)
        )

        # ========== 收益率塔（Returns Tower Head） ==========
        self.returns_head = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),  # 第一层：保持完整维度
            nn.LayerNorm(self.hidden_size),  # 支持batch=1
            nn.GELU(),
            nn.Dropout(self.dropout * 0.4),
            nn.Linear(self.hidden_size, self.hidden_size // 2),  # 第二层：降维
            nn.LayerNorm(self.hidden_size // 2),  # 支持batch=1
            nn.GELU(),
            nn.Dropout(self.dropout * 0.3),
            nn.Linear(self.hidden_size // 2, self.num_horizons)  # 输出7个时间间隔的收益率
        )

        # ========== 波动率塔（Volatility Tower Head） ==========
        # 波动率特征提取（用于获取中间表示，作为收益率预测的上下文）
        self.volatility_feature = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.LayerNorm(self.hidden_size // 2),  # 支持batch=1
            nn.LeakyReLU(0.1),
            nn.Dropout(self.dropout * 0.3),
            nn.Linear(self.hidden_size // 2, self.hidden_size)  # 输出hidden_size维度的特征
        )
        # 波动率输出层（使用波动率特征）
        self.volatility_head = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 4),
            nn.LeakyReLU(0.1),
            nn.Dropout(self.dropout * 0.2),
            nn.Linear(self.hidden_size // 4, 1),
            nn.Softplus()  # 确保波动率预测始终为正
        )

    def forward(self, x):
        # ========== 共享骨架前向传播 ==========
        # 1. Pattern Attention
        p_scores = self.pattern_attn_fc(x)
        p_scores = p_scores / (torch.sqrt(torch.tensor(self.hidden_size, dtype=p_scores.dtype, device=p_scores.device)))
        p_weights = torch.softmax(p_scores, dim=2)
        x_weighted = x * p_weights

        # 2. LSTM Forward
        lstm_out, _ = self.lstm(x_weighted)  # (batch, seq_len, hidden_size)

        # 获取维度信息
        batch_size, seq_len, hidden_size = lstm_out.shape

        # ========== 3. 多时间尺度注意力（Multi-Scale Time Attention） ==========
        # 3.1 使用逆向切片法定义索引（短期对应最后N天）
        # 短期：最后 short_len 天
        short_term_indices = list(range(max(0, seq_len - self.short_len), seq_len))
        # 中期：短期之前的 mid_len 天
        mid_start = max(0, seq_len - self.short_len - self.mid_len)
        mid_end = max(0, seq_len - self.short_len)
        mid_term_indices = list(range(mid_start, mid_end))
        # 长期：剩余的所有早期天数
        long_term_indices = list(range(0, mid_start))

        # 3.2 封装空索引保护函数
        def compute_context_with_protection(lstm_segment, indices, attn_layer):
            """计算时间尺度注意力context，若indices为空则返回全零张量"""
            if len(indices) > 0:
                t_scores = attn_layer(lstm_segment)  # (batch, len(indices), 1)
                t_scores = t_scores / (t_scores.std(dim=1, keepdim=True) + 1e-8)
                t_weights = torch.softmax(t_scores, dim=1)  # (batch, len(indices), 1)
                context = torch.sum(lstm_segment * t_weights, dim=1)  # (batch, hidden)
                return context, t_weights
            else:
                context = torch.zeros(batch_size, hidden_size, device=lstm_out.device)
                t_weights = torch.zeros(batch_size, 0, 1, device=lstm_out.device)
                return context, t_weights

        # 3.3 计算各时间尺度的注意力
        # 短期注意力
        if len(short_term_indices) > 0:
            lstm_short = lstm_out[:, short_term_indices, :]  # (batch, short_len, hidden)
            context_short, t_weights_short = compute_context_with_protection(
                lstm_short, short_term_indices, self.time_attn_short
            )
        else:
            context_short, t_weights_short = compute_context_with_protection(
                None, short_term_indices, self.time_attn_short
            )

        # 中期注意力
        if len(mid_term_indices) > 0:
            lstm_mid = lstm_out[:, mid_term_indices, :]  # (batch, mid_len, hidden)
            context_mid, t_weights_mid = compute_context_with_protection(
                lstm_mid, mid_term_indices, self.time_attn_mid
            )
        else:
            context_mid, t_weights_mid = compute_context_with_protection(
                None, mid_term_indices, self.time_attn_mid
            )

        # 长期注意力
        if len(long_term_indices) > 0:
            lstm_long = lstm_out[:, long_term_indices, :]  # (batch, long_len, hidden)
            context_long, t_weights_long = compute_context_with_protection(
                lstm_long, long_term_indices, self.time_attn_long
            )
        else:
            context_long, t_weights_long = compute_context_with_protection(
                None, long_term_indices, self.time_attn_long
            )

        # 3.4 可学习权重融合
        contexts_concat = torch.cat([context_short, context_mid, context_long], dim=1)  # (batch, hidden*3)
        fusion_weights = self.scale_fusion(contexts_concat)  # (batch, 3) - [short_weight, mid_weight, long_weight]
        context = (fusion_weights[:, 0:1] * context_short +
                  fusion_weights[:, 1:2] * context_mid +
                  fusion_weights[:, 2:3] * context_long)  # (batch, hidden)

        # 3.5 构建完整的时间注意力权重（用于可视化）
        t_weights_full = torch.zeros(batch_size, seq_len, device=lstm_out.device)
        if len(short_term_indices) > 0:
            short_weights_2d = t_weights_short.squeeze(-1)  # (batch, len(short_term_indices))
            t_weights_full[:, short_term_indices] = short_weights_2d * fusion_weights[:, 0:1]
        if len(mid_term_indices) > 0:
            mid_weights_2d = t_weights_mid.squeeze(-1)  # (batch, len(mid_term_indices))
            t_weights_full[:, mid_term_indices] = mid_weights_2d * fusion_weights[:, 1:2]
        if len(long_term_indices) > 0:
            long_weights_2d = t_weights_long.squeeze(-1)  # (batch, len(long_term_indices))
            t_weights_full[:, long_term_indices] = long_weights_2d * fusion_weights[:, 2:3]
        t_weights_full = t_weights_full / (t_weights_full.sum(dim=1, keepdim=True) + 1e-10)

        # 4. LayerNorm
        context = self.layer_norm(context)

        # ========== 独立头部前向传播 ==========
        # 波动率特征提取和预测
        volatility_feature = self.volatility_feature(context)  # (batch, hidden_size)
        volatility_pred = self.volatility_head(volatility_feature)

        # 收益率专属特征提取（引入波动率上下文）
        returns_context_input = torch.cat([context, volatility_feature], dim=1)  # (batch, hidden_size * 2)
        returns_context = self.returns_feature(returns_context_input)  # (batch, hidden_size)
        returns_pred = self.returns_head(returns_context)

        # 合并输出
        # 【安全网】如果 trend 层缺失，输出零张量
        if hasattr(self, 'trend_feature') and hasattr(self, 'trend_head'):
            trend_context = self.trend_feature(torch.cat([context, volatility_feature], dim=1))
            trend_pred = self.trend_head(trend_context)
        else:
            trend_pred = torch.zeros(batch_size, 3, device=context.device)
        output = torch.cat([returns_pred, volatility_pred, trend_pred], dim=1)

        # ========== 返回：输出、模式注意力、完整时间注意力、时间注意力信息 ==========
        time_attention_info = {
            'full': t_weights_full,  # 完整时间注意力权重 (batch, seq_len)
            'short': t_weights_short.squeeze(-1) if len(short_term_indices) > 0 else None,
            'mid': t_weights_mid.squeeze(-1) if len(mid_term_indices) > 0 else None,
            'long': t_weights_long.squeeze(-1) if len(long_term_indices) > 0 else None,
            'fusion_weights': fusion_weights,  # 融合权重 (batch, 3) - [short, mid, long]
            'indices': {
                'short': short_term_indices,
                'mid': mid_term_indices,
                'long': long_term_indices
            }
        }
        return output, p_weights, t_weights_full, time_attention_info

print("✅ STEP 3.1 完成：模型架构已定义（所有参数从 cfg 注入）")



[STEP 3.1] 构建双塔模型架构...
  设备: cpu
  输入特征数: 83
  序列长度: 30
  预测时间间隔: [1, 5, 10, 15, 20, 25, 30]
✅ STEP 3.1 完成：模型架构已定义（所有参数从 cfg 注入）


In [10]:
# ============================================================
# STEP 3.2: HYPERPARAMETER OPTIMIZATION (支持强制重算与跨时空加载)
# ============================================================
import json
import logging
import gc
import os
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score
from pathlib import Path
from datetime import datetime

# ==================== 前置依赖检查 ====================
cfg.check_dependency('feature')

run_mode = str(str(globals().get("RUN_MODE", os.getenv("RUN_MODE", "INFERENCE"))).strip().upper()).strip().upper()

# ============================================================
# 【P7】HPO 主控制开关断言
# ============================================================
assert isinstance(RUN_HPO, bool), f"RUN_HPO 必须是 bool，实际是 {type(RUN_HPO).__name__}: {RUN_HPO}"
print(f"[STEP 3.2] HPO 主控制: RUN_MODE={RUN_MODE}, RUN_HPO={RUN_HPO}")


# 统一保存路径至当天运行目录
HYPERPARAMS_SAVE_PATH = cfg.TODAY_RUN_DIR / 'best_hyperparameters.json'
print(f"[STEP 3.2] 检查超参数版本状态...")

# ============================================================
# 🔍 历史超参数文件加载设置 (交互式支持)
# ============================================================
skip_optimization = (run_mode == "INFERENCE")
target_hpo_file = None
if run_mode == "INFERENCE":
    print("ℹ️ [STEP 3.2] 当前为 INFERENCE 模式，跳过 Optuna 超参数优化，仅尝试加载历史超参数文件。")

print("\n" + "-"*50)
print("🔍 历史超参数文件加载设置")
print("支持跨日期、跨 run_dir 加载历史最佳超参数文件。")
print("👉 选项 1: 输入历史 best_hyperparameters.json 的绝对或相对路径")
print("👉 选项 2: 直接按【回车键】，系统将自动搜索并加载全局最新的超参数文件")
print("👉 选项 3: 输入字母 'R'，强制放弃历史经验，从零开始重新跑 Optuna 搜索（回测截断时必须选此项！）")

user_input = "" if os.getenv("RUN_AUTO", "0") == "1" else input("\n请输入你的选择 (路径 / 回车 / R): ").strip()

if user_input.upper() == 'R':
    print("✅ 用户指令: 强制重新进行 Optuna 超参数优化！")
    # 直接跳过后续查找逻辑，skip_optimization 保持 False
else:
    # 1. 尝试使用用户输入的路径
    if user_input:
        if os.path.exists(user_input):
            target_hpo_file = Path(user_input)
            print(f"✅ 将使用指定的超参数文件: {target_hpo_file}")
        else:
            print(f"⚠️ 您输入的路径不存在: '{user_input}'，系统将回退到自动搜索机制！")
            
    # 2. 回退：自动寻找全局最新的超参数文件
    if target_hpo_file is None:
        try:
            runs_dir = cfg.SAVE_DIR  # 检索全部历史
            hpo_files = list(runs_dir.rglob("best_hyperparameters*.json"))
            if hpo_files:
                target_hpo_file = max(hpo_files, key=os.path.getmtime)
        except Exception as e:
            print(f"⚠️ 跨目录检索超参数文件时出错: {e}")

    # 3. 加载选中的超参数
    if target_hpo_file and os.path.exists(target_hpo_file):
        print(f"🔍 最终锁定加载的历史超参数文件: {target_hpo_file}")
        try:
            with open(target_hpo_file, 'r', encoding='utf-8') as f:
                best_hyperparameters = json.load(f)
            
            required_keys = ['hidden_size', 'num_layers', 'dropout', 'learning_rate']
            if all(k in best_hyperparameters for k in required_keys):
                print("✅ 成功加载历史超参数，为保证模型稳定性，将跳过 Optuna 优化！")
                skip_optimization = True
                
                # 📂 备份至当天的运行目录
                if target_hpo_file.resolve() != HYPERPARAMS_SAVE_PATH.resolve():
                    with open(HYPERPARAMS_SAVE_PATH, 'w', encoding='utf-8') as f:
                        json.dump(best_hyperparameters, f, indent=4)
                    print(f"📂 已将本次使用的超参数名单备份至沙盒目录: {HYPERPARAMS_SAVE_PATH}")
                
                # 将加载的参数应用到 cfg
                cfg.HIDDEN_SIZE = best_hyperparameters['hidden_size']
                cfg.NUM_LAYERS = best_hyperparameters['num_layers']
                cfg.DROPOUT = best_hyperparameters['dropout']
                cfg.LEARNING_RATE = best_hyperparameters['learning_rate']
                if 'weight_decay' in best_hyperparameters: cfg.WEIGHT_DECAY = best_hyperparameters['weight_decay']
                if 'loss_weight_returns' in best_hyperparameters: cfg.LOSS_WEIGHT_RETURNS = best_hyperparameters['loss_weight_returns']
                if 'loss_weight_volatility' in best_hyperparameters: cfg.LOSS_WEIGHT_VOLATILITY = best_hyperparameters['loss_weight_volatility']
                if 'loss_weight_direction' in best_hyperparameters: cfg.LOSS_WEIGHT_DIRECTION = best_hyperparameters['loss_weight_direction']
                if 'sigmoid_scale' in best_hyperparameters: cfg.SIGMOID_SCALE = best_hyperparameters['sigmoid_scale']
                if 'horizon_weight_1d' in best_hyperparameters: 
                    cfg.HORIZON_WEIGHT_1D = best_hyperparameters['horizon_weight_1d']
                
                print("✅ STEP 3.2 完成：已加载锁定版本的超参数。")
            else:
                print("⚠️ 历史文件缺少关键字段，将重新进行 Optuna 优化。")
        except Exception as e:
            print(f"⚠️ 读取历史超参数文件失败 ({e})，将重新进行 Optuna 优化。")
    else:
        print("ℹ️ 未发现或未提供任何有效的 best_hyperparameters*.json，将进行全新 Optuna 优化。")

print("-" * 50 + "\n")

# 如果跳过优化，则直接结束这个 Cell 的核心逻辑
if skip_optimization:
    pass
else:
    if run_mode != "TRAIN":
        print("⚠️ [STEP 3.2] 非 TRAIN 模式下禁止执行优化，已跳过。")
        skip_optimization = True
    # 变量存在性检查（仅 TRAIN 优化路径）
    if 'X_train_tensor' not in globals():
        raise NameError("X_train_tensor未定义，请先运行STEP 2进行数据预处理")
    if 'y_train_tensor' not in globals():
        raise NameError("y_train_tensor未定义，请先运行STEP 2进行数据预处理")
    if 'X_test_tensor' not in globals():
        raise NameError("X_test_tensor未定义，请先运行STEP 2进行数据预处理")
    if 'y_test_tensor' not in globals():
        raise NameError("y_test_tensor未定义，请先运行STEP 2进行数据预处理")
    # ============================================================
    # Optuna 优化逻辑 (首次运行或强制重算)
    # ============================================================
    print(f"🚀 [首次/强制训练] 开始超参数优化（Optuna）...")

    # 从配置获取参数
    device = cfg.device
    SEQUENCE_LENGTH = cfg.SEQUENCE_LENGTH
    NUM_PATTERNS = X_train_tensor.shape[2]
    PREDICTION_HORIZONS = cfg.PREDICTION_HORIZONS
    NUM_HORIZONS = cfg.NUM_HORIZONS
    VAL_SPLIT_RATIO = cfg.VAL_SPLIT_RATIO

    print(f"[STEP 3.2] 启动超参数优化（Optuna）...")
    print(f"  试验次数: {cfg.OPTIMIZATION_TRIALS}")
    print(f"  每个试验最大epoch: {cfg.OPTIMIZATION_EPOCHS}")
    print(f"  早停耐心值: {cfg.OPTIMIZATION_PATIENCE}")

    # ============================================================
    # 数值保护的组合损失函数
    # ============================================================
    def combined_loss(pred, target, cfg_instance):
        pred = torch.where(torch.isfinite(pred), pred, torch.zeros_like(pred))
        target = torch.where(torch.isfinite(target), target, torch.zeros_like(target))
        
        pred_returns = pred[:, 0:NUM_HORIZONS]
        pred_volatility = pred[:, NUM_HORIZONS:NUM_HORIZONS+1]
        target_returns = target[:, 0:NUM_HORIZONS]
        target_volatility = target[:, NUM_HORIZONS:NUM_HORIZONS+1]
        
        # 1. 收益率损失
        criterion_returns = nn.HuberLoss(delta=cfg_instance.HUBER_DELTA)
        horizon_weights = cfg_instance.HORIZON_WEIGHTS
        loss_returns = 0
        for i in range(NUM_HORIZONS):
            loss_returns += horizon_weights[i] * criterion_returns(pred_returns[:, i:i+1], target_returns[:, i:i+1])
        loss_returns = loss_returns / (sum(horizon_weights) + 1e-8)
        
        # 2. 波动率损失
        criterion_volatility = nn.MSELoss()
        loss_volatility = criterion_volatility(pred_volatility, target_volatility)
        
        # 3. 方向损失
        pred_returns_1d = pred_returns[:, 0:1]
        pred_returns_1d = torch.clamp(pred_returns_1d, -2.0, 2.0)
        
        pred_returns_1d = torch.where(torch.isfinite(pred_returns_1d), pred_returns_1d, torch.zeros_like(pred_returns_1d))
        target_returns_1d = target_returns[:, 0:1]
        target_returns_1d = torch.where(torch.isfinite(target_returns_1d), target_returns_1d, torch.zeros_like(target_returns_1d))
        
        target_direction = (target_returns_1d > 0).float()
        target_direction = torch.clamp(target_direction, min=0.0, max=1.0)
        
        sigmoid_scale = cfg_instance.SIGMOID_SCALE
        pred_direction_probs = torch.clamp(torch.sigmoid(pred_returns_1d * sigmoid_scale), min=1e-7, max=1-1e-7)
        
        loss_direction = nn.functional.binary_cross_entropy(
            pred_direction_probs, target_direction, reduction='mean'
        )
        
        total_loss = (cfg_instance.LOSS_WEIGHT_RETURNS * loss_returns + 
                      cfg_instance.LOSS_WEIGHT_VOLATILITY * loss_volatility + 
                      cfg_instance.LOSS_WEIGHT_DIRECTION * loss_direction)
        return total_loss, loss_returns, loss_volatility, loss_direction

    # ============================================================
    # Optuna 优化目标函数
    # ============================================================
    def objective(trial):
        hidden_size = trial.suggest_int('hidden_size', 32, 192, step=32)
        num_layers = trial.suggest_int('num_layers', 1, 3)
        dropout = trial.suggest_float('dropout', 0.2, 0.5, step=0.1)
        learning_rate = trial.suggest_float('learning_rate', 1e-5, 5e-4, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-4, 5e-3, log=True)
        
        loss_weight_returns = trial.suggest_float('loss_weight_returns', 0.05, 0.2, step=0.05)
        loss_weight_volatility = trial.suggest_float('loss_weight_volatility', 0.2, 0.4, step=0.1)
        loss_weight_direction = 1.0 - loss_weight_returns - loss_weight_volatility
        if loss_weight_direction < 0.4:
            loss_weight_direction = 0.4
            total = loss_weight_returns + loss_weight_volatility + loss_weight_direction
            loss_weight_returns /= total
            loss_weight_volatility /= total
            loss_weight_direction /= total
        
        sigmoid_scale = trial.suggest_float('sigmoid_scale', 10.0, 40.0, step=10.0)
        horizon_weight_1d = trial.suggest_float('horizon_weight_1d', 0.7, 0.9, step=0.05)
        
        temp_cfg = type('TempConfig', (), {})()
        temp_cfg.HIDDEN_SIZE = hidden_size
        temp_cfg.NUM_LAYERS = num_layers
        temp_cfg.DROPOUT = dropout
        temp_cfg.LEARNING_RATE = learning_rate
        temp_cfg.WEIGHT_DECAY = weight_decay
        temp_cfg.LOSS_WEIGHT_RETURNS = loss_weight_returns
        temp_cfg.LOSS_WEIGHT_VOLATILITY = loss_weight_volatility
        temp_cfg.LOSS_WEIGHT_DIRECTION = loss_weight_direction
        temp_cfg.SIGMOID_SCALE = sigmoid_scale
        temp_cfg.HUBER_DELTA = cfg.HUBER_DELTA
        
        remaining_weight = 1.0 - horizon_weight_1d
        temp_cfg.HORIZON_WEIGHTS = [
            horizon_weight_1d, remaining_weight * 0.3, remaining_weight * 0.2,
            remaining_weight * 0.15, remaining_weight * 0.15, remaining_weight * 0.1, remaining_weight * 0.1
        ]
        
        val_split_idx = int(len(X_train_tensor) * (1 - VAL_SPLIT_RATIO))
        X_train_final = X_train_tensor[:val_split_idx]
        y_train_final = y_train_tensor[:val_split_idx]
        X_val_tensor_split = X_train_tensor[val_split_idx:]
        y_val_tensor_split = y_train_tensor[val_split_idx:]
        
        model = DualTowerPatternAwareLSTM(
            num_patterns=NUM_PATTERNS, hidden_size=hidden_size, num_layers=num_layers,
            dropout=dropout, seq_length=SEQUENCE_LENGTH, num_horizons=NUM_HORIZONS
        ).to(device)
        
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', patience=cfg.SCHEDULER_PATIENCE, factor=cfg.SCHEDULER_FACTOR
        )
        
        train_loader = DataLoader(TensorDataset(X_train_final, y_train_final), batch_size=cfg.BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(TensorDataset(X_val_tensor_split, y_val_tensor_split), batch_size=cfg.BATCH_SIZE, shuffle=False)
        
        best_val_composite_score = float('-inf')
        patience_counter = 0
        
        for epoch in range(cfg.OPTIMIZATION_EPOCHS):
            model.train()
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer.zero_grad()
                outputs, _, _, _ = model(batch_X)
                loss, _, _, _ = combined_loss(outputs, batch_y, temp_cfg)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.GRAD_CLIP_NORM)
                optimizer.step()
            
            model.eval()
            epoch_val_loss = 0
            val_pred_returns_list, val_target_returns_list = [], []
            
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    outputs, _, _, _ = model(batch_X)
                    val_loss, _, _, _ = combined_loss(outputs, batch_y, temp_cfg)
                    epoch_val_loss += val_loss.item()
                    val_pred_returns_list.append(outputs[:, :NUM_HORIZONS].cpu().numpy())
                    val_target_returns_list.append(batch_y[:, :NUM_HORIZONS].cpu().numpy())
            
            scheduler.step(epoch_val_loss / len(val_loader))
            
            val_pred_returns_all = np.concatenate(val_pred_returns_list, axis=0)
            val_target_returns_all = np.concatenate(val_target_returns_list, axis=0)
            
            try:
                horizon_1d = PREDICTION_HORIZONS[0]
                val_pred_returns_1d_denorm = scaler_target_returns[horizon_1d].inverse_transform(val_pred_returns_all[:, 0:1]).flatten()
                val_target_returns_1d_denorm = scaler_target_returns[horizon_1d].inverse_transform(val_target_returns_all[:, 0:1]).flatten()
                
                valid_mask = np.isfinite(val_pred_returns_1d_denorm) & np.isfinite(val_target_returns_1d_denorm)
                if valid_mask.sum() > 0:
                    val_r2_returns = r2_score(val_target_returns_1d_denorm[valid_mask], val_pred_returns_1d_denorm[valid_mask])
                    pred_direction = (val_pred_returns_1d_denorm[valid_mask] > 0).astype(int)
                    actual_direction = (val_target_returns_1d_denorm[valid_mask] > 0).astype(int)
                    val_direction_accuracy = (pred_direction == actual_direction).mean()
                    val_direction_score = (val_direction_accuracy - 0.5) * 2.0
                    composite_score = (0.35 * max(val_r2_returns, 0) + 0.65 * val_direction_score)
                else:
                    composite_score = -1.0
            except:
                composite_score = -epoch_val_loss / len(val_loader)
            
            if composite_score > best_val_composite_score:
                best_val_composite_score = composite_score
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= cfg.OPTIMIZATION_PATIENCE:
                    break
            
            import optuna
            trial.report(composite_score, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
                
        final_score = float(best_val_composite_score)
        del model, optimizer, train_loader, val_loader
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()
        return final_score

    # ============================================================
    # 运行 Optuna 优化
    # ============================================================
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import MedianPruner
    
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    )

    print("\n开始超参数优化...")
    study.optimize(
        objective,
        n_trials=cfg.OPTIMIZATION_TRIALS,
        timeout=cfg.OPTIMIZATION_TIMEOUT,
        show_progress_bar=True
    )

    # ============================================================
    # 保存最优超参数
    # ============================================================
    best_params = study.best_params
    best_composite_score = study.best_value

    print(f"\n✅ 超参数优化完成！")
    print(f"最佳组合评分: {best_composite_score:.6f}")
    print(f"最优超参数:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")

    best_loss_weight_returns = best_params['loss_weight_returns']
    best_loss_weight_volatility = best_params['loss_weight_volatility']
    best_loss_weight_direction = 1.0 - best_loss_weight_returns - best_loss_weight_volatility
    if best_loss_weight_direction < 0.4:
        best_loss_weight_direction = 0.4
        total = best_loss_weight_returns + best_loss_weight_volatility + best_loss_weight_direction
        best_loss_weight_returns /= total
        best_loss_weight_volatility /= total
        best_loss_weight_direction /= total

    best_horizon_weight_1d = best_params['horizon_weight_1d']
    remaining_weight = 1.0 - best_horizon_weight_1d
    best_horizon_weights = [
        best_horizon_weight_1d, remaining_weight * 0.3, remaining_weight * 0.2,
        remaining_weight * 0.15, remaining_weight * 0.15, remaining_weight * 0.1, remaining_weight * 0.1
    ]

    best_hyperparameters = {
        'hidden_size': int(best_params['hidden_size']),
        'num_layers': int(best_params['num_layers']),
        'dropout': float(best_params['dropout']),
        'learning_rate': float(best_params['learning_rate']),
        'weight_decay': float(best_params['weight_decay']),
        'loss_weight_returns': float(best_loss_weight_returns),
        'loss_weight_volatility': float(best_loss_weight_volatility),
        'loss_weight_direction': float(best_loss_weight_direction),
        'sigmoid_scale': float(best_params['sigmoid_scale']),
        'horizon_weight_1d': float(best_horizon_weight_1d),
        'horizon_weights': [float(w) for w in best_horizon_weights],
        'best_composite_score': float(best_composite_score)
    }

    # 保存文件到当天的运行目录中
    with open(HYPERPARAMS_SAVE_PATH, 'w', encoding='utf-8') as f:
        json.dump(best_hyperparameters, f, indent=4)

    print(f"\n✅ 最优超参数已保存至: {HYPERPARAMS_SAVE_PATH}")

    # 将新优化的参数赋给cfg
    cfg.HIDDEN_SIZE = best_hyperparameters['hidden_size']
    cfg.NUM_LAYERS = best_hyperparameters['num_layers']
    cfg.DROPOUT = best_hyperparameters['dropout']
    cfg.LEARNING_RATE = best_hyperparameters['learning_rate']
    cfg.WEIGHT_DECAY = best_hyperparameters['weight_decay']
    cfg.LOSS_WEIGHT_RETURNS = best_hyperparameters['loss_weight_returns']
    cfg.LOSS_WEIGHT_VOLATILITY = best_hyperparameters['loss_weight_volatility']
    cfg.LOSS_WEIGHT_DIRECTION = best_hyperparameters['loss_weight_direction']
    cfg.SIGMOID_SCALE = best_hyperparameters['sigmoid_scale']
    cfg.HORIZON_WEIGHT_1D = best_hyperparameters['horizon_weight_1d']

    # 内存清理
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    gc.collect()

[STEP 3.2] HPO 主控制: RUN_MODE=INFERENCE, RUN_HPO=False
[STEP 3.2] 检查超参数版本状态...
ℹ️ [STEP 3.2] 当前为 INFERENCE 模式，跳过 Optuna 超参数优化，仅尝试加载历史超参数文件。

--------------------------------------------------
🔍 历史超参数文件加载设置
支持跨日期、跨 run_dir 加载历史最佳超参数文件。
👉 选项 1: 输入历史 best_hyperparameters.json 的绝对或相对路径
👉 选项 2: 直接按【回车键】，系统将自动搜索并加载全局最新的超参数文件
👉 选项 3: 输入字母 'R'，强制放弃历史经验，从零开始重新跑 Optuna 搜索（回测截断时必须选此项！）
✅ 用户指令: 强制重新进行 Optuna 超参数优化！
--------------------------------------------------



In [11]:
# ============================================================
# STEP 3.3: FINAL MODEL TRAINING (参数驱动全量训练 + 严谨 1.5 SD 几何通道)
# ============================================================
import json
import os
import gc
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from datetime import datetime
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ==================== 前置依赖检查 ====================
cfg.check_dependency('feature')

run_mode = str(str(globals().get("RUN_MODE", os.getenv("RUN_MODE", "INFERENCE"))).strip().upper()).strip().upper()
if run_mode != "TRAIN":
    print("ℹ️ [STEP 3.3] 当前为 INFERENCE 模式，跳过 FINAL MODEL TRAINING。")
else:
    # 变量存在性检查
    required_vars = ['X_train_tensor', 'y_train_tensor', 'X_test_tensor', 'y_test_tensor', 
                     'scaler_target_returns', 'scaler_target_volatility']
    for var in required_vars:
        if var not in globals():
            raise NameError(f"{var}未定义，请先运行相关步骤进行预处理")

    # ============================================================
    # 自动热加载：检测并加载最优超参数
    # ============================================================
    best_hyperparams_path = cfg.TODAY_RUN_DIR / 'best_hyperparameters.json'

    if best_hyperparams_path.exists():
        print("✅ 检测到最优超参数文件，开始自动加载...")
        with open(best_hyperparams_path, 'r', encoding='utf-8') as f:
            best_hyperparams = json.load(f)
    
        cfg.HIDDEN_SIZE = best_hyperparams['hidden_size']
        cfg.NUM_LAYERS = best_hyperparams['num_layers']
        cfg.DROPOUT = best_hyperparams['dropout']
        cfg.LEARNING_RATE = best_hyperparams['learning_rate']
        cfg.WEIGHT_DECAY = best_hyperparams['weight_decay']
        cfg.LOSS_WEIGHT_RETURNS = best_hyperparams['loss_weight_returns']
        cfg.LOSS_WEIGHT_VOLATILITY = best_hyperparams['loss_weight_volatility']
        cfg.LOSS_WEIGHT_DIRECTION = best_hyperparams['loss_weight_direction']
        cfg.SIGMOID_SCALE = best_hyperparams['sigmoid_scale']
    
        if 'horizon_weight_1d' in best_hyperparams:
            cfg.HORIZON_WEIGHT_1D = best_hyperparams['horizon_weight_1d']
    else:
        print("ℹ️  未检测到最优超参数文件，将使用默认参数")

    # ==================== 从配置获取参数 ====================
    device = cfg.device
    SEQUENCE_LENGTH = cfg.SEQUENCE_LENGTH
    NUM_PATTERNS = X_train_tensor.shape[2]
    PREDICTION_HORIZONS = cfg.PREDICTION_HORIZONS
    NUM_HORIZONS = cfg.NUM_HORIZONS
    VAL_SPLIT_RATIO = cfg.VAL_SPLIT_RATIO
    EPOCHS = cfg.EPOCHS
    PATIENCE = cfg.PATIENCE
    BATCH_SIZE = cfg.BATCH_SIZE
    LEARNING_RATE = cfg.LEARNING_RATE
    WEIGHT_DECAY = cfg.WEIGHT_DECAY
    HIDDEN_SIZE = cfg.HIDDEN_SIZE
    NUM_LSTM_LAYERS = cfg.NUM_LAYERS
    DROPOUT = cfg.DROPOUT
    VOLATILITY_SCALING_METHOD = getattr(cfg, 'VOLATILITY_SCALING_METHOD', 'standard')

    # ============================================================
    # 🛠️ 物理激活与数据隔离
    # ============================================================
    def init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
        elif isinstance(m, nn.LSTM):
            for name, param in m.named_parameters():
                if 'weight_ih' in name or 'weight_hh' in name:
                    nn.init.orthogonal_(param.data)
                elif 'bias' in name:
                    nn.init.zeros_(param.data)
                    param.data[param.size(0)//4 : param.size(0)//2].fill_(1.0)

    if 'X_train_full_raw' in globals():
        del X_train_full_raw

    total_train_samples = X_train_tensor.shape[0]
    val_split_idx = int(total_train_samples * (1 - VAL_SPLIT_RATIO))

    X_train_final = X_train_tensor[:val_split_idx].clone()
    y_train_final = y_train_tensor[:val_split_idx].clone()
    X_val_tensor_split = X_train_tensor[val_split_idx:].clone()
    y_val_tensor_split = y_train_tensor[val_split_idx:].clone()

    X_train_tensor = X_train_final
    y_train_tensor = y_train_final

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_tensor_split, y_val_tensor_split), batch_size=BATCH_SIZE, shuffle=False)

    # ============================================================
    # 模型初始化
    # ============================================================
    model = DualTowerPatternAwareLSTM(
        num_patterns=NUM_PATTERNS,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LSTM_LAYERS,
        dropout=DROPOUT,
        seq_length=SEQUENCE_LENGTH,
        num_horizons=NUM_HORIZONS
    ).to(device)

    model.apply(init_weights)
    # 【安全网】确保 trend 层存在（防止旧 checkpoint 缺少这些层）
    if not hasattr(model, 'trend_feature'):
        hs = HIDDEN_SIZE
        dr = DROPOUT
        model.trend_feature = nn.Sequential(
            nn.Linear(hs * 2, hs),
            nn.LayerNorm(hs),
            nn.LeakyReLU(0.1),
            nn.Dropout(dr * 0.3),
            nn.Linear(hs, hs // 2)
        )
    if not hasattr(model, 'trend_head'):
        model.trend_head = nn.Sequential(
            nn.Linear(HIDDEN_SIZE // 2, HIDDEN_SIZE // 4),
            nn.LeakyReLU(0.1),
            nn.Dropout(DROPOUT * 0.2),
            nn.Linear(HIDDEN_SIZE // 4, 3)
        )
    print("[STEP 3.3] trend head 安全网已就位")
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=cfg.SCHEDULER_PATIENCE, factor=cfg.SCHEDULER_FACTOR)

    # ============================================================
    # 🛠️ 训练循环
    # ============================================================
    criterion = nn.HuberLoss(delta=1.0)
    # 趋势损失：使用 CrossEntropyLoss，提升下跌类(0)权重
    trend_class_weights = torch.FloatTensor([2.0, 1.0, 1.5]).to(cfg.device)  # 下跌权重更高
    trend_criterion = nn.CrossEntropyLoss(weight=trend_class_weights)
    print(f"\n🚀 开始正式训练 (最大Epoch: {EPOCHS})...")

    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_val_direction_accuracy = 0.5
    patience_counter = 0
    TASK_ISOLATION_EPOCHS = 50
    LOCAL_MODEL_PATH = cfg.TODAY_RUN_DIR / 'best_model.pth'
    best_model_state = None

    for epoch in range(EPOCHS):
        model.train()
        epoch_train_loss = 0
        is_task_isolation = epoch < TASK_ISOLATION_EPOCHS
    
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs, _, _, _ = model(batch_X)
        
            if is_task_isolation:
                loss = criterion(outputs[:, 0:1], batch_y[:, 0:1])
            else:
                loss_returns = criterion(outputs[:, 0:NUM_HORIZONS], batch_y[:, 0:NUM_HORIZONS])
                loss_volatility = criterion(outputs[:, NUM_HORIZONS:NUM_HORIZONS+1], batch_y[:, NUM_HORIZONS:NUM_HORIZONS+1])
                # 趋势损失：从 trend head (indices NUM_HORIZONS+1 to NUM_HORIZONS+3) 取预测
                # batch_y: [7 returns (0-6) | 1 vol (7) | 1 trend (8)] = 9 列
                trend_logits = outputs[:, NUM_HORIZONS+1:NUM_HORIZONS+4]  # 模型输出 3 类
                trend_labels = batch_y[:, NUM_HORIZONS+1].long()  # trend label 在 index 8
                loss_trend = trend_criterion(trend_logits, trend_labels)
                loss = (cfg.LOSS_WEIGHT_RETURNS * loss_returns + 
                        cfg.LOSS_WEIGHT_VOLATILITY * loss_volatility + 
                        cfg.LOSS_WEIGHT_DIRECTION * loss_trend)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.GRAD_CLIP_NORM)
            optimizer.step()
            epoch_train_loss += loss.item()
    
        model.eval()
        epoch_val_loss = 0
        val_pred_returns_list, val_target_returns_list = [], []
    
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs, _, _, _ = model(batch_X)
            
                if is_task_isolation:
                    val_loss = criterion(outputs[:, 0:1], batch_y[:, 0:1])
                else:
                    loss_returns = criterion(outputs[:, 0:NUM_HORIZONS], batch_y[:, 0:NUM_HORIZONS])
                    loss_volatility = criterion(outputs[:, NUM_HORIZONS:NUM_HORIZONS+1], batch_y[:, NUM_HORIZONS:NUM_HORIZONS+1])
                    trend_logits = outputs[:, NUM_HORIZONS+1:NUM_HORIZONS+4]
                    trend_labels = batch_y[:, NUM_HORIZONS+1].long()
                    loss_trend = trend_criterion(trend_logits, trend_labels)
                    val_loss = (cfg.LOSS_WEIGHT_RETURNS * loss_returns + 
                                cfg.LOSS_WEIGHT_VOLATILITY * loss_volatility + 
                                cfg.LOSS_WEIGHT_DIRECTION * loss_trend)
                    val_pred_returns_list.append(outputs[:, 0:NUM_HORIZONS].cpu().numpy())
                    val_target_returns_list.append(batch_y[:, 0:NUM_HORIZONS].cpu().numpy())
                
                epoch_val_loss += val_loss.item()
            
        avg_train_loss = epoch_train_loss / len(train_loader)
        avg_val_loss = epoch_val_loss / len(val_loader)
        scheduler.step(avg_val_loss)
    
        is_better = False
        if not is_task_isolation and len(val_pred_returns_list) > 0:
            val_pred_ret = np.concatenate(val_pred_returns_list, axis=0)
            val_target_ret = np.concatenate(val_target_returns_list, axis=0)
            val_pred_1d = scaler_target_returns[PREDICTION_HORIZONS[0]].inverse_transform(val_pred_ret[:, 0:1]).flatten()
            val_target_1d = scaler_target_returns[PREDICTION_HORIZONS[0]].inverse_transform(val_target_ret[:, 0:1]).flatten()
        
            val_direction_accuracy = ((val_pred_1d > 0) == (val_target_1d > 0)).mean()
            if val_direction_accuracy > best_val_direction_accuracy:
                is_better = True
                best_val_direction_accuracy = val_direction_accuracy
                patience_counter = 0
            else:
                patience_counter += 1
        else:
            val_direction_accuracy = 0.5
            if avg_val_loss < best_val_loss:
                is_better = True
                best_val_loss = avg_val_loss
                patience_counter = 0
            else:
                patience_counter += 1
            
        if is_better:
            best_model_state = {'model_state_dict': model.state_dict(), 'epoch': epoch + 1}
            torch.save(best_model_state, LOCAL_MODEL_PATH)
        
        if (epoch + 1) % 5 == 0 or is_better:
            mode_str = "[Isolation]" if is_task_isolation else "[Joint]"
            best_str = "⭐ BEST" if is_better else ""
            print(f"Epoch {epoch+1:03d}/{EPOCHS} {mode_str} | Tr Loss: {avg_train_loss:.5f} | Vl Loss: {avg_val_loss:.5f} | Acc: {val_direction_accuracy*100:.2f}% {best_str}")
        
        if patience_counter >= PATIENCE and not is_task_isolation:
            print(f"\n⚠️ 触发早停：Epoch {epoch+1}")
            break

    # ============================================================
    # 🎯 最终模型评估：精度指标复盘与 1.5 SD 情景推演计算
    # ============================================================
    print("\n" + "="*80)
    print("📊 正在执行全量测试集推断与极值防御推演...")
    print("="*80)

    if best_model_state is not None:
        # 【安全网】加载时忽略旧 checkpoint 中没有的新层 keys
        pretrained = best_model_state['model_state_dict']
        model_dict = model.state_dict()
        matched = {k: v for k, v in pretrained.items() if k in model_dict and v.shape == model_dict[k].shape}
        missing = set(model_dict.keys()) - set(pretrained.keys())
        if missing:
            print(f"⚠️ 加载 checkpoint 缺少层: {missing}，用随机初始化")
        model.load_state_dict(matched, strict=False)

    model.eval()
    with torch.no_grad():
        test_pred_tensor, _, _, _ = model(X_test_tensor.to(device))
    
    test_pred = test_pred_tensor.cpu().numpy()
    y_test_np = y_test_tensor.cpu().numpy()

    # 1. 拆分 Returns 和 Volatility
    test_pred_returns = test_pred[:, 0:NUM_HORIZONS]
    test_pred_volatility = test_pred[:, NUM_HORIZONS:NUM_HORIZONS+1]
    y_test_returns = y_test_np[:, 0:NUM_HORIZONS]
    y_test_volatility = y_test_np[:, NUM_HORIZONS:NUM_HORIZONS+1]

    # 2. 反标准化 Returns
    test_pred_returns_denorm = np.zeros_like(test_pred_returns)
    y_test_returns_denorm = np.zeros_like(y_test_returns)

    for i, horizon in enumerate(PREDICTION_HORIZONS):
        if isinstance(scaler_target_returns, dict) and horizon in scaler_target_returns:
            test_pred_returns_denorm[:, i] = scaler_target_returns[horizon].inverse_transform(test_pred_returns[:, i:i+1]).flatten()
            y_test_returns_denorm[:, i] = scaler_target_returns[horizon].inverse_transform(y_test_returns[:, i:i+1]).flatten()

    # 3. 反标准化 Volatility
    try:
        if VOLATILITY_SCALING_METHOD == 'log_standard':
            test_pred_vol_denorm = np.expm1(scaler_target_volatility.inverse_transform(test_pred_volatility)).flatten()
            y_test_vol_denorm = np.expm1(scaler_target_volatility.inverse_transform(y_test_volatility)).flatten()
        else:
            test_pred_vol_denorm = scaler_target_volatility.inverse_transform(test_pred_volatility).flatten()
            y_test_vol_denorm = scaler_target_volatility.inverse_transform(y_test_volatility).flatten()
    except Exception as e:
        print(f"⚠️ 波动率反归一化异常: {e}")
        test_pred_vol_denorm = test_pred_volatility.flatten()
        y_test_vol_denorm = y_test_volatility.flatten()

    # 4. 📈 恢复完整的测试集评分日志 (R2, MAE, 准确率)
    test_r2 = r2_score(y_test_returns_denorm[:, 0], test_pred_returns_denorm[:, 0])
    test_mae = mean_absolute_error(y_test_returns_denorm[:, 0], test_pred_returns_denorm[:, 0])

    test_pred_direction = (test_pred_returns_denorm[:, 0] > 0).astype(int)
    test_actual_direction = (y_test_returns_denorm[:, 0] > 0).astype(int)
    direction_accuracy = (test_pred_direction == test_actual_direction).mean() * 100

    print(f"\n【核心评价指标】")
    print(f"   最终测试集表现 -> R2: {test_r2:.4f} | MAE: {test_mae:.4f} | 方向准确率 (T+1): {direction_accuracy:.2f}%")

    # ============================================================
    # 🛡️ 极值防御推演：1.5 Standard Deviation 置信区间 (修复量纲匹配)
    # ============================================================
    sd_multiplier = 1.5
    pred_mean_1d = test_pred_returns_denorm[:, 0]
    actual_1d = y_test_returns_denorm[:, 0]

    # 【核心逻辑修复】智能探测波动率量纲
    mean_vol_level = np.mean(test_pred_vol_denorm)
    is_annualized = mean_vol_level > 5.0  # 如果均值>5%，基本确定是年化波动率

    if is_annualized:
        print(f"ℹ️  探测到波动率均值为 {mean_vol_level:.2f}，判断为【年化波动率】量纲，将折算为单日波动率 (÷√252)。")
        daily_volatility = test_pred_vol_denorm / np.sqrt(252.0)
    else:
        print(f"ℹ️  探测到波动率均值为 {mean_vol_level:.2f}，判断为【单日波动率】量纲。")
        daily_volatility = test_pred_vol_denorm

    # 严格匹配单日波动率计算区间边界
    pred_upper_bounds = pred_mean_1d + (sd_multiplier * daily_volatility)
    pred_lower_bounds = pred_mean_1d - (sd_multiplier * daily_volatility)

    within_bounds = (actual_1d <= pred_upper_bounds) & (actual_1d >= pred_lower_bounds)
    hit_rate = np.mean(within_bounds) * 100

    print("\n【极值防御推演】")
    print(f"   1.5σ 区间测试集命中率: {hit_rate:.2f}% (理论正态分布基准: 86.64%)")

    # ============================================================
    # 🎯 T+1 情景推演报告 (同步对数正态 Log-Normal 数学内核)
    # ============================================================
    try:
        latest_close = df['Close'].iloc[-1]
    except:
        latest_close = 10000.0  

    last_pred_ret = pred_mean_1d[-1]
    last_daily_vol = daily_volatility[-1]

    # 与 Step 6 & 8 完全对齐的严谨数学：Drift 修正与对数缩放
    mu_decimal = last_pred_ret / 100.0
    sigma_decimal = last_daily_vol / 100.0
    drift = mu_decimal - 0.5 * (sigma_decimal ** 2)

    expected_price = latest_close * (1.0 + mu_decimal)
    upper_price = latest_close * np.exp(drift + sd_multiplier * sigma_decimal)
    lower_price = latest_close * np.exp(drift - sd_multiplier * sigma_decimal)

    last_upper_ret = (upper_price / latest_close - 1) * 100
    last_lower_ret = (lower_price / latest_close - 1) * 100
    last_exp_ret = (expected_price / latest_close - 1) * 100

    print(f"\n🎯 【T+1 情景推演报告 (当前基准价: {latest_close:.2f} | Log-Normal 缩放)】")
    print("-" * 65)
    print(f"🟢 [乐观情景 (+1.5σ)]: 目标位 {upper_price:.2f} ({last_upper_ret:+.2f}%)")
    print(f"🔵 [中性情景 (Mean)]:  目标位 {expected_price:.2f} ({last_exp_ret:+.2f}%)")
    print(f"🔴 [悲观情景 (-1.5σ)]: 目标位 {lower_price:.2f} ({last_lower_ret:+.2f}%)")
    print("-" * 65)

    # ============================================================
    # 📈 绘制 1.5 SD 预测通道图
    # ============================================================
    plot_window = min(60, len(actual_1d))
    y_true_plot = actual_1d[-plot_window:]
    y_pred_plot = pred_mean_1d[-plot_window:]
    upper_plot = pred_upper_bounds[-plot_window:]
    lower_plot = pred_lower_bounds[-plot_window:]

    try:
        dates_plot = df.index[-plot_window:]
    except:
        dates_plot = np.arange(plot_window)

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.fill_between(dates_plot, lower_plot, upper_plot, color='lightblue', alpha=0.5, label='1.5 SD Band')
    ax.plot(dates_plot, y_pred_plot, color='blue', linestyle='--', linewidth=2, label='Predicted Mean')

    in_bound_mask = within_bounds[-plot_window:]
    out_bound_mask = ~within_bounds[-plot_window:]
    ax.scatter(np.array(dates_plot)[in_bound_mask], y_true_plot[in_bound_mask], color='green', s=30, label='Actual Return (Inside)', zorder=5)
    ax.scatter(np.array(dates_plot)[out_bound_mask], y_true_plot[out_bound_mask], color='red', marker='x', s=60, label='Actual Breakout!', zorder=5)

    ax.axhline(0, color='black', linewidth=1, alpha=0.5)
    ax.set_title(f'{getattr(cfg, "TICKER_SYMBOL", "Stock")} - Test Set Returns vs 1.5 SD Prediction Band\nOverall Hit Rate: {hit_rate:.2f}%', fontsize=14, fontweight='bold')
    ax.set_ylabel('Returns (%)', fontsize=12)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()

    band_plot_path = cfg.TODAY_RUN_DIR / 'Confidence_Interval_1.5SD.png'
    plt.savefig(band_plot_path, dpi=300)
    plt.show()

    # ============================================================
    # 🛠️ 核心修复：强制物理保存 config.json (准生证)
    # ============================================================
    config_record = {
        'num_patterns': NUM_PATTERNS,
        'hidden_size': HIDDEN_SIZE,
        'num_layers': NUM_LSTM_LAYERS,
        'dropout': DROPOUT,
        'sequence_length': SEQUENCE_LENGTH,
        'learning_rate': LEARNING_RATE,
        'best_epoch': best_model_state['epoch'] if best_model_state else 'N/A'
    }

    # 明确写入 config.json，确保下游模块能够无缝跨沙盒读取
    with open(cfg.TODAY_RUN_DIR / 'config.json', 'w', encoding='utf-8') as f:
        json.dump(config_record, f, indent=4)

    # ============================================================
    # 保存最终结果与深度复盘闭环
    # ============================================================
    local_run_metrics = {
        "run_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "metrics": {
            "directional_accuracy_test": float(direction_accuracy),
            "test_r2": float(test_r2),
            "test_mae": float(test_mae),
            "1.5SD_hit_rate": float(hit_rate)
        },
        "model_config": config_record
    }

    with open(cfg.TODAY_RUN_DIR / 'run_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(local_run_metrics, f, indent=4)

    # 恢复写入横向对比的总账本 CSV
    results_csv_path = cfg.SAVE_DIR / "daily_run_results.csv"
    csv_record = {
        "run_timestamp": local_run_metrics["run_timestamp"],
        "run_date": datetime.now().strftime("%Y-%m-%d"),
        "ticker_symbol": getattr(cfg, 'TICKER_SYMBOL', 'UNKNOWN'),
        "test_acc": f"{direction_accuracy:.2f}%",
        "test_r2": f"{test_r2:.4f}",
        "1.5SD_hit_rate": f"{hit_rate:.2f}%",
        "best_epoch": config_record['best_epoch'],
        "model_path": str(LOCAL_MODEL_PATH)
    }

    try:
        df_results = pd.DataFrame([csv_record])
        if results_csv_path.exists():
            df_results.to_csv(results_csv_path, mode='a', header=False, index=False, encoding='utf-8-sig')
        else:
            df_results.to_csv(results_csv_path, mode='w', header=True, index=False, encoding='utf-8-sig')
        print(f"✅ 深度复盘结果已追加至: {results_csv_path.name}")
    except Exception as e:
        print(f"❌ 保存CSV结果失败: {e}")

    cfg.set_status('model', True)
    print(f"\n✅ STEP 3.3 训练彻底完成！")
    print(f"   模型与复盘资产 (含 config.json) 已锁定在: {cfg.TODAY_RUN_DIR}")

ℹ️ [STEP 3.3] 当前为 INFERENCE 模式，跳过 FINAL MODEL TRAINING。


In [12]:
run_mode = str(str(globals().get("RUN_MODE", os.getenv("RUN_MODE", "INFERENCE"))).strip().upper()).strip().upper()
if run_mode != "TRAIN":
    print("ℹ️ [STEP 4] 当前为 INFERENCE 模式，跳过训练集/测试集评估步骤。")
else:
    # ============================================================
    # STEP 4: ANALYZE PATTERN ATTENTION (深度诊断版 - 终极修复残影bug)
    # ============================================================
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import json
    import gc
    import torch
    from scipy.stats import pearsonr
    from pathlib import Path

    # ============================================================
    # 0. 自动对齐与强制现场推理
    # ============================================================
    print("[STEP 4] 启动特征诊断系统...")

    if 'cfg' not in globals():
        raise NameError("GlobalConfig未初始化，请先运行 cfg = GlobalConfig(ticker_symbol='^HSI')")

    PREDICTION_HORIZONS = cfg.PREDICTION_HORIZONS
    NUM_HORIZONS = cfg.NUM_HORIZONS
    VOLATILITY_SCALING_METHOD = getattr(cfg, 'VOLATILITY_SCALING_METHOD', 'standard')
    device = cfg.device

    REPORT_SAVE_DIR = cfg.TODAY_RUN_DIR
    REPORT_SAVE_DIR.mkdir(parents=True, exist_ok=True)

    # 强制现场推理：无视内存里的旧幽灵变量，只要有模型和测试集，重新算一遍！
    if 'model' in globals() and 'X_test_tensor' in globals() and 'y_test_tensor' in globals():
        print("🔄 正在执行现场全量推理以确保数据维度 100% 匹配...")
        model.eval()
        with torch.no_grad():
            test_pred_tensor, test_pattern_attn_tensor, test_time_attn_tensor, test_time_info = model(X_test_tensor.to(device))
        
            test_pred = test_pred_tensor.cpu().numpy()
            test_pattern_attn = test_pattern_attn_tensor.mean(dim=1).cpu().numpy()  # (Batch, Patterns)
            test_time_attn = test_time_attn_tensor.cpu().numpy()
            y_test_np = y_test_tensor.cpu().numpy()
        
        print(f"✅ 现场推理完成！测试集样本数: {len(test_pred)}")
    # 【P6新增】下跌趋势专项评估
    # ============================================================
    # 解析 trend predictions (index NUM_HORIZONS+1 to NUM_HORIZONS+3)
    if test_pred.shape[1] >= NUM_HORIZONS + 4:
        trend_pred_idx = np.argmax(test_pred[:, NUM_HORIZONS+1:NUM_HORIZONS+4], axis=1)
        trend_true = y_test_np[:, NUM_HORIZONS+4].astype(int)
        
        # 过滤掉 NaN（测试集末尾可能为 NaN）
        valid_mask = ~np.isnan(trend_true) & ~np.isnan(trend_pred_idx.astype(float))
        trend_pred_valid = trend_pred_idx[valid_mask]
        trend_true_valid = trend_true[valid_mask].astype(int)
        
        print("\n" + "=" * 60)
        print("【P6】下跌趋势专项评估报告")
        print("=" * 60)
        
        # 分类报告
        target_names = ['下跌(0)', '震荡(1)', '上涨(2)']
        print(classification_report(trend_true_valid, trend_pred_valid, target_names=target_names))
        
        # Confusion Matrix
        cm = confusion_matrix(trend_true_valid, trend_pred_valid)
        print(f"Confusion Matrix:\n{cm}")
        
        # 下跌类专项指标
        downtrend_mask = trend_true_valid == 0
        if downtrend_mask.sum() > 0:
            downtrend_pred = trend_pred_valid[downtrend_mask]
            downtrend_recall = (downtrend_pred == 0).sum() / downtrend_mask.sum()
            downtrend_precision = (downtrend_pred == 0).sum() / max((trend_pred_valid == 0).sum(), 1)
            downtrend_f1 = 2 * downtrend_recall * downtrend_precision / max(downtrend_recall + downtrend_precision, 1e-8)
            print(f"下跌趋势 Recall: {downtrend_recall:.4f}")
            print(f"下跌趋势 Precision: {downtrend_precision:.4f}")
            print(f"下跌趋势 F1: {downtrend_f1:.4f}")
        
        # 大跌样本预测明细
        large_down_mask = trend_true_valid == 0
        n_down = large_down_mask.sum()
        n_pred_down = (trend_pred_valid == 0).sum()
        print(f"大跌样本数: {n_down}, 预测为大跌数: {n_pred_down}")
        print(f"漏报率: {(trend_pred_valid[large_down_mask] != 0).sum()}/{n_down}")
    else:
        print("⚠️ test_pred 维度不足，跳过趋势评估")
        raise NameError("❌ 找不到 model 或 X_test_tensor，请确保已顺利完成 STEP 2 和 STEP 3.3！")

    # ============================================================
    # 1. 配置与数据准备 (加入溯源水印提取逻辑)
    # ============================================================

    if 'patterns_df' not in globals():
        raise NameError("patterns_df未定义，请先运行STEP 1进行特征工程")

    pattern_names = None
    feature_source_path = "Full Features (No Selection)"
    applied_features_path = cfg.TODAY_RUN_DIR / 'selected_features_applied.json'

    if applied_features_path.exists():
        try:
            with open(applied_features_path, 'r', encoding='utf-8') as f:
                selected_features_metadata = json.load(f)
                if 'selected_features' in selected_features_metadata:
                    pattern_names = selected_features_metadata['selected_features']
                    feature_source_path = selected_features_metadata.get('source', str(applied_features_path))
                    print(f"✅ 已从当天目录 {applied_features_path.name} 加载实际使用的特征名称 ({len(pattern_names)} 个)")
        except Exception as e:
            print(f"⚠️ 警告：无法从 {applied_features_path} 加载: {e}")

    # 备用方案
    if pattern_names is None and cfg.SELECTED_FEATURES_PATH.exists():
        try:
            with open(cfg.SELECTED_FEATURES_PATH, 'r', encoding='utf-8') as f:
                data = json.load(f)
                pattern_names = data.get('selected_features', data) if isinstance(data, dict) else data
                feature_source_path = str(cfg.SELECTED_FEATURES_PATH)
                print(f"✅ 从全局配置加载了特征名称 ({len(pattern_names)} 个)")
        except Exception as e: pass

    if pattern_names is None:
        pattern_names = patterns_df.columns.tolist()
        print(f"⚠️ 警告：使用 patterns_df 的所有列名作为特征名称 ({len(pattern_names)} 个)")

    actual_n_features = test_pattern_attn.shape[1] if test_pattern_attn.ndim > 1 else len(test_pattern_attn)
    if len(pattern_names) != actual_n_features:
        print(f"⚠️ 警告：特征名称数量 ({len(pattern_names)}) 与注意力张量维度 ({actual_n_features}) 不匹配，强制截断对齐")
        pattern_names = pattern_names[:actual_n_features]

    avg_pattern_attention = np.mean(test_pattern_attn, axis=0)

    # 构建全局水印文本
    hpo_source_path = str(cfg.TODAY_RUN_DIR / 'best_hyperparameters.json')
    watermark_text = (
        f"Model Traceability Info:\n"
        f"► Run DIR : {cfg.TODAY_RUN_DIR.name}\n"
        f"► Features: {Path(feature_source_path).name}\n"
        f"► HPO File: {Path(hpo_source_path).name}"
    )

    # ============================================================
    # 2. 加载 Scaler (从当天目录加载)
    # ============================================================
    import pickle

    scaler_vol_path = cfg.TODAY_RUN_DIR / 'scaler_volatility.pkl'
    if 'scaler_target_volatility' not in globals() or scaler_vol_path.exists():
        with open(scaler_vol_path, 'rb') as f:
            scaler_target_volatility = pickle.load(f)

    scaler_ret_path = cfg.TODAY_RUN_DIR / 'scaler_returns.pkl'
    if 'scaler_target_returns' not in globals() or scaler_ret_path.exists():
        with open(scaler_ret_path, 'rb') as f:
            scaler_target_returns = pickle.load(f)

    # ============================================================
    # 3. 反归一化
    # ============================================================
    test_pred_returns = test_pred[:, :NUM_HORIZONS]
    y_test_returns = y_test_np[:, :NUM_HORIZONS]
    test_pred_volatility = test_pred[:, NUM_HORIZONS:NUM_HORIZONS+1]
    y_test_volatility = y_test_np[:, NUM_HORIZONS:NUM_HORIZONS+1]

    try:
        if isinstance(scaler_target_returns, dict):
            horizon_1d = PREDICTION_HORIZONS[0]
            test_pred_returns_denorm = scaler_target_returns[horizon_1d].inverse_transform(test_pred_returns[:, 0:1])
            y_test_returns_denorm = scaler_target_returns[horizon_1d].inverse_transform(y_test_returns[:, 0:1])
        else:
            test_pred_returns_denorm = scaler_target_returns.inverse_transform(test_pred_returns[:, 0:1])
            y_test_returns_denorm = scaler_target_returns.inverse_transform(y_test_returns[:, 0:1])
    except:
        test_pred_returns_denorm = test_pred_returns[:, 0:1]
        y_test_returns_denorm = y_test_returns[:, 0:1]

    try:
        if VOLATILITY_SCALING_METHOD == 'log_standard':
            test_pred_volatility_denorm = np.expm1(scaler_target_volatility.inverse_transform(test_pred_volatility))
            y_test_volatility_denorm = np.expm1(scaler_target_volatility.inverse_transform(y_test_volatility))
        else:
            test_pred_volatility_denorm = scaler_target_volatility.inverse_transform(test_pred_volatility)
            y_test_volatility_denorm = scaler_target_volatility.inverse_transform(y_test_volatility)
    except:
        test_pred_volatility_denorm = test_pred_volatility
        y_test_volatility_denorm = y_test_volatility

    test_pred_returns_denorm = test_pred_returns_denorm[:, 0].flatten()
    y_test_returns_denorm = y_test_returns_denorm[:, 0].flatten()
    test_pred_volatility_denorm = test_pred_volatility_denorm.flatten()
    y_test_volatility_denorm = y_test_volatility_denorm.flatten()

    # ============================================================
    # 4. 特征大类映射
    # ============================================================
    extended_categories = {
        'Basic Technical': ['Returns', 'Log_Ret', 'Volatility_Ratio', 'Gap_Up', 'Gap_Down'],
        'Volume Factors': ['Vol_Rel_20', 'Vol_Rel_5', 'Vol_Surge', 'Vol_Dry_Up', 'Vol_Price_Up_Confirm', 'Vol_Price_Down_Panic', 'Vol_Pullback_Low', 'Vol_Bias_20', 'Vol_Trend_Consistency', 'Vol_Rel', 'High_Volume'],
        'Technical Analysis': ['RSI_Overbought', 'RSI_Oversold', 'RSI_Near_50', 'RSI_Divergence', 'RSI_Trend', 'RSI_Score', 'MACD_Bullish_Cross', 'MACD_Bearish_Cross', 'MACD_Hist_Slope', 'BB_Width', 'BB_Touch_Upper', 'BB_Touch_Lower', 'BB_Width_Compressed', 'BB_Pos_Norm', 'BB_Z_Score', 'KDJ_K', 'KDJ_D', 'KDJ_J', 'ADX', 'ADOSC'],
        'Pattern Recognition': ['Gap_Up_Reversal', 'Gap_Down_Reversal', 'Trend_2Day', 'Trend_Consistency', '2Day_Uptrend', '2Day_Downtrend', '3Day_Uptrend', 'Breakout_High', 'Breakdown_Low', 'Breakout_Confirm', 'Drawdown_Bounce', 'Inside_Day', 'Outside_Day', 'Shadow_Upper_Pct', 'Shadow_Lower_Pct', 'Range_Pct', 'Open_Close_Ratio', 'Open_Norm'],
        'Volatility': ['High_Volatility', 'Low_Volatility', 'Volatility_ATR_Ratio'],
        'Momentum': ['Strong_Momentum_Up', 'Trend_Efficiency', 'Trend_Alignment', 'Trend_Score_3D', 'Trend_Day_C'],
        'Market Sentiment': ['Sentiment_Fear_Rank', 'Sentiment_Change', 'Market_Up_Down_Ratio', 'VIX_Quantile', 'VIX_Change_Rate', 'VIX_Rank_Year', 'Corr_Price_VIX', 'VVIX_Proxy'],
        'Macro Finance': ['CNH_Change_5D', 'CNH_Trend_Dev', 'US10Y_Change', 'US10Y_Stress', 'DXY_Trend', 'HKD_Flow_Position', 'Macro_Triple_Bear', 'Cross_Ret_SSE', 'SSE_Gap', 'RS_vs_SSE', 'Corr_SSE_60', 'Gold_DXY_Ratio', 'Industry_Diff'],
        'Smart Money': ['OBV', 'MFI', 'Vol_Price_Trend_Sync'],
        'Statistical Features': ['Ret_ZScore_20', 'Price_Z_Score_60', 'Return_Volatility', 'Volatility_Cluster'],
        'Market Fractal': ['Choppiness_Index', 'Trend_Efficiency', 'Trend_Alignment'],
        'Time Features': ['Trend_Macro_200d', 'MOM_Quarter'],
        'Moving Average': ['MA10_MA20_Diff', 'Dist_from_High', 'Dist_from_Low']
    }

    feature_to_category = {}
    for cat, feats in extended_categories.items():
        for f in feats:
            feature_to_category[f] = cat

    # ============================================================
    # 5. 计算全量特征统计数据，并构建 DataFrame
    # ============================================================
    pred_direction_returns = (test_pred_returns_denorm > 0).astype(int)
    actual_direction_returns = (y_test_returns_denorm > 0).astype(int)
    correct_returns = (pred_direction_returns == actual_direction_returns)

    eps = 1e-8
    volatility_error = np.abs(test_pred_volatility_denorm - y_test_volatility_denorm) / (y_test_volatility_denorm + eps)
    volatility_median_error = np.median(volatility_error)
    correct_volatility = volatility_error < volatility_median_error

    n_correct_returns, n_wrong_returns = np.sum(correct_returns), np.sum(~correct_returns)
    n_correct_volatility, n_wrong_volatility = np.sum(correct_volatility), np.sum(~correct_volatility)

    correct_returns_attn = np.mean(test_pattern_attn[correct_returns], axis=0) if n_correct_returns > 0 else np.zeros(len(pattern_names))
    wrong_returns_attn = np.mean(test_pattern_attn[~correct_returns], axis=0) if n_wrong_returns > 0 else np.zeros(len(pattern_names))

    correct_volatility_attn = np.mean(test_pattern_attn[correct_volatility], axis=0) if n_correct_volatility > 0 else np.zeros(len(pattern_names))
    wrong_volatility_attn = np.mean(test_pattern_attn[~correct_volatility], axis=0) if n_wrong_volatility > 0 else np.zeros(len(pattern_names))

    diff_returns = (correct_returns_attn - wrong_returns_attn).flatten()
    diff_volatility = (correct_volatility_attn - wrong_volatility_attn).flatten()

    # 计算相关性
    corrs_ret_list, corrs_vol_list, corrs_dir_list = [], [], []

    if test_pattern_attn.ndim == 3 and test_pattern_attn.shape[2] > 1:
        test_pattern_attn_flat = np.mean(test_pattern_attn, axis=2)
    elif test_pattern_attn.ndim > 2:
        test_pattern_attn_flat = test_pattern_attn.reshape(test_pattern_attn.shape[0], -1)
    else:
        test_pattern_attn_flat = test_pattern_attn

    for i, pattern in enumerate(pattern_names):
        attn_values = test_pattern_attn_flat[:, i].flatten()
    
        try:
            corr_ret, _ = pearsonr(attn_values, np.abs(test_pred_returns_denorm))
        except: corr_ret = 0.0
        corrs_ret_list.append(corr_ret)
    
        try:
            corr_vol, _ = pearsonr(attn_values, test_pred_volatility_denorm)
        except: corr_vol = 0.0
        corrs_vol_list.append(corr_vol)
    
        try:
            corr_dir, _ = pearsonr(attn_values, test_pred_returns_denorm)
        except: corr_dir = 0.0
        corrs_dir_list.append(corr_dir)

    # 构建 DataFrame 保存所有特征数据
    feature_report_df = pd.DataFrame({
        'Feature_Name': pattern_names,
        'Category': [feature_to_category.get(p, 'Other/Unknown') for p in pattern_names],
        'Mean_Attention': avg_pattern_attention,
        'Ret_Dir_Correct_Attn': correct_returns_attn,
        'Ret_Dir_Wrong_Attn': wrong_returns_attn,
        'Ret_Dir_Diff_(Hero_Score)': diff_returns,
        'Vol_Accurate_Attn': correct_volatility_attn,
        'Vol_Inaccurate_Attn': wrong_volatility_attn,
        'Vol_Diff_(Hero_Score)': diff_volatility,
        'Corr_with_Abs_Return': corrs_ret_list,
        'Corr_with_Volatility': corrs_vol_list,
        'Corr_with_Direction': corrs_dir_list
    })

    # 排序并保存到 CSV，便于复盘
    feature_report_df = feature_report_df.sort_values(by='Mean_Attention', ascending=False)
    csv_save_path = REPORT_SAVE_DIR / 'pattern_attention_full_report.csv'
    feature_report_df.to_csv(csv_save_path, index=False, encoding='utf-8-sig')

    print("\n" + "=" * 80)
    print(f"✅ 完整特征诊断数据已落盘保存至：{csv_save_path}")
    print("=" * 80)

    # ============================================================
    # 6. 控制台详细输出 (全部特征完整列表)
    # ============================================================
    print(f"\n【收益率方向预测 - 完整特征贡献度榜单】")
    print(f"说明: 正数代表该特征预测正确时出力更多(功臣)，负数代表预测错误时被它带偏(内鬼)")
    print("-" * 80)

    # 按照 Hero_Score 从大到小（功臣 -> 内鬼）对所有特征进行完整排序并打印
    full_sorted_df = feature_report_df.sort_values(by='Ret_Dir_Diff_(Hero_Score)', ascending=False)

    for rank, (_, row) in enumerate(full_sorted_df.iterrows(), 1):
        feature_name = row['Feature_Name']
        category = row['Category']
        score = row['Ret_Dir_Diff_(Hero_Score)']
    
        # 稍微做点可视化标识
        if score > 0.005:
            marker = "🟢 [强力功臣]"
        elif score > 0:
            marker = "🟩 [轻微正向]"
        elif score > -0.005:
            marker = "🟨 [轻微带偏]"
        else:
            marker = "🔴 [严重内鬼]"
        
        print(f" {rank:2d}. {feature_name:<30} | {category:<15} | 差异: {score:+.4f} {marker}")

    print("-" * 80)
    # ============================================================
    # 7. 分类权重聚合 (按大类)
    # ============================================================
    cat_summary_df = feature_report_df.groupby('Category').agg({
        'Mean_Attention': 'mean',
        'Ret_Dir_Correct_Attn': 'mean',
        'Vol_Accurate_Attn': 'mean'
    }).reset_index()

    sorted_cats_returns = cat_summary_df.sort_values(by='Ret_Dir_Correct_Attn', ascending=False)[['Category', 'Ret_Dir_Correct_Attn']].values.tolist()
    sorted_cats_volatility = cat_summary_df.sort_values(by='Vol_Accurate_Attn', ascending=False)[['Category', 'Vol_Accurate_Attn']].values.tolist()
    sorted_cats_overall = cat_summary_df.sort_values(by='Mean_Attention', ascending=False)[['Category', 'Mean_Attention']].values.tolist()

    # ============================================================
    # 8. 极端样本案例研究
    # ============================================================
    print("\n" + "=" * 80)
    print("极端样本案例研究 (Extreme Case Study)")
    print("=" * 80)

    if 'test_dates' not in globals():
        test_dates = np.arange(len(test_pred_returns_denorm))

    max_gain_idx = np.argmax(y_test_returns_denorm)
    max_loss_idx = np.argmin(y_test_returns_denorm)

    direction_wrong = ~correct_returns
    pred_abs = np.abs(test_pred_returns_denorm)
    worst_mistake_idx = np.argmax(pred_abs * direction_wrong.astype(float)) if np.sum(direction_wrong) > 0 else 0

    for name, idx in [("最大涨幅日", max_gain_idx), ("最大跌幅日", max_loss_idx), ("滑铁卢日(严重反向)", worst_mistake_idx)]:
        if idx < len(test_dates):
            date = test_dates[idx]
            actual = y_test_returns_denorm[idx]
            predicted = test_pred_returns_denorm[idx]
            print(f"\n【{name}】")
            print(f"  日期/索引: {date.date() if hasattr(date, 'date') else idx}")
            print(f"  实际收益: {actual:+.4f}% | 预测: {predicted:+.4f}%")
        
            attn_weights = np.asarray(test_pattern_attn_flat[idx]).flatten()
            top_patterns = sorted(list(zip(pattern_names, attn_weights)), key=lambda x: x[1], reverse=True)[:5]
            print(f"  Top 5 触发特征:")
            for i, (p, w) in enumerate(top_patterns, 1):
                print(f"    {i}. {p:<25} ({w*100:5.2f}%) -> [{feature_to_category.get(p, 'Other')}]")

    # ============================================================
    # 9. 可视化图表 (带溯源水印)
    # ============================================================
    fig = plt.figure(figsize=(20, 16))
    fig.suptitle(f'Pattern Attention Diagnostic Report - {cfg.TICKER_SYMBOL}', fontsize=16, fontweight='bold', y=0.995)

    # 图1: 收益率方向 对/错对比
    ax1 = plt.subplot(3, 3, 1)
    hero_df = feature_report_df.sort_values(by='Ret_Dir_Diff_(Hero_Score)', ascending=False).head(10)
    villain_df = feature_report_df.sort_values(by='Ret_Dir_Diff_(Hero_Score)', ascending=True).head(10)
    p_hero, d_hero = hero_df['Feature_Name'].tolist(), hero_df['Ret_Dir_Diff_(Hero_Score)'].tolist()
    p_villain, d_villain = villain_df['Feature_Name'].tolist(), villain_df['Ret_Dir_Diff_(Hero_Score)'].tolist()

    x_pos = np.arange(len(p_hero))
    ax1.barh(x_pos, d_hero, color='green', alpha=0.7, label='Hero Features')
    ax1.barh(x_pos + len(p_hero), d_villain, color='red', alpha=0.7, label='Villain Features')
    ax1.set_yticks(list(x_pos) + list(x_pos + len(p_hero)))
    ax1.set_yticklabels(p_hero + p_villain, fontsize=8)
    ax1.set_xlabel('Weight Delta (Correct - Wrong)', fontsize=10)
    ax1.set_title('Returns Direction: Hero vs Villain', fontsize=11, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='x')
    ax1.axvline(0, color='black', linestyle='--', linewidth=1)

    # 图2: 波动率 对/错对比
    ax2 = plt.subplot(3, 3, 2)
    v_hero_df = feature_report_df.sort_values(by='Vol_Diff_(Hero_Score)', ascending=False).head(10)
    v_villain_df = feature_report_df.sort_values(by='Vol_Diff_(Hero_Score)', ascending=True).head(10)
    p_v_hero, d_v_hero = v_hero_df['Feature_Name'].tolist(), v_hero_df['Vol_Diff_(Hero_Score)'].tolist()
    p_v_villain, d_v_villain = v_villain_df['Feature_Name'].tolist(), v_villain_df['Vol_Diff_(Hero_Score)'].tolist()

    x_pos_v = np.arange(len(p_v_hero))
    ax2.barh(x_pos_v, d_v_hero, color='green', alpha=0.7, label='Hero Features')
    ax2.barh(x_pos_v + len(p_v_hero), d_v_villain, color='red', alpha=0.7, label='Villain Features')
    ax2.set_yticks(list(x_pos_v) + list(x_pos_v + len(p_v_hero)))
    ax2.set_yticklabels(p_v_hero + p_v_villain, fontsize=8)
    ax2.set_xlabel('Weight Delta (Accurate - Inaccurate)', fontsize=10)
    ax2.set_title('Volatility: Hero vs Villain', fontsize=11, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='x')
    ax2.axvline(0, color='black', linestyle='--', linewidth=1)

    # 图3: 分类权重 (收益率方向)
    ax3 = plt.subplot(3, 3, 3)
    cats_ret = [c[0] for c in sorted_cats_returns]
    weights_ret = [c[1] for c in sorted_cats_returns]
    colors_cat = plt.cm.Set3(range(len(cats_ret)))
    ax3.barh(range(len(cats_ret)), weights_ret, color=colors_cat, edgecolor='black')
    ax3.set_yticks(range(len(cats_ret)))
    ax3.set_yticklabels(cats_ret, fontsize=9)
    ax3.set_xlabel('Mean Attention Weight', fontsize=10)
    ax3.set_title('Returns Direction: Category Average', fontsize=11, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='x')
    ax3.invert_yaxis()

    # 图4: 分类权重 (波动率)
    ax4 = plt.subplot(3, 3, 4)
    cats_vol = [c[0] for c in sorted_cats_volatility]
    weights_vol = [c[1] for c in sorted_cats_volatility]
    colors_cat_vol = plt.cm.Set3(range(len(cats_vol)))
    ax4.barh(range(len(cats_vol)), weights_vol, color=colors_cat_vol, edgecolor='black')
    ax4.set_yticks(range(len(cats_vol)))
    ax4.set_yticklabels(cats_vol, fontsize=9)
    ax4.set_xlabel('Mean Attention Weight', fontsize=10)
    ax4.set_title('Volatility: Category Average', fontsize=11, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='x')
    ax4.invert_yaxis()

    # 图5: 相关性 (收益率绝对值)
    ax5 = plt.subplot(3, 3, 5)
    corr_ret_df = feature_report_df.sort_values(by='Corr_with_Abs_Return', key=abs, ascending=False).head(10)
    p_corr_ret, c_ret = corr_ret_df['Feature_Name'].tolist(), corr_ret_df['Corr_with_Abs_Return'].tolist()
    colors_corr = ['red' if c < 0 else 'green' for c in c_ret]
    ax5.barh(range(len(p_corr_ret)), c_ret, color=colors_corr, alpha=0.7, edgecolor='black')
    ax5.set_yticks(range(len(p_corr_ret)))
    ax5.set_yticklabels(p_corr_ret, fontsize=8)
    ax5.set_xlabel('Pearson Correlation', fontsize=10)
    ax5.set_title('Returns Magnitude vs Attention', fontsize=11, fontweight='bold')
    ax5.grid(True, alpha=0.3, axis='x')
    ax5.axvline(0, color='black', linestyle='--', linewidth=1)
    ax5.invert_yaxis()

    # 图6: 相关性 (波动率)
    ax6 = plt.subplot(3, 3, 6)
    corr_vol_df = feature_report_df.sort_values(by='Corr_with_Volatility', key=abs, ascending=False).head(10)
    p_corr_vol, c_vol = corr_vol_df['Feature_Name'].tolist(), corr_vol_df['Corr_with_Volatility'].tolist()
    colors_corr_vol = ['red' if c < 0 else 'green' for c in c_vol]
    ax6.barh(range(len(p_corr_vol)), c_vol, color=colors_corr_vol, alpha=0.7, edgecolor='black')
    ax6.set_yticks(range(len(p_corr_vol)))
    ax6.set_yticklabels(p_corr_vol, fontsize=8)
    ax6.set_xlabel('Pearson Correlation', fontsize=10)
    ax6.set_title('Volatility vs Attention', fontsize=11, fontweight='bold')
    ax6.grid(True, alpha=0.3, axis='x')
    ax6.axvline(0, color='black', linestyle='--', linewidth=1)
    ax6.invert_yaxis()

    # 图7: 相关性 (方向)
    ax7 = plt.subplot(3, 3, 7)
    corr_dir_df = feature_report_df.sort_values(by='Corr_with_Direction', key=abs, ascending=False).head(10)
    p_corr_dir, c_dir = corr_dir_df['Feature_Name'].tolist(), corr_dir_df['Corr_with_Direction'].tolist()
    colors_corr_dir = ['red' if c < 0 else 'green' for c in c_dir]
    ax7.barh(range(len(p_corr_dir)), c_dir, color=colors_corr_dir, alpha=0.7, edgecolor='black')
    ax7.set_yticks(range(len(p_corr_dir)))
    ax7.set_yticklabels(p_corr_dir, fontsize=8)
    ax7.set_title('Direction vs Attention', fontsize=11, fontweight='bold')
    ax7.grid(True, alpha=0.3, axis='x')
    ax7.axvline(0, color='black', linestyle='--', linewidth=1)
    ax7.invert_yaxis()

    # 图8: 整体分类权重对比
    ax8 = plt.subplot(3, 3, 8)
    cats_all = [c[0] for c in sorted_cats_overall]
    weights_all = [c[1] for c in sorted_cats_overall]
    colors_all = plt.cm.Set3(range(len(cats_all)))
    ax8.barh(range(len(cats_all)), weights_all, color=colors_all, edgecolor='black')
    ax8.set_yticks(range(len(cats_all)))
    ax8.set_yticklabels(cats_all, fontsize=9)
    ax8.set_xlabel('Mean Attention Weight', fontsize=10)
    ax8.set_title('Overall Average: Category Weights', fontsize=11, fontweight='bold')
    ax8.grid(True, alpha=0.3, axis='x')
    ax8.invert_yaxis()

    # 图9: 准确率统计
    ax9 = plt.subplot(3, 3, 9)
    accuracy_returns = np.sum(correct_returns) / len(correct_returns) * 100
    accuracy_volatility = np.sum(correct_volatility) / len(correct_volatility) * 100
    bars = ax9.bar(['Returns', 'Volatility'], [accuracy_returns, accuracy_volatility], color=['steelblue', 'coral'], alpha=0.8, edgecolor='black')
    ax9.set_ylim([0, 100])
    ax9.axhline(50, color='gray', linestyle='--', linewidth=2, label='Random Guess (50%)')
    ax9.set_title(f'Prediction Accuracy', fontsize=11, fontweight='bold')
    ax9.legend()
    for bar, acc in zip(bars, [accuracy_returns, accuracy_volatility]):
        ax9.text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{acc:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

    # 【新增核心逻辑】应用底部布局调整并添加溯源水印
    plt.tight_layout(rect=[0, 0.06, 1, 0.99])
    fig.text(0.01, 0.01, watermark_text, ha='left', va='bottom', fontsize=9, color='dimgray', family='monospace')

    # 图片落盘至 TODAY_RUN_DIR
    img_save_path = REPORT_SAVE_DIR / 'attention_diagnostic_report.png'
    fig.savefig(img_save_path, dpi=300, bbox_inches='tight')
    print(f"✅ 诊断图表已保存至: {img_save_path}")

    plt.show()

    # ============================================================
    # 10. 内存清理
    # ============================================================
    gc.collect()

ℹ️ [STEP 4] 当前为 INFERENCE 模式，跳过训练集/测试集评估步骤。


In [13]:
run_mode = str(str(globals().get("RUN_MODE", os.getenv("RUN_MODE", "INFERENCE"))).strip().upper()).strip().upper()
if run_mode != "TRAIN":
    print("ℹ️ [TRAIN-only Eval] 当前为 INFERENCE 模式，跳过训练评估步骤。")
else:
    # ============================================================
    # STEP 5: CREATE COMPREHENSIVE VISUALIZATIONS (终极对齐版)
    # ============================================================
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import json
    import pickle
    import gc
    import torch
    from pathlib import Path
    from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

    # ============================================================
    # 0. 强制现场推理：消灭一切“内存幽灵变量”
    # ============================================================
    print("🔄 [STEP 5] 正在执行现场全量推理以确保可视化数据维度 100% 匹配...")
    if 'model' in globals() and 'X_test_tensor' in globals() and 'y_test_tensor' in globals():
        model.eval()
        with torch.no_grad():
            test_pred_tensor, test_pattern_attn_tensor, test_time_attn_tensor, test_time_info = model(X_test_tensor.to(cfg.device))
        
            test_pred = test_pred_tensor.cpu().numpy()
            test_pattern_attn = test_pattern_attn_tensor.mean(dim=1).cpu().numpy()
            test_time_attn = test_time_attn_tensor.cpu().numpy()
            y_test_np = y_test_tensor.cpu().numpy()
            y_test = y_test_np  # 兼容旧变量名
        
            if test_time_info is not None and 'fusion_weights' in test_time_info:
                test_fusion_weights = test_time_info['fusion_weights'].cpu().numpy()
            else:
                test_fusion_weights = None
            
        print(f"✅ 现场推理完成！测试集样本数: {len(test_pred)}")
    else:
        raise NameError("❌ 找不到 model 或 X_test_tensor，请确保已顺利完成 STEP 2 和 STEP 3.3！")


    # ============================================================
    # 1. 反标准化与对齐协议 (Scaling Sync) & 提取溯源路径
    # ============================================================

    # 1.1 从当天运行目录(TODAY_RUN_DIR)加载 scaler，保证版本绝对一致
    scaler_ret_path = cfg.TODAY_RUN_DIR / 'scaler_returns.pkl'
    scaler_vol_path = cfg.TODAY_RUN_DIR / 'scaler_volatility.pkl'

    if not scaler_ret_path.exists():
        scaler_ret_path = cfg.SCALER_RETURNS_PATH
    if not scaler_vol_path.exists():
        scaler_vol_path = cfg.SCALER_VOLATILITY_PATH

    if not scaler_ret_path.exists():
        raise FileNotFoundError(f"Scaler文件不存在: {scaler_ret_path}")
    if not scaler_vol_path.exists():
        raise FileNotFoundError(f"Volatility scaler文件不存在: {scaler_vol_path}")

    with open(scaler_ret_path, 'rb') as f:
        scaler_target_returns = pickle.load(f)

    with open(scaler_vol_path, 'rb') as f:
        scaler_target_volatility = pickle.load(f)

    volatility_metadata = {}
    if hasattr(cfg, 'VOLATILITY_METADATA_PATH') and cfg.VOLATILITY_METADATA_PATH.exists():
        with open(cfg.VOLATILITY_METADATA_PATH, 'r', encoding='utf-8') as f:
            volatility_metadata = json.load(f)

    # 1.2 提取 test_dates
    if 'test_df' in globals() and test_df is not None and len(test_df) > 0:
        test_dates = test_df.index.values
    elif 'df' in globals() and df is not None:
        split_idx = int(len(df) * getattr(cfg, 'TRAIN_TEST_SPLIT_RATIO', 0.8))
        test_dates = df.index[split_idx:].values
    else:
        test_dates = np.arange(len(y_test))

    if len(test_dates) != len(y_test):
        test_dates = test_dates[:len(y_test)]

    # 1.3 重新执行反标准化（确保绝对准确）
    test_pred_returns = test_pred[:, 0] if test_pred.ndim > 1 else test_pred
    test_pred_volatility = test_pred[:, 1] if test_pred.ndim > 1 else np.zeros_like(test_pred_returns)

    y_test_returns = y_test[:, 0] if y_test.ndim > 1 else y_test
    y_test_volatility = y_test[:, 1] if y_test.ndim > 1 else np.zeros_like(y_test_returns)

    horizon_1d = cfg.PREDICTION_HORIZONS[0] if hasattr(cfg, 'PREDICTION_HORIZONS') else 1
    if isinstance(scaler_target_returns, dict):
        if horizon_1d in scaler_target_returns:
            test_pred_returns_denorm = scaler_target_returns[horizon_1d].inverse_transform(test_pred_returns.reshape(-1, 1)).flatten()
            y_test_returns_denorm = scaler_target_returns[horizon_1d].inverse_transform(y_test_returns.reshape(-1, 1)).flatten()
        else:
            first_available = list(scaler_target_returns.keys())[0]
            test_pred_returns_denorm = scaler_target_returns[first_available].inverse_transform(test_pred_returns.reshape(-1, 1)).flatten()
            y_test_returns_denorm = scaler_target_returns[first_available].inverse_transform(y_test_returns.reshape(-1, 1)).flatten()
    else:
        test_pred_returns_denorm = scaler_target_returns.inverse_transform(test_pred_returns.reshape(-1, 1)).flatten()
        y_test_returns_denorm = scaler_target_returns.inverse_transform(y_test_returns.reshape(-1, 1)).flatten()

    volatility_scaling_method = volatility_metadata.get('scaling_method', 'standard')
    if volatility_scaling_method == 'log_standard':
        test_pred_volatility_denorm = np.expm1(scaler_target_volatility.inverse_transform(test_pred_volatility.reshape(-1, 1))).flatten()
        y_test_volatility_denorm = np.expm1(scaler_target_volatility.inverse_transform(y_test_volatility.reshape(-1, 1))).flatten()
    else:
        test_pred_volatility_denorm = scaler_target_volatility.inverse_transform(test_pred_volatility.reshape(-1, 1)).flatten()
        y_test_volatility_denorm = scaler_target_volatility.inverse_transform(y_test_volatility.reshape(-1, 1)).flatten()

    # 1.4 加载 selected_features 并提取溯源路径 (Watermark Source)
    selected_features_list = None
    feature_source_path = "Full Features (No Selection)"
    applied_features_path = cfg.TODAY_RUN_DIR / 'selected_features_applied.json'

    if applied_features_path.exists():
        with open(applied_features_path, 'r', encoding='utf-8') as f:
            selected_features_metadata = json.load(f)
            selected_features_list = selected_features_metadata.get('selected_features', [])
            # 提取在 STEP 2 中记录的原始引用文件路径
            feature_source_path = selected_features_metadata.get('source', str(applied_features_path))
    elif hasattr(cfg, 'SELECTED_FEATURES_PATH') and cfg.SELECTED_FEATURES_PATH.exists():
        with open(cfg.SELECTED_FEATURES_PATH, 'r', encoding='utf-8') as f:
            selected_features_metadata = json.load(f)
            selected_features_list = selected_features_metadata.get('selected_features', [])
            feature_source_path = str(cfg.SELECTED_FEATURES_PATH)

    # 为了防止后续代码报错，提前定义 pattern_names (如果缺失)
    if 'pattern_names' not in globals() or len(pattern_names) != test_pattern_attn.shape[1]:
        if selected_features_list and len(selected_features_list) == test_pattern_attn.shape[1]:
            pattern_names = selected_features_list
        else:
            pattern_names = [f'Feature_{i}' for i in range(test_pattern_attn.shape[1])]

    hpo_source_path = str(cfg.TODAY_RUN_DIR / 'best_hyperparameters.json')

    # 构建全局水印文本
    watermark_text = (
        f"Model Traceability Info:\n"
        f"► Run DIR : {cfg.TODAY_RUN_DIR.name}\n"
        f"► Features: {Path(feature_source_path).name}\n"
        f"► HPO File: {Path(hpo_source_path).name}"
    )

    # ============================================================
    # 2. 准备可视化数据
    # ============================================================
    y_test_patterns = y_test.flatten()
    test_pred_flat = test_pred.flatten()
    avg_pattern_attention = np.mean(test_pattern_attn, axis=0)
    pattern_importance = list(zip(pattern_names, avg_pattern_attention))
    pattern_importance.sort(key=lambda x: x[1], reverse=True)
    losses = [{'val_loss_mse': loss} for loss in train_losses] if 'train_losses' in globals() else []

    test_r2_returns = r2_score(y_test_returns_denorm, test_pred_returns_denorm) if len(test_pred_returns_denorm) > 0 else 0.0
    test_r2_volatility = r2_score(y_test_volatility_denorm, test_pred_volatility_denorm) if len(test_pred_volatility_denorm) > 0 else 0.0
    test_r2 = r2_score(y_test_patterns, test_pred_flat)

    fig = plt.figure(figsize=(18, 14))

    # 1. Pattern Importance Bar Chart
    ax1 = plt.subplot(3, 3, 1)
    top_10_patterns = pattern_importance[:10]
    patterns_top10 = [p[0] for p in top_10_patterns]
    weights_top10 = [p[1] for p in top_10_patterns]
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.7, 10))
    ax1.barh(range(len(patterns_top10)), weights_top10, color=colors, edgecolor='black', alpha=0.8)
    ax1.set_yticks(range(len(patterns_top10)))
    ax1.set_yticklabels(patterns_top10, fontsize=9)
    ax1.set_xlabel('Average Attention Weight', fontsize=10)
    ax1.set_title(f'Top 10 Most Important Patterns ({cfg.TICKER_SYMBOL})', fontsize=11, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')
    ax1.invert_yaxis()

    # 2. Pattern Attention Heatmap
    ax2 = plt.subplot(3, 3, 2)
    sample_size = 20
    sample_attn = test_pattern_attn[:sample_size]
    im = ax2.imshow(sample_attn.T, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    ax2.set_xlabel('Test Sample', fontsize=10)
    ax2.set_ylabel('Pattern', fontsize=10)
    ax2.set_xticks(np.arange(0, sample_size, max(1, sample_size // 10)))
    ax2.set_xticklabels(np.arange(0, sample_size, max(1, sample_size // 10)), fontsize=8)
    y_step = max(1, len(pattern_names) // 20)  
    y_ticks = np.arange(0, len(pattern_names), y_step)
    ax2.set_yticks(y_ticks)
    ax2.set_yticklabels([pattern_names[i] for i in y_ticks], fontsize=7)
    ax2.set_title('Pattern Attention Heatmap (First 20)', fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax2, label='Attention')

    # 3. Multi-Scale Time Attention Distribution
    ax3 = plt.subplot(3, 3, 3)
    avg_time_attention = np.mean(test_time_attn, axis=0)
    seq_len = len(avg_time_attention)
    time_positions = np.arange(1, seq_len + 1)
    ax3.bar(time_positions, avg_time_attention, alpha=0.6, color='steelblue', edgecolor='black', label='Overall')
    ax3.set_xlabel('Time Position (Left=Oldest, Right=T-1)', fontsize=10)
    ax3.set_title(f'Multi-Scale Time Attention ({cfg.TICKER_SYMBOL})', fontsize=11, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    xticks = np.linspace(1, seq_len, min(8, seq_len), dtype=int)
    ax3.set_xticks(xticks)
    ax3.set_xticklabels([f'T-{seq_len - pos + 1}' for pos in xticks], fontsize=8)

    # 4. Training Loss
    ax4 = plt.subplot(3, 3, 4)
    val_losses = [l['val_loss_mse'] for l in losses]
    ax4.plot(val_losses, linewidth=2, color='purple', alpha=0.7)
    ax4.set_xlabel('Epoch', fontsize=10)
    ax4.set_title('Validation Loss History', fontsize=11, fontweight='bold')
    ax4.grid(True, alpha=0.3)

    # 5. Predictions vs Actual (仅显示 1D Returns)
    ax5 = plt.subplot(3, 3, 5)
    ax5.scatter(y_test_returns_denorm, test_pred_returns_denorm, alpha=0.5, s=30, color='coral', edgecolors='black', linewidths=0.5)
    min_val = min(y_test_returns_denorm.min(), test_pred_returns_denorm.min())
    max_val = max(y_test_returns_denorm.max(), test_pred_returns_denorm.max())
    ax5.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction', alpha=0.7)
    ax5.set_xlabel('Actual Returns (1D, %)', fontsize=10)
    ax5.set_ylabel('Predicted Returns (1D, %)', fontsize=10)
    ax5.set_title(f'Returns 1D Alignment\nR²: {test_r2_returns:.4f}', fontsize=11, fontweight='bold')
    ax5.grid(True, alpha=0.3)

    # 6. Time Series Predictions
    ax6 = plt.subplot(3, 3, 6)
    plot_range = slice(0, min(50, len(y_test_returns_denorm)))
    plot_dates = test_dates[plot_range] if len(test_dates) > 0 else np.arange(len(y_test_returns_denorm[plot_range]))
    ax6.plot(plot_dates, y_test_returns_denorm[plot_range], label='Actual', color='steelblue', marker='o', markersize=3)
    ax6.plot(plot_dates, test_pred_returns_denorm[plot_range], label='Predicted', color='coral', marker='s', markersize=3)
    ax6.set_title(f'Returns Time Series (First {len(y_test_returns_denorm[plot_range])} Days)', fontsize=11, fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    ax6.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

    # 7. Pattern Category Analysis
    ax7 = plt.subplot(3, 3, 7)
    categories = {
        'Gaps': ['Gap_Up', 'Gap_Down', 'Gap_Up_Reversal', 'Gap_Down_Reversal'],
        'Trends': ['2Day_Uptrend', '2Day_Downtrend', '3Day_Uptrend', '3Day_Downtrend', 'Trend_Score_3D', 'MACD_Bullish_Cross', 'MACD_Bearish_Cross'],
        'Breakouts': ['Breakout_High', 'Breakdown_Low', 'Drawdown_Bounce', 'Dist_from_High', 'Dist_from_Low'],
        'Volatility': ['High_Volatility', 'Low_Volatility', 'Inside_Day', 'Outside_Day', 'Volatility_ATR_Ratio', 'BB_Width_Compressed'],
        'Volume': ['High_Volume', 'Low_Volume', 'Volume_Change', 'Vol_Rel'],
        'Momentum': ['Strong_Momentum_Up', 'Returns', 'Log_Ret', 'RSI_Overbought', 'RSI_Oversold', 'RSI_Near_50', 'RSI_Score', 'BB_Touch_Upper', 'BB_Touch_Lower', 'BB_Z_Score']
    }
    category_weights = {}
    for cat, patterns in categories.items():
        try:
            valid_patterns = [p for p in patterns if p in pattern_names]
            if selected_features_list is not None and len(selected_features_list) > 0:
                valid_patterns = [p for p in valid_patterns if p in selected_features_list]
            if len(valid_patterns) > 0:
                weights = [avg_pattern_attention[pattern_names.index(p)] for p in valid_patterns]
                category_weights[cat] = np.mean(weights)
            else:
                category_weights[cat] = np.nan
        except:
            category_weights[cat] = np.nan

    cats = [cat for cat, weight in category_weights.items() if not np.isnan(weight)]
    cat_weights = [category_weights[cat] for cat in cats]
    if len(cats) > 0:
        ax7.bar(cats, cat_weights, color=plt.cm.Set3(np.linspace(0, 1, len(cats))), edgecolor='black', alpha=0.8)
        ax7.set_title(f'Category Importance', fontsize=11, fontweight='bold')
        ax7.tick_params(axis='x', rotation=45)
    else:
        ax7.set_title('Category Importance (No valid categories)')

    # 8. Attention Distribution
    ax8 = plt.subplot(3, 3, 8)
    all_pattern_attn = test_pattern_attn.flatten()
    ax8.hist(all_pattern_attn, bins=50, color='teal', alpha=0.7, edgecolor='black')
    ax8.axvline(np.mean(all_pattern_attn), color='red', linestyle='--', linewidth=2)
    ax8.set_title('Attention Distribution', fontsize=11, fontweight='bold')
    ax8.grid(True, alpha=0.3, axis='y')

    # 9. Top Patterns Over Time
    ax9 = plt.subplot(3, 3, 9)
    top_pattern_per_sample = np.argmax(test_pattern_attn, axis=1)
    pattern_counts = {}
    for idx in top_pattern_per_sample:
        pattern_counts[pattern_names[idx]] = pattern_counts.get(pattern_names[idx], 0) + 1
    sorted_counts = sorted(pattern_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    top_patterns = [p[0] for p in sorted_counts]
    top_counts = [p[1] for p in sorted_counts]
    ax9.barh(range(len(top_patterns)), top_counts, color=plt.cm.Spectral(np.linspace(0, 1, len(top_patterns))), edgecolor='black')
    ax9.set_yticks(range(len(top_patterns)))
    ax9.set_yticklabels(top_patterns, fontsize=9)
    ax9.set_title('Most Frequently Dominant Patterns', fontsize=11, fontweight='bold')
    ax9.invert_yaxis()

    # 【新增核心逻辑】应用底部布局调整并添加溯源水印
    plt.tight_layout(rect=[0, 0.06, 1, 0.99])  # 底部留出 6% 空间
    fig.text(0.01, 0.01, watermark_text, ha='left', va='bottom', fontsize=9, color='dimgray', family='monospace')

    fig.savefig(cfg.TODAY_RUN_DIR / 'Returns_Diagnostic.png', dpi=300, bbox_inches='tight')
    print(f"\n✅ 诊断图已保存: {cfg.TODAY_RUN_DIR / 'Returns_Diagnostic.png'}")
    plt.show()


    # ============================================================
    # ADDITIONAL ANALYSIS: Dual-Target Performance & Error Analysis
    # ============================================================

    mae_returns = mean_absolute_error(y_test_returns_denorm, test_pred_returns_denorm)
    rmse_returns = np.sqrt(mean_squared_error(y_test_returns_denorm, test_pred_returns_denorm))
    r2_returns = r2_score(y_test_returns_denorm, test_pred_returns_denorm)

    mae_volatility = mean_absolute_error(y_test_volatility_denorm, test_pred_volatility_denorm)
    rmse_volatility = np.sqrt(mean_squared_error(y_test_volatility_denorm, test_pred_volatility_denorm))
    r2_volatility = r2_score(y_test_volatility_denorm, test_pred_volatility_denorm)

    pred_direction_returns = (test_pred_returns_denorm > 0).astype(int)
    actual_direction_returns = (y_test_returns_denorm > 0).astype(int)
    direction_correct = (pred_direction_returns == actual_direction_returns).astype(int)

    window_size = min(30, len(direction_correct) // 4)
    rolling_accuracy = pd.Series(direction_correct).rolling(window=window_size, min_periods=1).mean() * 100

    attention_entropy = -np.sum(test_pattern_attn * np.log(test_pattern_attn + 1e-10), axis=1)
    prediction_uncertainty = attention_entropy / np.log(len(pattern_names)) 

    fig2 = plt.figure(figsize=(20, 14))

    # 1. Dual-Target Performance Comparison
    ax1 = plt.subplot(3, 3, 1)
    metrics = ['MAE', 'RMSE', 'R²']
    returns_metrics = [mae_returns, rmse_returns, r2_returns]
    volatility_metrics = [mae_volatility, rmse_volatility, r2_volatility]
    x = np.arange(len(metrics))
    width = 0.35
    ax1.bar(x - width/2, returns_metrics, width, label='Returns', color='steelblue', alpha=0.8)
    ax1.bar(x + width/2, volatility_metrics, width, label='Volatility', color='coral', alpha=0.8)
    ax1.set_title('Dual-Target Performance Comparison', fontsize=11, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(metrics)
    ax1.legend()

    # 2. Error Distribution - Returns
    ax2 = plt.subplot(3, 3, 2)
    errors_returns = test_pred_returns_denorm - y_test_returns_denorm
    ax2.hist(errors_returns, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    ax2.axvline(0, color='red', linestyle='--', linewidth=2)
    ax2.set_title(f'Returns Error Distribution', fontsize=11, fontweight='bold')

    # 3. Error Distribution - Volatility
    ax3 = plt.subplot(3, 3, 3)
    errors_volatility = test_pred_volatility_denorm - y_test_volatility_denorm
    ax3.hist(errors_volatility, bins=50, color='coral', alpha=0.7, edgecolor='black')
    ax3.axvline(0, color='red', linestyle='--', linewidth=2)
    ax3.set_title(f'Volatility Error Distribution', fontsize=11, fontweight='bold')

    # 4. Error Quantiles - Returns
    ax4 = plt.subplot(3, 3, 4)
    quantiles = [10, 25, 50, 75, 90, 95, 99]
    error_quantiles_returns = [np.percentile(np.abs(errors_returns), q) for q in quantiles]
    ax4.bar(range(len(quantiles)), error_quantiles_returns, color='steelblue', alpha=0.8, edgecolor='black')
    ax4.set_xticks(range(len(quantiles)))
    ax4.set_xticklabels([f'{q}%' for q in quantiles])
    ax4.set_title('Returns Error Quantiles', fontsize=11, fontweight='bold')

    # 5. Error Quantiles - Volatility
    ax5 = plt.subplot(3, 3, 5)
    error_quantiles_volatility = [np.percentile(np.abs(errors_volatility), q) for q in quantiles]
    ax5.bar(range(len(quantiles)), error_quantiles_volatility, color='coral', alpha=0.8, edgecolor='black')
    ax5.set_xticks(range(len(quantiles)))
    ax5.set_xticklabels([f'{q}%' for q in quantiles])
    ax5.set_title('Volatility Error Quantiles', fontsize=11, fontweight='bold')

    # 6. Direction Accuracy Over Time
    ax6 = plt.subplot(3, 3, 6)
    time_steps = np.arange(len(rolling_accuracy))
    ax6.plot(time_steps, rolling_accuracy, linewidth=2, color='green', alpha=0.8)
    ax6.axhline(50, color='red', linestyle='--', linewidth=1)
    ax6.set_title(f'Direction Prediction Accuracy Over Time', fontsize=11, fontweight='bold')

    # 7. Prediction Uncertainty (Attention Entropy)
    ax7 = plt.subplot(3, 3, 7)
    direction_correct_bool = (pred_direction_returns == actual_direction_returns)
    ax7.scatter(prediction_uncertainty[direction_correct_bool], np.abs(errors_returns)[direction_correct_bool], 
               alpha=0.6, s=20, color='forestgreen')
    ax7.scatter(prediction_uncertainty[~direction_correct_bool], np.abs(errors_returns)[~direction_correct_bool], 
               alpha=0.6, s=20, color='crimson')
    ax7.set_title('Uncertainty vs Error: Returns', fontsize=11, fontweight='bold')

    # 8. Prediction Uncertainty Distribution
    ax8 = plt.subplot(3, 3, 8)
    ax8.hist(prediction_uncertainty, bins=50, color='purple', alpha=0.7, edgecolor='black')
    ax8.set_title('Prediction Uncertainty Distribution', fontsize=11, fontweight='bold')

    # 9. Uncertainty vs Volatility Error
    ax9 = plt.subplot(3, 3, 9)
    ax9.scatter(prediction_uncertainty, np.abs(errors_volatility), alpha=0.5, s=20, color='orange')
    ax9.set_title('Uncertainty vs Error: Volatility', fontsize=11, fontweight='bold')

    # 【新增核心逻辑】应用底部布局调整并添加溯源水印
    plt.tight_layout(rect=[0, 0.06, 1, 0.99])
    fig2.text(0.01, 0.01, watermark_text, ha='left', va='bottom', fontsize=9, color='dimgray', family='monospace')

    fig2.savefig(cfg.TODAY_RUN_DIR / 'Error_Uncertainty.png', dpi=300, bbox_inches='tight')
    print(f"✅ 错误与不确定性诊断图已保存: {cfg.TODAY_RUN_DIR / 'Error_Uncertainty.png'}")
    plt.show()

    # ============================================================
    # MULTI-SCALE TIME ATTENTION VISUALIZATION
    # ============================================================

    if 'test_fusion_weights' in globals() and test_fusion_weights is not None and len(test_fusion_weights) > 0:
        fig3 = plt.figure(figsize=(20, 12))
    
        # 1. Fusion Weights Distribution (Short/Mid/Long)
        ax1 = plt.subplot(2, 3, 1)
        avg_fusion = np.mean(test_fusion_weights, axis=0)  
        std_fusion = np.std(test_fusion_weights, axis=0)
        scales = ['Short-term', 'Mid-term', 'Long-term']
        colors_scale = ['#FF6B6B', '#FFA500', '#4ECDC4']
        ax1.bar(scales, avg_fusion, color=colors_scale, alpha=0.8, edgecolor='black', yerr=std_fusion, capsize=5)
        ax1.set_title('Multi-Scale Fusion Weights', fontsize=12, fontweight='bold')
    
        # 2. Fusion Weights Distribution (Histogram)
        ax2 = plt.subplot(2, 3, 2)
        ax2.hist(test_fusion_weights[:, 0], bins=30, alpha=0.6, color='#FF6B6B', label='Short', edgecolor='black')
        ax2.hist(test_fusion_weights[:, 1], bins=30, alpha=0.6, color='#FFA500', label='Mid', edgecolor='black')
        ax2.hist(test_fusion_weights[:, 2], bins=30, alpha=0.6, color='#4ECDC4', label='Long', edgecolor='black')
        ax2.set_title('Fusion Weights Distribution', fontsize=12, fontweight='bold')
        ax2.legend()
    
        # 3. Time Attention Comparison (Short vs Mid vs Long)
        ax3 = plt.subplot(2, 3, 3)
        ax3.barh([0], [avg_fusion[0]], color='#FF6B6B', alpha=0.8, height=0.3)
        ax3.barh([1], [avg_fusion[1]], color='#FFA500', alpha=0.8, height=0.3)
        ax3.barh([2], [avg_fusion[2]], color='#4ECDC4', alpha=0.8, height=0.3)
        ax3.set_yticks([0, 1, 2])
        ax3.set_yticklabels(scales)
        ax3.set_title('Time Scale Importance', fontsize=12, fontweight='bold')
    
        # 4. Fusion Weights Over Time
        ax4 = plt.subplot(2, 3, 4)
        sample_indices = np.arange(len(test_fusion_weights))
        ax4.plot(sample_indices, test_fusion_weights[:, 0], color='#FF6B6B', linewidth=2, alpha=0.7)
        ax4.plot(sample_indices, test_fusion_weights[:, 1], color='#FFA500', linewidth=2, alpha=0.7)
        ax4.plot(sample_indices, test_fusion_weights[:, 2], color='#4ECDC4', linewidth=2, alpha=0.7)
        ax4.set_title('Fusion Weights Over Time', fontsize=12, fontweight='bold')
    
        # 5. Attention Heatmap by Scale
        ax5 = plt.subplot(2, 3, 5)
        seq_len = test_time_attn.shape[1]
        short_end = min(5, seq_len // 4)
        mid_end = min(short_end + 10, seq_len // 2 + 5)
        sample_size = min(50, len(test_time_attn))
        scale_attn_matrix = np.zeros((3, sample_size))
    
        if short_end > 0:
            short_indices = list(range(seq_len - short_end, seq_len))
            scale_attn_matrix[0, :] = np.mean(test_time_attn[:sample_size, short_indices], axis=1)
        if mid_end > short_end:
            mid_indices = list(range(seq_len - mid_end, seq_len - short_end))
            scale_attn_matrix[1, :] = np.mean(test_time_attn[:sample_size, mid_indices], axis=1)
        if seq_len - mid_end > 0:
            long_indices = list(range(0, seq_len - mid_end))
            scale_attn_matrix[2, :] = np.mean(test_time_attn[:sample_size, long_indices], axis=1)
    
        im = ax5.imshow(scale_attn_matrix, aspect='auto', cmap='YlOrRd', interpolation='nearest')
        ax5.set_yticks([0, 1, 2])
        ax5.set_yticklabels(scales)
        ax5.set_title('Attention Heatmap by Scale', fontsize=12, fontweight='bold')
    
        # 6. Correlation
        ax6 = plt.subplot(2, 3, 6)
        pred_errors = np.abs(test_pred_returns_denorm - y_test_returns_denorm)
        corr_short = np.corrcoef(test_fusion_weights[:, 0], pred_errors)[0, 1]
        corr_mid = np.corrcoef(test_fusion_weights[:, 1], pred_errors)[0, 1]
        corr_long = np.corrcoef(test_fusion_weights[:, 2], pred_errors)[0, 1]
        corrs = [corr_short, corr_mid, corr_long]
        ax6.bar(scales, corrs, color=colors_scale, alpha=0.8, edgecolor='black')
        ax6.axhline(0, color='black', linestyle='--', linewidth=1)
        ax6.set_title('Scale Weight vs Prediction Error', fontsize=12, fontweight='bold')
    
        # 【新增核心逻辑】应用底部布局调整并添加溯源水印
        plt.tight_layout(rect=[0, 0.08, 1, 0.99])  # 对于较小的图增加更多底部空间
        fig3.text(0.01, 0.01, watermark_text, ha='left', va='bottom', fontsize=9, color='dimgray', family='monospace')

        fig3.savefig(cfg.TODAY_RUN_DIR / 'Temporal_Attention.png', dpi=300, bbox_inches='tight')
        print(f"✅ 时间注意力诊断图已保存: {cfg.TODAY_RUN_DIR / 'Temporal_Attention.png'}")
        plt.show()

    # ============================================================
    # 内存清理 (Memory Cleanup)
    # ============================================================
    # 注释掉下面这行，否则 Jupyter 网页里的图会被直接清空无法显示
    # plt.close('all')  
    gc.collect() 
    print(f"\n✅ STEP 5 完成：所有可视化图表已生成并保存到当天记录文件夹: {cfg.TODAY_RUN_DIR}")

ℹ️ [TRAIN-only Eval] 当前为 INFERENCE 模式，跳过训练评估步骤。


In [14]:
# ============================================================================================
# STEP 8: PROFESSIONAL PROPORTIONAL PANORAMA (STRESS RESPONSE + LOG-NORMAL CONE)
# 版本：3.0 Final (Ultimate UI Restored) - 严谨比例化缩放 + 完整三图全景 + 收益率横向对比
# ============================================================================================
import os
import json
import pickle
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from pandas.tseries.offsets import BDay

# --------------------------------------------------------------------------------------------
# 0. 确认环境与全局配置
# --------------------------------------------------------------------------------------------
print("=" * 115)
print(f"{'STEP 8: LOG-NORMAL PROPORTIONAL DECISION TERMINAL':^115}")
print("=" * 115)

if 'cfg' not in globals(): 
    raise NameError("❌ 致命错误：GlobalConfig 未定义。")

if 'get_future_trading_dates' not in globals():
    raise NameError("请先运行公共函数区（Step0/Step1前置）：get_future_trading_dates 未定义")

PREDICTION_TICKER = str(cfg.TICKER_SYMBOL).strip()
device = cfg.device

# 🚀 历史模型热加载中心
run_mode = str(globals().get("RUN_MODE", os.getenv("RUN_MODE", "INFERENCE"))).strip().upper().strip().upper()
if run_mode == "INFERENCE":
    user_run_dir = os.getenv("MODEL_RUN_DIR", "")  # 空则自动选最新
    if user_run_dir:
        ACTIVE_RUN_DIR = Path(user_run_dir)
    else:
        try:
            ACTIVE_RUN_DIR = cfg.get_latest_model_run_dir()
            print(f"ℹ️ INFERENCE 自动加载最新历史模型目录: {ACTIVE_RUN_DIR}")
        except FileNotFoundError as e:
            raise FileNotFoundError(f"{e}\n❌ 当前 ticker 无历史模型，INFERENCE 无法继续。请先运行 TRAIN 产出基础模型。")
else:
    user_run_dir = os.getenv("MODEL_RUN_DIR", "")  # 空则使用 TODAY_RUN_DIR
    ACTIVE_RUN_DIR = Path(user_run_dir) if user_run_dir else cfg.TODAY_RUN_DIR

if not ACTIVE_RUN_DIR.exists():
    raise FileNotFoundError(f"❌ 找不到运行目录: {ACTIVE_RUN_DIR}")

# --------------------------------------------------------------------------------------------
# 1. 核心配件加载 (模型/Scaler/特征名单)
# --------------------------------------------------------------------------------------------
model_config = None
for fname in ['config.json', 'model_config.json', 'best_hyperparameters.json']:
    cp = ACTIVE_RUN_DIR / fname
    if cp.exists():
        with open(cp, 'r', encoding='utf-8') as f: model_config = json.load(f)
        break
if not model_config: raise FileNotFoundError("❌ 配置文件缺失。")

# 加载特征名单
feat_path = ACTIVE_RUN_DIR / 'selected_features_applied.json'
if not feat_path.exists(): feat_path = ACTIVE_RUN_DIR / 'selected_features.json'
with open(feat_path, 'r', encoding='utf-8') as f:
    f_data = json.load(f)
    selected_features = f_data.get('selected_features', []) if isinstance(f_data, dict) else f_data

# 加载 Scalers
with open(ACTIVE_RUN_DIR / 'scaler_returns.pkl', 'rb') as f: sc_ret = pickle.load(f)
with open(ACTIVE_RUN_DIR / 'scaler_volatility.pkl', 'rb') as f: sc_vol = pickle.load(f)
with open(ACTIVE_RUN_DIR / 'scaler_features.pkl', 'rb') as f: sc_feat = pickle.load(f)

# 初始化模型
model = DualTowerPatternAwareLSTM(
    num_patterns=len(selected_features),
    hidden_size=model_config.get('hidden_size', cfg.HIDDEN_SIZE),
    seq_length=model_config.get('sequence_length', cfg.SEQUENCE_LENGTH),
    num_layers=model_config.get('num_layers', cfg.NUM_LAYERS),
    dropout=model_config.get('dropout', cfg.DROPOUT)
).to(device)

best_model_path = ACTIVE_RUN_DIR / 'best_model.pth'
if not best_model_path.exists(): best_model_path = ACTIVE_RUN_DIR / 'model_latest.pth'
ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
# 【安全网】确保 model 有 trend 层（从新模型保存的 checkpoint 加载时需要）
hidden_size = cfg.HIDDEN_SIZE
dropout = cfg.DROPOUT
if not hasattr(model, 'trend_feature'):
    model.trend_feature = nn.Sequential(
        nn.Linear(hidden_size * 2, hidden_size),
        nn.LayerNorm(hidden_size),
        nn.LeakyReLU(0.1),
        nn.Dropout(dropout * 0.3),
        nn.Linear(hidden_size, hidden_size // 2)
    )
if not hasattr(model, 'trend_head'):
    model.trend_head = nn.Sequential(
        nn.Linear(hidden_size // 2, hidden_size // 4),
        nn.LeakyReLU(0.1),
        nn.Dropout(dropout * 0.2),
        nn.Linear(hidden_size // 4, 3)
    )
# 【安全网】加载时处理新旧层不匹配
pretrained = ckpt['model_state_dict']
model_dict = model.state_dict()
matched = {k: v for k, v in pretrained.items() if k in model_dict and v.shape == model_dict[k].shape}
missing = set(model_dict.keys()) - set(pretrained.keys())
if missing:
    print(f"⚠️ checkpoint 缺少层: {missing}，用随机初始化")
model.load_state_dict(matched, strict=False)
model.eval()

print(f"✅ 环境加载完毕 | 标的: {PREDICTION_TICKER} | 权重: {best_model_path.name}")

# --------------------------------------------------------------------------------------------
# 2. ⚡ 核心推演引擎：基于当前数据的 AI 推理
# --------------------------------------------------------------------------------------------
SEQUENCE_LENGTH = model_config.get('sequence_length', cfg.SEQUENCE_LENGTH)
PREDICTION_HORIZONS = getattr(cfg, 'PREDICTION_HORIZONS', [1, 5, 10, 15, 20, 25, 30])
NUM_HORIZONS = len(PREDICTION_HORIZONS)

latest_date = df.index[-1]
last_close_real = df['Close'].iloc[-1]
hist_vol_fallback = df['Close'].pct_change().tail(60).std() * np.sqrt(252) * 100

def get_base_inference(u_df):
    u_all_feat = detect_trading_patterns(u_df, config=cfg)
    u_feat = u_all_feat[selected_features].tail(SEQUENCE_LENGTH)
    u_scaled = np.clip(sc_feat.transform(u_feat.values), -5.0, 5.0)
    
    # 【诊断】打印模型结构维度
    print(f"  [DIAG] num_patterns={len(selected_features)}, HIDDEN={HIDDEN_SIZE}")
    print(f"  [DIAG] model.pattern_attn_fc.weight.shape={model.pattern_attn_fc.weight.shape}")
    if hasattr(model, 'trend_feature'):
        print(f"  [DIAG] model.trend_feature[0].weight.shape={model.trend_feature[0].weight.shape}")
    print(f"  [DIAG] u_scaled.shape={u_scaled.shape}")
    with torch.no_grad():
        out, _, _, _ = model(torch.FloatTensor(u_scaled).unsqueeze(0).to(device))
        u_rets_s = out[0, 0:NUM_HORIZONS].cpu().numpy()
        u_vol_s = out[0, NUM_HORIZONS].item()

    rets_dn = [sc_ret[h].inverse_transform([[u_rets_s[idx]]])[0,0] for idx, h in enumerate(PREDICTION_HORIZONS)]
    try:
        v_raw = sc_vol.inverse_transform([[u_vol_s]])
        v_ann = np.expm1(v_raw)[0,0] if getattr(cfg, 'VOLATILITY_SCALING_METHOD', '') == 'log_standard' else v_raw[0,0]
        if np.isnan(v_ann) or v_ann <= 0: v_ann = hist_vol_fallback
    except: v_ann = hist_vol_fallback
        
    return {'rets': rets_dn, 'vol_ann': v_ann}

print(f"\n🎲 正在执行 AI 推理与 Log-Normal 比例化缩放计算...")
base_results = get_base_inference(df)

# --------------------------------------------------------------------------------------------
# 3. 🎯 核心数学修正：与 Step 3.3 绝对对齐 (智能量纲 + 算术锚点 + 对数漏斗)
# --------------------------------------------------------------------------------------------
prediction_results = {
    'dates': [], 'horizons': [], 'h_vols': [],
    'prices_mean': [], 'prices_upper': [], 'prices_lower': [],
    'returns_mean': [], 'returns_upper': [], 'returns_lower': [] 
}

SD_MULT = 1.5

# 💡【对齐 Step 3.3】：智能探测波动率量纲
raw_vol = base_results['vol_ann']
is_annualized = raw_vol > 5.0
if is_annualized:
    print(f"ℹ️  智能探测：当前模型预测波动率为 {raw_vol:.2f}% (判定为年化)，应用 Root-Time 法则缩放。")
else:
    print(f"ℹ️  智能探测：当前模型预测波动率为 {raw_vol:.2f}% (判定为单频)，执行区间直接放大。")

future_dates_step8 = get_future_trading_dates(PREDICTION_TICKER, latest_date, PREDICTION_HORIZONS, step6_dates=globals().get("future_dates"))

for i, h in enumerate(PREDICTION_HORIZONS):
    future_date = future_dates_step8[i]
    
    # 💡【对齐 Step 3.3】：绝对忠于模型预测的算术基准价格
    mu_base = base_results['rets'][i] / 100.0
    p_mean = last_close_real * (1.0 + mu_base) 
    
    # 💡【对齐 Step 3.3】：根据量纲计算对应 Horizon 的波动率
    if is_annualized:
        sigma_h = (raw_vol / 100.0) * np.sqrt(h / 252.0)
        v_disp = raw_vol * np.sqrt(h / 252.0)
    else:
        sigma_h = (raw_vol / 100.0) * np.sqrt(h)
        v_disp = raw_vol * np.sqrt(h)
    
    # 完美的对数正态对称发散 (Log-Normal Cone)
    p_upper = p_mean * np.exp(SD_MULT * sigma_h)
    p_lower = p_mean * np.exp(-SD_MULT * sigma_h)

    # 换算回具体的百分比 Return，确保图表和报表数据零误差
    r_mean = base_results['rets'][i]
    r_upper = (p_upper / last_close_real - 1) * 100
    r_lower = (p_lower / last_close_real - 1) * 100

    prediction_results['dates'].append(future_date)
    prediction_results['horizons'].append(h)
    prediction_results['h_vols'].append(v_disp)
    
    prediction_results['prices_mean'].append(p_mean)
    prediction_results['prices_upper'].append(p_upper)
    prediction_results['prices_lower'].append(p_lower)
    
    prediction_results['returns_mean'].append(r_mean)
    prediction_results['returns_upper'].append(r_upper)
    prediction_results['returns_lower'].append(r_lower)
    
# --------------------------------------------------------------------------------------------
# 4. 可视化绘制 (100% 原始视觉保留，增加 Return 图)
# --------------------------------------------------------------------------------------------
history_days = 60
history_p = df['Close'].tail(history_days)
ticker_disp = PREDICTION_TICKER.replace("^", "")
REPORT_OUTPUT_DIR = cfg.TODAY_RUN_DIR
REPORT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
watermark = f"Model: {ACTIVE_RUN_DIR.name} | Run: {REPORT_OUTPUT_DIR.name} | Scaling: Log-Normal Proportional"

# ================= 📈 图1：价格全景漏斗图 =================
fig1 = plt.figure(figsize=(18, 12))
ax1 = fig1.add_subplot(1, 1, 1)

# 绘制历史
ax1.plot(history_p.index, history_p.values, color='#34495e', lw=3, label='Historical Price', alpha=0.9, zorder=2)

p_dates = [latest_date] + prediction_results['dates']
l_mean = [last_close_real] + prediction_results['prices_mean']
l_up   = [last_close_real] + prediction_results['prices_upper']
l_low  = [last_close_real] + prediction_results['prices_lower']

# 绘制比例化漏斗阴影
ax1.fill_between(p_dates, l_low, l_up, color='#3498db', alpha=0.15, label=f'±{SD_MULT}σ Log-Normal Cone', zorder=1)

# 绘制三路路径
ax1.plot(p_dates, l_up, color='#e74c3c', ls='--', lw=2.5, marker='^', markersize=6, label=f'+{SD_MULT}σ Bull Bound (Proportional)', zorder=3)
ax1.plot(p_dates, l_low, color='#27ae60', ls='--', lw=2.5, marker='v', markersize=6, label=f'-{SD_MULT}σ Bear Bound (Proportional)', zorder=3)
ax1.plot(p_dates, l_mean, color='#2980b9', ls='-', lw=4, marker='o', markersize=10, label='AI Expected Base Path', zorder=4)

ax1.scatter([latest_date], [last_close_real], color='black', s=200, zorder=5, label=f'T=0 Anchor ({last_close_real:.2f})', edgecolors='white', linewidths=2)
ax1.axvline(latest_date, color='gray', ls=':', lw=2, alpha=0.7)

# 彩色节点标注 (保持你原本优美的设计)
colors_map = {1: 'red', 5: 'orange', 10: 'gold', 15: 'green', 20: 'cyan', 25: 'blue', 30: 'purple'}
for d, p, h in zip(prediction_results['dates'], prediction_results['prices_mean'], prediction_results['horizons']):
    c = colors_map.get(h, 'gray')
    ax1.scatter([d], [p], color=c, s=450 if h==30 else 180, zorder=6, edgecolors='black', marker='*' if h==30 else 'o')
    lbl = f'{h}D\n(Trend)' if h==30 else f'{h}D'
    ax1.annotate(lbl, (d, p), xytext=(0,15), textcoords="offset points", ha='center', fontsize=10, fontweight='bold', color=c, 
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor=c))

ax1.set_xlim([history_p.index[0], p_dates[-1] + BDay(2)])
ax1.set_title(
    f"{ticker_disp} Prediction:{pd.to_datetime(p_dates[1]).strftime('%Y-%m-%d')} ~ {pd.to_datetime(p_dates[-1]).strftime('%Y-%m-%d')}",
    fontsize=18,
    fontweight='bold'
)
ax1.legend(loc='upper left', fontsize=12); ax1.grid(True, alpha=0.3, ls='--')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d')); plt.setp(ax1.get_xticklabels(), rotation=30)
ax1.set_ylabel('Asset Price', fontsize=12)

plt.tight_layout(rect=[0, 0.05, 1, 0.99])
fig1.text(0.01, 0.01, watermark, ha='left', va='bottom', fontsize=10, color='dimgray', family='monospace')
fig1.savefig(REPORT_OUTPUT_DIR / 'Step8_Proportional_Price_Path.png', dpi=300, bbox_inches='tight')
plt.show()

# ================= 📉 图2：波动率路径 (完美还原) =================
fig2 = plt.figure(figsize=(18, 8))
ax_v = fig2.add_subplot(1, 1, 1)
v_d = [latest_date] + prediction_results['dates']
v_v = [base_results['vol_ann'] * np.sqrt(1/252)] + prediction_results['h_vols']

ax_v.plot(v_d, v_v, 'purple', lw=3, marker='s', markersize=10, label='Expected Horizon Volatility', zorder=3)
ax_v.fill_between(v_d, v_v, alpha=0.2, color='purple', zorder=2)
ax_v.scatter(v_d, v_v, color='purple', s=100, zorder=4, edgecolors='black', linewidths=1.5)

for d, v in zip(v_d, v_v):
    ax_v.annotate(f'{v:.2f}%', (d, v), textcoords="offset points", xytext=(0,10), ha='center', fontsize=10, fontweight='bold')

ax_v.set_title("Expected Uncertainty Growth (Horizon Volatility)", fontsize=14, fontweight='bold')
ax_v.grid(True, alpha=0.3, ls='--'); ax_v.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.setp(ax_v.get_xticklabels(), rotation=30)
plt.tight_layout(rect=[0, 0.05, 1, 0.99])
fig2.text(0.01, 0.01, watermark, ha='left', va='bottom', fontsize=10, color='dimgray', family='monospace')
fig2.savefig(REPORT_OUTPUT_DIR / 'Step8_Volatility_Path.png', dpi=300, bbox_inches='tight')
plt.show()

# ================= 📊 图3：三维收益率横向对比图 (核心补全) =================
fig3 = plt.figure(figsize=(18, 10))
ax3 = fig3.add_subplot(1, 1, 1)

r_dates = [latest_date] + prediction_results['dates']
r_mean_path = [0] + prediction_results['returns_mean']
r_up_path   = [0] + prediction_results['returns_upper']
r_low_path  = [0] + prediction_results['returns_lower']

# 绘制收益率对比线
ax3.plot(r_dates, r_up_path, color='#e74c3c', ls='--', lw=2, marker='^', markersize=8, label=f'+{SD_MULT}σ Return (Bull)', alpha=0.8)
ax3.plot(r_dates, r_low_path, color='#27ae60', ls='--', lw=2, marker='v', markersize=8, label=f'-{SD_MULT}σ Return (Bear)', alpha=0.8)
ax3.plot(r_dates, r_mean_path, color='#2980b9', ls='-', lw=4, marker='o', markersize=10, label='Base Expected Return', zorder=5)

for d, r_m, r_u, r_l, h in zip(prediction_results['dates'], prediction_results['returns_mean'], prediction_results['returns_upper'], prediction_results['returns_lower'], prediction_results['horizons']):
    ax3.annotate(f'{r_u:+.2f}%', (d, r_u), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold', color='#e74c3c')
    ax3.annotate(f'{r_m:+.2f}%\n({h}D)', (d, r_m), textcoords="offset points", xytext=(15,0), ha='left', fontsize=10, fontweight='bold', color='#2980b9', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='#2980b9'))
    ax3.annotate(f'{r_l:+.2f}%', (d, r_l), textcoords="offset points", xytext=(0,-15), ha='center', fontsize=9, fontweight='bold', color='#27ae60')

ax3.axhline(0, color='black', ls='-', lw=1.5, zorder=1)
ax3.set_xticks(r_dates)
ax3.set_xticklabels(['Current'] + [f'{d.strftime("%m-%d")}\n({h}D)' for d, h in zip(prediction_results['dates'], prediction_results['horizons'])])
ax3.set_title('Proportional Scenario Returns Comparison', fontsize=16, fontweight='bold')
ax3.legend(loc='best', fontsize=12); ax3.grid(True, alpha=0.3, ls='--', axis='y')

plt.tight_layout(rect=[0, 0.05, 1, 0.99])
fig3.text(0.01, 0.01, watermark, ha='left', va='bottom', fontsize=10, color='dimgray', family='monospace')
fig3.savefig(REPORT_OUTPUT_DIR / 'Step8_Returns_Comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# --------------------------------------------------------------------------------------------
# 5. 专业对齐报表生成 (还原详尽文字与趋势分析)
# --------------------------------------------------------------------------------------------
summary = []
def log(m): print(m); summary.append(m)

log("\n" + "█" * 155)
log(f"{'QUANTITATIVE TERMINAL: PROPORTIONAL LOG-NORMAL REPORT & RETURN ANALYSIS':^155}")
log("█" * 155)
log(f"  📌 Current T+0: {last_close_real:.2f} | Scaling Mode: Exponential (Log-Normal) | Conf: ±{SD_MULT}σ")
log("-" * 155)
log(f"{'Horizon':<8} | {'Date':<12} | {'🔴 Bear Price (Return)':<26} | {'🎯 Base Price (Return)':<26} | {'🟢 Bull Price (Return)':<26} | {'⚡ Vol(%)'}")
log("-" * 155)

short_term_rets, med_term_rets, long_term_rets = [], [], []

for i in range(len(prediction_results['horizons'])):
    h, dt = prediction_results['horizons'][i], prediction_results['dates'][i].date()
    p_l, p_m, p_u = prediction_results['prices_lower'][i], prediction_results['prices_mean'][i], prediction_results['prices_upper'][i]
    r_l, r_m, r_u = prediction_results['returns_lower'][i], prediction_results['returns_mean'][i], prediction_results['returns_upper'][i]
    v = prediction_results['h_vols'][i]
    
    if h <= 5: short_term_rets.append(r_m)
    elif h <= 15: med_term_rets.append(r_m)
    else: long_term_rets.append(r_m)
    
    log(f"T+{h:<6} | {dt!s:<12} | {p_l:>10.2f} ({r_l:>+7.2f}%) | {p_m:>10.2f} ({r_m:>+7.2f}%) | {p_u:>10.2f} ({r_u:>+7.2f}%) | {v:>8.2f}%")

log("-" * 155)
log("\n📈 KEY TRENDS & LOG-NORMAL ANALYSIS:")
log("-" * 155)
if short_term_rets:
    log(f"  ► Short-Term (1-5 days)  : {'Upward' if np.mean(short_term_rets)>0 else 'Downward'} drift with average base return of {np.mean(short_term_rets):+.2f}%")
if med_term_rets:
    log(f"  ► Medium-Term (10-15 days): {'Upward' if np.mean(med_term_rets)>0 else 'Downward'} drift with average base return of {np.mean(med_term_rets):+.2f}%")
if long_term_rets:
    log(f"  ► Long-Term (20-30 days) : {'Upward' if np.mean(long_term_rets)>0 else 'Downward'} drift with average base return of {np.mean(long_term_rets):+.2f}%")

log(f"\n  ► Symmetry Verification  : Note that the Log-Normal model ensures mathematical proportionality.")
log(f"                             (Upper Price / Mean Price) strictly equals (Mean Price / Lower Price).")
log("=" * 155)

with open(REPORT_OUTPUT_DIR / 'Proportional_Inference_Report.txt', 'w', encoding='utf-8') as f: 
    f.write("\n".join(summary))

print(f"\n✅ 顶级比例化推演与三图联播彻底完成！报表已落盘至：{REPORT_OUTPUT_DIR}")

                                 STEP 8: LOG-NORMAL PROPORTIONAL DECISION TERMINAL                                 
ℹ️ INFERENCE 自动加载最新历史模型目录: model_artifacts/HSI/20260611/run_224430
✅ 环境加载完毕 | 标的: ^HSI | 权重: best_model.pth

🎲 正在执行 AI 推理与 Log-Normal 比例化缩放计算...
✅ P5新增下跌特征: Drawdown_From_20D_High, Drawdown_From_60D_High, Down_Days_5D, Down_Days_10D, Large_Down_Move, Close_Below_MA20, MA5_Below_MA20, Vol_Expansion_Down, Bear_Regime_Score


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x128 and 256x128)

In [ ]:
run_mode = str(str(globals().get("RUN_MODE", os.getenv("RUN_MODE", "INFERENCE"))).strip().upper()).strip().upper()
if run_mode != "TRAIN":
    print("ℹ️ [STEP 9] 当前为 INFERENCE 模式，跳过训练回测与诊断步骤。")
else:
    if 'get_future_trading_dates' not in globals():
        raise NameError("请先运行公共函数区（Step0/Step1前置）：get_future_trading_dates 未定义")

    # ============================================================
    # STEP 9: SCENARIO BACKTESTING (FINAL STABLE VERSION)
    # ============================================================
    import os
    import json
    import torch
    import numpy as np
    import pandas as pd
    import yfinance as yf
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    import scipy.stats as stats
    from pathlib import Path
    from datetime import datetime, timedelta
    from pandas.tseries.offsets import BDay
    from collections import OrderedDict

    print("\n" + "=" * 70)
    print("STEP 9: SCENARIO BACKTESTING (FINAL STABLE VERSION)")

    # 保存ticker_symbol（防止被后续代码覆盖）
    if 'ticker_symbol' in globals():
        STEP9_TICKER_SYMBOL = globals()['ticker_symbol']
    else:
        STEP9_TICKER_SYMBOL = '^HSI'  # 默认值
    print("=" * 70)

    # ============================================================
    # 定义预测时间间隔常量（与STEP 2一致）
    # ============================================================
    if 'PREDICTION_HORIZONS' not in globals():
        PREDICTION_HORIZONS = [1, 5, 10, 15, 20, 25, 30]  # 7个时间间隔
        NUM_HORIZONS = len(PREDICTION_HORIZONS)  # 7
        print(f"✅ 已定义预测时间间隔: {PREDICTION_HORIZONS} 天")

    # ========== 从当天目录的配置文件读取参数 ==========
    if 'cfg' not in globals():
        raise NameError("❌ 错误：GlobalConfig未初始化，请先运行 cfg = GlobalConfig(ticker_symbol='^HSI')")

    # 全部对齐 TODAY_RUN_DIR
    run_mode = str(str(globals().get("RUN_MODE", os.getenv("RUN_MODE", "INFERENCE"))).strip().upper()).strip().upper()
    if run_mode == "INFERENCE":
        model_run_dir = cfg.get_latest_model_run_dir()
    else:
        model_run_dir = cfg.TODAY_RUN_DIR

    model_config_path = model_run_dir / 'config.json'
    if not model_config_path.exists():
        alt_cfg_path = model_run_dir / 'model_config.json'
        if alt_cfg_path.exists():
            model_config_path = alt_cfg_path

    if not model_config_path.exists():
        raise FileNotFoundError(
            f"❌ 错误：找不到模型配置文件 {model_config_path}\n"
            f"   当前模型目录: {model_run_dir}。请先运行 TRAIN 生成配置文件。"
        )

    # 读取模型配置
    with open(model_config_path, 'r', encoding='utf-8') as f:
        model_config = json.load(f)

    required_config_keys = [
        'loss_weight_returns', 'loss_weight_volatility', 'loss_weight_direction',
        'weight_decay', 'epochs'
    ]
    missing_config_keys = [key for key in required_config_keys if key not in model_config]

    LOSS_WEIGHT_RETURNS = model_config.get('loss_weight_returns', 0.63)
    LOSS_WEIGHT_VOLATILITY = model_config.get('loss_weight_volatility', 0.25)
    LOSS_WEIGHT_DIRECTION = model_config.get('loss_weight_direction', 0.12)
    SEQUENCE_LENGTH = model_config.get('sequence_length') or model_config.get('seq_length', 30)
    HIDDEN_SIZE = model_config.get('hidden_size', 64)
    NUM_LSTM_LAYERS = model_config.get('num_layers', 1)
    DROPOUT = model_config.get('dropout', 0.4)
    LEARNING_RATE = model_config.get('learning_rate', 0.0001)
    WEIGHT_DECAY = model_config.get('weight_decay', 1e-4)

    if missing_config_keys:
        print(f"   ⚠️ 注意：配置文件缺少 {len(missing_config_keys)} 个参数，已使用兜底默认值")

    print(f"✅ 已从当天配置文件加载参数:")
    print(f"   损失权重: Returns={LOSS_WEIGHT_RETURNS}, Volatility={LOSS_WEIGHT_VOLATILITY}, Direction={LOSS_WEIGHT_DIRECTION}")
    print(f"   模型参数: HIDDEN_SIZE={HIDDEN_SIZE}, NUM_LSTM_LAYERS={NUM_LSTM_LAYERS}, DROPOUT={DROPOUT}")
    print(f"   训练参数: SEQUENCE_LENGTH={SEQUENCE_LENGTH}, LEARNING_RATE={LEARNING_RATE}, WEIGHT_DECAY={WEIGHT_DECAY}")

    # ====================================================================
    # 🚀 STEP 9.0: 强制使用 GlobalConfig - Ticker 符号读取
    # ====================================================================
    ticker_symbol = cfg.TICKER_SYMBOL
    print(f"✅ 从 GlobalConfig (cfg) 读取 ticker_symbol: {ticker_symbol}（物理锁定，禁止手动输入）")

    print(f"\n正在验证代码 {ticker_symbol} 的有效性...")
    temp_ticker = yf.Ticker(ticker_symbol)

    try:
        temp_df = temp_ticker.history(period="1d")
        if temp_df is None or temp_df.empty:
            raise ValueError("Empty dataframe")
    except (TypeError, ValueError, Exception) as e:
        raise ValueError(f"❌ 错误：ticker_symbol '{ticker_symbol}' 验证失败: {str(e)}")
    else:
        print(f"✅ 代码 {ticker_symbol} 验证通过！")
        STEP9_TICKER_SYMBOL = ticker_symbol
        ticker = yf.Ticker(ticker_symbol)

        end_date = datetime.now() + timedelta(days=1)
        start_date = end_date - timedelta(days=20*365) # 使用 20 年数据

        df = ticker.history(start=start_date, end=end_date)
        df.index = pd.to_datetime(df.index).tz_localize(None) 
        df['Log_Ret'] = np.log(df['Close'] / df['Close'].shift(1))

        print("   [FIX] Fetching VIX (^HSIL) and external assets...")
        vix_ticker = yf.Ticker("^HSIL")
        vix_df = vix_ticker.history(start=start_date, end=end_date)
        vix_df.index = pd.to_datetime(vix_df.index).tz_localize(None)
        if not vix_df.empty:
            df = df.join(vix_df['Close'].rename('VIX_Close').shift(1), how='left')
        else:
            print("      ⚠️ Warning: ^HSIL data is empty. Using fallback value 20.0.")
            df['VIX_Close'] = 20.0

        tickers_dict = {
            'SPX_Close': '^GSPC', 'SSE_Close': '000001.SS',
            'USDCNH_Close': 'CNH=F', 'US10Y_Close': '^TNX',
            'USDHKD_Close': 'HKD=X', 'DXY_Close': 'DX-Y.NYB',
            'GOLD_Close': 'GC=F'
        }
        for col_name, ext_ticker_symbol in tickers_dict.items():
            try:
                ext_df = yf.Ticker(ext_ticker_symbol).history(start=start_date, end=end_date)
                ext_df.index = pd.to_datetime(ext_df.index).tz_localize(None)
                if not ext_df.empty:
                    df = df.join(ext_df['Close'].rename(col_name).shift(1), how='left')
                else:
                    df[col_name] = np.nan 
            except Exception as e:
                df[col_name] = np.nan

        external_cols = list(tickers_dict.keys()) + ['VIX_Close']
        df[external_cols] = df[external_cols].ffill()
        for col in external_cols:
            if col in df.columns:
                df[col] = df[col].fillna(df[col].rolling(30, min_periods=1).mean())
        for col in external_cols:
            if col in df.columns and df[col].isnull().any():
                col_mean = df[col].mean()
                if not np.isnan(col_mean):
                    df[col] = df[col].fillna(col_mean)
                else:
                    df[col] = df[col].fillna(1.0)

        df['SPX_Ret'] = df['SPX_Close'].pct_change(fill_method=None).fillna(0)
        df['SSE_Ret'] = df['SSE_Close'].pct_change(fill_method=None).fillna(0)

        original_df_rows = len(df)
        df = df.dropna(subset=['Close', 'Open', 'High', 'Low'] + external_cols)
        if df.empty:
            raise ValueError("Critical Error: DataFrame became empty after dropping rows with NaNs.")
        else:
            latest_date = df.index[-1].date()
            print(f"   Dropped {original_df_rows - len(df)} rows due to NaNs or zero volume.")

        df = df[
            (df['Volume'] > 0) | 
            (df.index.date == latest_date) 
        ]

        if (df.index.date == latest_date).any():
            latest_volume = df.loc[df.index.date == latest_date, 'Volume'].iloc[0]
            if latest_volume == 0:
                prev_volume = df.loc[df.index.date < latest_date, 'Volume'].iloc[-1]
                print(f"⚠️ 最新日期 {latest_date} 的Volume为0，已填充为前一日值: {prev_volume}")
                df.loc[df.index.date == latest_date, 'Volume'] = prev_volume

        if df.empty:
            raise ValueError("Critical Error: DataFrame became empty after filtering for Volume > 0.")

        if not pd.api.types.is_datetime64_any_dtype(df.index):
            df.index = pd.to_datetime(df.index)

        # ====================================================================
        # 重新生成 patterns_df
        # ====================================================================
        if 'detect_trading_patterns' not in globals():
            raise NameError("The 'detect_trading_patterns' function is not defined.")
    
        patterns_df = detect_trading_patterns(df.copy(), config=cfg)

        if patterns_df.empty:
            raise ValueError("Critical Error: patterns_df is empty after feature detection.")

        original_patterns_rows = len(patterns_df)
        patterns_df = patterns_df.dropna() 
        if patterns_df.empty:
            raise ValueError("Critical Error: patterns_df became empty after dropping NaNs.")

        if original_patterns_rows != len(patterns_df):
            print(f"   Dropped {original_patterns_rows - len(patterns_df)} rows from patterns_df due to internal NaNs.")

        df = df.loc[patterns_df.index]
        if df.empty:
            raise ValueError("Critical Error: df became empty after aligning with patterns_df.")

        print(f"   [Re-initialized] df has {len(df)} rows.")
        print(f"   [Re-initialized] patterns_df has {len(patterns_df)} rows and {patterns_df.shape[1]} features.")

        device = cfg.device
        print(f"   Using Device: {device}")

        # SCENARIO_EPOCHS 取自 cfg
        SCENARIO_EPOCHS = getattr(cfg, 'EPOCHS', 50)
        forecast_horizon_days = 45
        comparison_horizon = 15

        latest_date_pd = df.index[-1]
        print(f"   最新数据日期: {latest_date_pd.date()}")

        # ===================== 场景分析函数 =====================
        def run_scenario_analysis(train_cutoff_date, label):
            from sklearn.preprocessing import StandardScaler as _StandardScaler
            from torch.utils.data import DataLoader, TensorDataset
        
            torch.manual_seed(42)
            np.random.seed(42)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(42)
        
            print(f"\n🚀 Running Scenario: [{label}]")
            print(f"   Cutoff Date: {train_cutoff_date.date()}")
            print(f"   ✅ 随机种子已设置（确保蒙特卡洛模拟的确定性）")

            def garman_klass_volatility(high, low, close, open_price, window=20, min_periods=10):
                log_hl = np.log(high / (low + 1e-10))
                log_co = np.log(close / (open_price + 1e-10))
                gk = 0.5 * (log_hl**2) - (2*np.log(2) - 1) * (log_co**2)
                gk_rolling = gk.rolling(window=window, min_periods=min_periods).mean()
                volatility = np.sqrt(gk_rolling * 252) * 100 
                return volatility

            # 参数对接
            NUM_LSTM_LAYERS = cfg.NUM_LAYERS 
            SEQUENCE_LENGTH = cfg.SEQUENCE_LENGTH
            HIDDEN_SIZE = cfg.HIDDEN_SIZE
            DROPOUT = cfg.DROPOUT
            BATCH_SIZE = cfg.BATCH_SIZE
            LEARNING_RATE = cfg.LEARNING_RATE
            NUM_HORIZONS = cfg.NUM_HORIZONS
            SCENARIO_EPOCHS = cfg.EPOCHS
            PREDICTION_HORIZONS = cfg.PREDICTION_HORIZONS
    
            train_mask = df.index <= train_cutoff_date
            train_df_scenario = df[train_mask].copy()
            holdout_df_real = df[~train_mask].copy()
            patterns_train_scenario = patterns_df[train_mask].copy()

            target_returns_dict = {}
            for horizon in PREDICTION_HORIZONS:
                future_close = train_df_scenario['Close'].shift(-horizon)
                current_close = train_df_scenario['Close']
                target_returns_dict[f'Target_Returns_{horizon}D'] = (future_close / current_close - 1) * 100
            target_returns_scenario = pd.DataFrame(target_returns_dict)
    
            target_volatility_scenario = garman_klass_volatility(
                train_df_scenario['High'], train_df_scenario['Low'],
                train_df_scenario['Close'], train_df_scenario['Open']
            ).shift(-1) * 100 
            target_volatility_scenario.name = 'Target_Volatility'
    
            target_series_scenario = pd.concat([target_returns_scenario, target_volatility_scenario], axis=1)
            combined_scenario = pd.concat([patterns_train_scenario, target_series_scenario], axis=1, join='inner')
            combined_scenario = combined_scenario.replace([np.inf, -np.inf], np.nan).dropna()
    
            patterns_train_aligned = combined_scenario.drop(columns=list(target_series_scenario.columns)).copy()
            target_columns_scenario = list(target_returns_scenario.columns) + ['Target_Volatility']
            y_train_raw = combined_scenario[target_columns_scenario].values 

            if len(patterns_train_aligned) < SEQUENCE_LENGTH + 1:
                return {
                    'label': label, 'train_cutoff': train_cutoff_date, 'mae': np.nan,
                    'full_sim_df': pd.DataFrame(), 'forecast_15': pd.DataFrame(), 'pattern_summary': [],
                    'train_sequences': 0, 'holdout_sequences': 0, 'ohclv_forecast': pd.DataFrame(),
                    'time_attention': None
                }

            # -----------------------------------------------------
            # 从当天目录中强制提取统一的特征列表并对齐
            # -----------------------------------------------------
            selected_features_path = cfg.TODAY_RUN_DIR / 'selected_features_applied.json'
            if not selected_features_path.exists():
                selected_features_path = cfg.SELECTED_FEATURES_PATH
            
            with open(selected_features_path, 'r', encoding='utf-8') as f:
                selected_features_metadata = json.load(f)
            
            if isinstance(selected_features_metadata, dict):
                selected_features = selected_features_metadata.get('selected_features', [])
            else:
                selected_features = selected_features_metadata
        
            expected_num_patterns = model_config.get('num_patterns', len(selected_features))
            if len(selected_features) != expected_num_patterns:
                selected_features = selected_features[:expected_num_patterns]
            
            patterns_train_aligned = patterns_train_aligned[selected_features].copy()
        
            # 特征标准化
            local_scaler = _StandardScaler()
            local_scaler_target = {}
            for horizon in PREDICTION_HORIZONS:
                local_scaler_target[horizon] = _StandardScaler()
    
            X_train_scaled = local_scaler.fit_transform(patterns_train_aligned.values)
    
            y_train_returns_scaled = np.zeros((y_train_raw.shape[0], NUM_HORIZONS))
            for i, horizon in enumerate(PREDICTION_HORIZONS):
                y_train_returns_scaled[:, i:i+1] = local_scaler_target[horizon].fit_transform(y_train_raw[:, i:i+1])
    
            y_train_volatility_log = np.log1p(y_train_raw[:, NUM_HORIZONS:NUM_HORIZONS+1])
            local_scaler_target_volatility = _StandardScaler()
            y_train_volatility_scaled = local_scaler_target_volatility.fit_transform(y_train_volatility_log)
    
            y_train_scaled = np.concatenate([y_train_returns_scaled, y_train_volatility_scaled], axis=1)

            X_seq, y_seq = [], []
            for i in range(len(X_train_scaled) - SEQUENCE_LENGTH):
                X_seq.append(X_train_scaled[i:i+SEQUENCE_LENGTH])
                y_seq.append(y_train_scaled[i+SEQUENCE_LENGTH]) 

            X_train_tensor = torch.FloatTensor(np.array(X_seq)).to(device)
            y_train_tensor = torch.FloatTensor(np.array(y_seq)).to(device)

            num_patterns_selected = patterns_train_aligned.shape[1] 
    
            # 物理隔绝模型
            local_model = DualTowerPatternAwareLSTM(
                num_patterns=num_patterns_selected,
                hidden_size=HIDDEN_SIZE,
                seq_length=SEQUENCE_LENGTH,
                num_layers=NUM_LSTM_LAYERS,
                dropout=DROPOUT,
                num_horizons=NUM_HORIZONS
            ).to(device)
    
            criterion_returns = nn.HuberLoss(delta=1.0)
            criterion_volatility = nn.HuberLoss(delta=1.0)
            criterion_direction = nn.BCELoss()
    
            def combined_loss(pred, target):
                pred = torch.where(torch.isfinite(pred), pred, torch.zeros_like(pred))
                target = torch.where(torch.isfinite(target), target, torch.zeros_like(target))
    
                pred_returns = pred[:, 0:NUM_HORIZONS] 
                pred_volatility = pred[:, NUM_HORIZONS:NUM_HORIZONS+1] 
                target_returns = target[:, 0:NUM_HORIZONS] 
                target_volatility = target[:, NUM_HORIZONS:NUM_HORIZONS+1] 
    
                horizon_weights = getattr(cfg, 'HORIZON_WEIGHTS', [0.70, 0.20, 0.05, 0.03, 0.01, 0.005, 0.005][:NUM_HORIZONS])
                loss_returns = 0
                for i in range(NUM_HORIZONS):
                    loss_returns += horizon_weights[i] * criterion_returns(pred_returns[:, i:i+1], target_returns[:, i:i+1])
                loss_returns = loss_returns / sum(horizon_weights)
    
                loss_volatility = criterion_volatility(pred_volatility, target_volatility)
                target_returns_1d = target_returns[:, 0:1] 
                target_returns_1d = torch.where(torch.isfinite(target_returns_1d), target_returns_1d, torch.zeros_like(target_returns_1d))
                target_direction = (target_returns_1d > 0).float()
                target_direction = torch.clamp(target_direction, min=0.0, max=1.0)
            
                pred_returns_1d = pred_returns[:, 0:1] 
                pred_returns_1d = torch.where(torch.isfinite(pred_returns_1d), pred_returns_1d, torch.zeros_like(pred_returns_1d))
                pred_direction_probs = torch.clamp(torch.sigmoid(pred_returns_1d * 50.0), min=1e-7, max=1-1e-7)
                loss_direction = criterion_direction(pred_direction_probs, target_direction)
    
                total_loss = (LOSS_WEIGHT_RETURNS * loss_returns + 
                              LOSS_WEIGHT_VOLATILITY * loss_volatility + 
                              LOSS_WEIGHT_DIRECTION * loss_direction)
                return total_loss, loss_returns, loss_volatility, loss_direction
    
            optimizer = torch.optim.AdamW(local_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=20, factor=0.5)
            train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)

            # -----------------------------------------------------
            # 执行快速重训
            # -----------------------------------------------------
            print(f"   🔄 正在为场景 [{label}] 执行本地隔离训练...")
        
            split_idx = int(len(X_train_tensor) * 0.8)
            X_tr = X_train_tensor[:split_idx]
            y_tr = y_train_tensor[:split_idx]
            X_va = X_train_tensor[split_idx:]
            y_va = y_train_tensor[split_idx:]
            t_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
            v_loader = DataLoader(TensorDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False)
        
            best_val_loss = float('inf')
            best_model_state = None
            PATIENCE = 25 
            patience_counter = 0
            best_composite_score = float('-inf') 
        
            for epoch in range(SCENARIO_EPOCHS):
                local_model.train()
                for batch_X, batch_y in t_loader:
                    optimizer.zero_grad()
                    outputs, _, _, _ = local_model(batch_X)
                    loss, _, _, _ = combined_loss(outputs, batch_y)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(local_model.parameters(), max_norm=1.0)
                    optimizer.step()
                
                local_model.eval()
                epoch_val_loss = 0
                val_pred_list = []
                val_targ_list = []
                with torch.no_grad():
                    for batch_X, batch_y in v_loader:
                        outputs, _, _, _ = local_model(batch_X)
                        val_loss, _, _, _ = combined_loss(outputs, batch_y)
                        epoch_val_loss += val_loss.item()
                        val_pred_list.append(outputs[:, 0].cpu().numpy())
                        val_targ_list.append(batch_y[:, 0].cpu().numpy())
            
                avg_val_loss = epoch_val_loss / len(v_loader)
                scheduler.step(avg_val_loss)
            
                # 计算早停评分
                try:
                    p_arr = np.concatenate(val_pred_list)
                    t_arr = np.concatenate(val_targ_list)
                    val_r2 = r2_score(t_arr, p_arr)
                    d_acc = np.mean((p_arr > 0) == (t_arr > 0))
                    composite_score = d_acc * 0.75 + max(0, val_r2) * 0.25
                except:
                    composite_score = -avg_val_loss
                
                if composite_score > best_composite_score:
                    best_composite_score = composite_score
                    best_val_loss = avg_val_loss
                    patience_counter = 0
                    best_model_state = local_model.state_dict().copy()
                else:
                    patience_counter += 1
                    if patience_counter >= PATIENCE:
                        break
        
            if best_model_state is not None:
                local_model.load_state_dict(best_model_state)
            local_model.eval()
        
            # -----------------------------------------------------
            # 自回归推演
            # -----------------------------------------------------
            holdout_days = len(holdout_df_real)
            total_sim_days = holdout_days + forecast_horizon_days
        
            current_history = train_df_scenario.copy()
            current_patterns = patterns_train_aligned.copy() 
            current_date_cursor = train_cutoff_date
            last_close = train_df_scenario['Close'].iloc[-1]
            train_feat_mean = patterns_train_aligned.mean(axis=0)

            sim_dates = []
            ohclv_data = []
            top_patterns_list = []
            all_t_attn = []
            bias_correction = 0.0
            holdout_predictions = []
            holdout_actuals = []

            for i in range(total_sim_days):
                is_holdout_period = (i < holdout_days)
                pred_volatility_denorm = 1.0 
        
                if len(current_patterns) >= SEQUENCE_LENGTH:
                    last_seq = current_patterns.iloc[-SEQUENCE_LENGTH:].values
                    last_seq_scaled = local_scaler.transform(last_seq)
                    input_tensor = torch.FloatTensor(last_seq_scaled).unsqueeze(0).to(device)

                    with torch.no_grad():
                        pred_ret, p_attn, t_attn, _ = local_model(input_tensor)
                        pred_returns_scaled = pred_ret[0, 0:NUM_HORIZONS].cpu().numpy() 
                        pred_volatility_scaled = pred_ret[0, NUM_HORIZONS].item() 
                
                        pred_returns_denorm = np.zeros(NUM_HORIZONS)
                        for idx, horizon in enumerate(PREDICTION_HORIZONS):
                            pred_returns_denorm[idx] = local_scaler_target[horizon].inverse_transform([[pred_returns_scaled[idx]]])[0][0]
                
                        pred_volatility_denorm = np.expm1(local_scaler_target_volatility.inverse_transform([[pred_volatility_scaled]])[0][0])
                        pred_volatility_denorm = max(min(pred_volatility_denorm, 10.0), 0.1) 
                
                        pred_returns_daily = np.array([pred_returns_denorm[idx] / PREDICTION_HORIZONS[idx] for idx in range(NUM_HORIZONS)])

                        t_step = i + 1 
                        short_weights = np.array([0.50, 0.30, 0.10, 0.05, 0.03, 0.01, 0.01])[:NUM_HORIZONS]
                        long_weights  = np.array([0.20, 0.15, 0.15, 0.15, 0.15, 0.10, 0.10])[:NUM_HORIZONS]

                        if t_step <= 15:
                            horizon_weights = short_weights
                        else:
                            transition = 1 / (1 + np.exp(-(t_step - 15) / 5)) 
                            horizon_weights = (1 - transition) * short_weights + transition * long_weights

                        horizon_weights_norm = horizon_weights / horizon_weights.sum()
                        pred_pct = float(np.dot(horizon_weights_norm, pred_returns_daily))
                        pred_pct = np.clip(pred_pct, -5.0, 5.0)

                        p_attn_flat = p_attn.mean(dim=1).cpu().numpy().flatten()
                        t_attn_flat = t_attn.squeeze().cpu().numpy()

                        if i == 0:
                            top_indices = np.argsort(p_attn_flat)[-10:][::-1]
                            for idx in top_indices:
                                if idx < len(selected_features):
                                    top_patterns_list.append((selected_features[idx], p_attn_flat[idx]))
                else:
                    pred_pct = 0.0
                    t_attn_flat = np.zeros(SEQUENCE_LENGTH)

                all_t_attn.append(t_attn_flat)
        
                # MC 模拟
                mu = pred_pct
                sigma = pred_volatility_denorm
                random_returns = np.random.normal(loc=mu, scale=sigma, size=100)
                if sigma > 0: random_returns = np.clip(random_returns, mu - 3*sigma, mu + 3*sigma)
                random_returns = np.clip(random_returns, -10.0, 10.0)
            
                simulated_prices = last_close * (1 + random_returns / 100)
            
                if len(current_history) > 0:
                    price_floor = max(0.01, current_history['Close'].min() * 0.5)
                    price_ceiling = current_history['Close'].max() * 1.2
                    simulated_prices = np.clip(simulated_prices, price_floor, price_ceiling)
                
                new_price = np.median(simulated_prices)
        
                if not is_holdout_period and abs(bias_correction) > 0:
                    original_price = new_price
                    new_price -= bias_correction 
                    price_change_pct = abs((new_price - original_price) / original_price) * 100
                    if price_change_pct > 5.0: 
                        actual_correction = np.sign(bias_correction) * min(abs(bias_correction), original_price * 0.05)
                        new_price = original_price - actual_correction
        
                next_date = _trading_day_offset_from_index(current_history.index, current_date_cursor, +1)
                recent_std = current_history['Close'].iloc[-20:].std() if len(current_history) >= 20 else last_close * 0.01
                recent_std = recent_std if not np.isnan(recent_std) else last_close * 0.01

                open_price = new_price * np.random.uniform(0.998, 1.002)
                close_price = new_price
                price_range = abs(open_price - close_price) + recent_std * np.random.uniform(0.5, 1.2)
                high_price = max(open_price, close_price) + price_range * 0.5
                low_price = max(min(open_price, close_price) - price_range * 0.5, close_price * 0.5)

                new_row_data = pd.DataFrame({
                    'Open': [open_price], 'High': [high_price], 'Low': [low_price], 'Close': [close_price],
                    'Volume': [current_history['Volume'].iloc[-20:].mean() if len(current_history) >= 20 else 5000],
                    'Log_Ret': [np.log(close_price / last_close)],
                }, index=[next_date])

                for col in external_cols:
                    if col in current_history.columns:
                        new_row_data[col] = current_history[col].iloc[-1]
                    else:
                        new_row_data[col] = 1.0

                if is_holdout_period:
                    if i < len(holdout_df_real):
                        real_row = holdout_df_real.iloc[i:i+1].copy()
                        real_row.index = [next_date] 
                        ohclv_data.append([next_date, new_row_data['Open'].iloc[0], new_row_data['High'].iloc[0],
                                           new_row_data['Low'].iloc[0], new_price, 
                                           real_row['Volume'].iloc[0] if 'Volume' in real_row.columns else new_row_data['Volume'].iloc[0]])
                
                        holdout_predictions.append(new_price)
                        holdout_actuals.append(real_row['Close'].iloc[0])
                        current_history = pd.concat([current_history, real_row])
                        last_close = real_row['Close'].iloc[0] 
                    else:
                        ohclv_data.append([next_date, new_row_data['Open'].iloc[0], new_row_data['High'].iloc[0],
                                           new_row_data['Low'].iloc[0], new_row_data['Close'].iloc[0], new_row_data['Volume'].iloc[0]])
                        current_history = pd.concat([current_history, new_row_data])
                else:
                    ohclv_data.append([next_date, new_row_data['Open'].iloc[0], new_row_data['High'].iloc[0],
                                       new_row_data['Low'].iloc[0], new_row_data['Close'].iloc[0], new_row_data['Volume'].iloc[0]])
                    current_history = pd.concat([current_history, new_row_data])
            
                current_history.index = pd.to_datetime(current_history.index)
                new_patterns_temp = detect_trading_patterns(current_history, config=cfg)
            
                for feat in selected_features:
                    if feat not in new_patterns_temp.columns:
                        new_patterns_temp[feat] = train_feat_mean.get(feat, 0.0)
                current_patterns = new_patterns_temp[selected_features].copy()

                if i == holdout_days:
                    if len(holdout_predictions) > 0:
                        all_errors = np.array(holdout_actuals) - np.array(holdout_predictions)
                        bias_correction = np.mean(all_errors)
                    elif holdout_days == 0 and len(train_df_scenario) >= 10:
                        recent_train_prices = train_df_scenario['Close'].iloc[-10:].values
                        avg_recent_return = np.mean(np.diff(recent_train_prices) / recent_train_prices[:-1] * 100)
                        bias_correction = train_df_scenario['Close'].iloc[-1] * (avg_recent_return / 100) * 0.5 

                current_date_cursor = next_date
                sim_dates.append(next_date)

            # 打包结果
            ohclv_full_df = pd.DataFrame(ohclv_data, columns=['Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
            forecast_full_df = ohclv_full_df[['Date', 'Close']].copy()

            holdout_comparison = pd.DataFrame() 
            mae_score = np.nan
            residual_stats = {}

            if holdout_days > 0:
                pred_holdout = forecast_full_df.iloc[:holdout_days].copy()
                pred_holdout['Date'] = pd.to_datetime(pred_holdout['Date'])
                actual_holdout = holdout_df_real[['Close']].copy().reset_index()
                actual_holdout.columns = ['Date', 'Actual_Close']
                actual_holdout['Date'] = pd.to_datetime(actual_holdout['Date'])
        
                pred_holdout = pred_holdout.rename(columns={'Close': 'Predicted_Close'})
                holdout_comparison = pd.merge(actual_holdout[['Date', 'Actual_Close']], pred_holdout[['Date', 'Predicted_Close']], on='Date', how='inner')
        
                if len(holdout_comparison) > 0:
                    actual = holdout_comparison['Actual_Close'].values
                    predicted = holdout_comparison['Predicted_Close'].values
                    mae_score = np.mean(np.abs(actual - predicted))
                    residuals = actual - predicted
                    residual_stats['residuals'] = residuals
                    residual_stats['mean_residual'] = np.mean(residuals)
                    residual_stats['std_residual'] = np.std(residuals)
                    if len(residuals) > 3:
                        residual_stats['skewness'] = stats.skew(residuals)
                        residual_stats['kurtosis'] = stats.kurtosis(residuals)
                        try:
                            _, residual_stats['normality_test_pvalue'] = stats.shapiro(residuals)
                        except: pass

            future_forecast = forecast_full_df.iloc[holdout_days:].copy()
            if not future_forecast.empty:
                future_forecast['Date'] = pd.to_datetime(future_forecast['Date'])
                future_forecast_filtered = future_forecast[future_forecast['Date'] > latest_date_pd].copy()
                forecast_15d = future_forecast_filtered.head(comparison_horizon).copy()
            else:
                forecast_15d = future_forecast.head(comparison_horizon)

            mean_t_attn = np.mean(all_t_attn, axis=0) if all_t_attn else None

            return {
                'label': label,
                'train_cutoff': train_cutoff_date,
                'train_sequences': len(X_seq),
                'holdout_sequences': holdout_days,
                'holdout_df': holdout_comparison,
                'forecast_df': future_forecast,
                'forecast_15': forecast_15d,
                'full_sim_df': forecast_full_df,
                'mae': mae_score,
                'pattern_summary': top_patterns_list,
                'time_attention': mean_t_attn,
                'ohclv_forecast': ohclv_full_df.iloc[holdout_days:].head(comparison_horizon),
                'residual_stats': residual_stats 
            }

    # ===================== 执行场景回测 =====================
    scenario_results = OrderedDict()

    cutoff_1m = _trading_day_offset_from_index(df.index, latest_date_pd, -22)
    if cutoff_1m not in df.index:
        valid_dates = df.index[df.index <= cutoff_1m]
        if len(valid_dates) > 0: cutoff_1m = valid_dates[-1]
    scenario_results["保留1个月"] = run_scenario_analysis(cutoff_1m, "保留1个月")

    cutoff_15d = _trading_day_offset_from_index(df.index, latest_date_pd, -15)
    if cutoff_15d not in df.index:
        valid_dates = df.index[df.index <= cutoff_15d]
        if len(valid_dates) > 0: cutoff_15d = valid_dates[-1]
    scenario_results["保留15天"] = run_scenario_analysis(cutoff_15d, "保留15天")

    scenario_results["不保留(全量训练)"] = run_scenario_analysis(latest_date_pd, "不保留(全量训练)")

    print("\n[STEP9] 已执行场景：")
    for scenario_name, scenario_res in scenario_results.items():
        print(f"   - {scenario_name} | train_cutoff={scenario_res['train_cutoff'].date()} | holdout_days={scenario_res['holdout_sequences']}")


    # ============================================================
    # 📊 结果可视化与水印打标
    # ============================================================
    print("\n[PLOTTING SCENARIO COMPARISON]")

    # 提取溯源路径用于水印
    feature_source_path = "Unknown"
    applied_features_path = model_run_dir / 'selected_features_applied.json'
    if applied_features_path.exists():
        with open(applied_features_path, 'r', encoding='utf-8') as f:
            feat_meta = json.load(f)
            feature_source_path = feat_meta.get('source', str(applied_features_path))

    watermark_text = (
        f"Model Traceability Info:\n"
        f"► Run DIR : {cfg.TODAY_RUN_DIR.name}\n"
        f"► Features: {Path(feature_source_path).name}\n"
        f"► Model   : Retrained Isolate Env"
    )

    fig, axes = plt.subplots(3, 2, figsize=(18, 18))
    colors = ['green', 'orange', 'red']

    # 图1：历史价格 vs 预测价格
    ax_hist = axes[0, 0]
    recent_start = latest_date_pd - pd.DateOffset(months=6)
    recent_hist = df[df.index >= recent_start]
    ax_hist.plot(recent_hist.index, recent_hist['Close'], color='black', linewidth=2, label='Actual Close')

    all_data_sim_df = scenario_results["不保留(全量训练)"]['full_sim_df']
    title_suffix = ""
    if not all_data_sim_df.empty:
        title_suffix = f" ({all_data_sim_df['Date'].iloc[0].strftime('%Y-%m-%d')} to {all_data_sim_df['Date'].iloc[-1].strftime('%Y-%m-%d')})"

    for (label, res), color in zip(scenario_results.items(), colors):
        if not res['full_sim_df'].empty:
            sim_df = res['full_sim_df']
            train_cutoff = res['train_cutoff']

            valid_dates = recent_hist.index[recent_hist.index <= train_cutoff]
            if len(valid_dates) == 0:
                continue
            last_hist_date = valid_dates[-1]
            last_hist_price = recent_hist.loc[last_hist_date, 'Close']
        
            sim_df = res['full_sim_df']
            if sim_df.empty:
                continue
            sim_df = sim_df.copy()
            sim_df['Date'] = pd.to_datetime(sim_df['Date'])
            first_forecast_date = pd.to_datetime(sim_df['Date'].iloc[0])
            first_forecast_price = sim_df['Close'].iloc[0]
        
            ax_hist.plot([last_hist_date, first_forecast_date], [last_hist_price, first_forecast_price], color=color, linestyle='--', linewidth=2, alpha=0.7)
            ax_hist.plot(sim_df['Date'], sim_df['Close'], color=color, linestyle='--', linewidth=2, label=f'{label}')
            ax_hist.axvline(res['train_cutoff'], color=color, linestyle=':', alpha=0.6)

    ax_hist.set_title(f'Scenario Analysis: Actual vs Forecast ({STEP9_TICKER_SYMBOL}){title_suffix}', fontsize=12, fontweight='bold')
    ax_hist.legend()
    ax_hist.grid(True, alpha=0.3)

    # 图2：验证集对比
    ax_holdout = axes[0, 1]
    ax_holdout.plot(recent_hist.tail(40).index, recent_hist.tail(40)['Close'], color='black', linewidth=3, alpha=0.7, label='Actual')
    for (label, res), color in zip(scenario_results.items(), colors):
        hold_df = res.get('holdout_df')
        if hold_df is None or hold_df.empty:
            continue
        hold_df = hold_df.copy()
        hold_df['Date'] = pd.to_datetime(hold_df['Date'])
        ax_holdout.plot(pd.to_datetime(res['holdout_df']['Date']), res['holdout_df']['Predicted_Close'], color=color, marker='o', markersize=4, linewidth=2, label=f"{label} (Pred)")
    ax_holdout.set_title(f'Holdout Validation (Recent 40 Days) - {STEP9_TICKER_SYMBOL}', fontsize=12, fontweight='bold')
    ax_holdout.legend()
    ax_holdout.grid(True, alpha=0.3)

    # 图3：MAE误差对比
    ax_mae = axes[1, 0]
    labels_mae = [l for l, r in scenario_results.items() if not np.isnan(r['mae'])]
    maes = [r['mae'] for r in scenario_results.values() if not np.isnan(r['mae'])]
    if maes:
        bars = ax_mae.bar(labels_mae, maes, color=colors[:len(maes)], alpha=0.7, edgecolor='black')
        ax_mae.set_title('Prediction Error (MAE) on Holdout Data', fontsize=12, fontweight='bold')
        for bar, mae in zip(bars, maes):
            ax_mae.text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{mae:.2f}', ha='center', va='bottom')
    else:
        ax_mae.text(0.5, 0.5, "No Valid Holdout Data", ha='center', va='center', transform=ax_mae.transAxes)

    # 图4：未来预测对比带波动率锥
    ax_fut = axes[1, 1]
    reference_date = latest_date_pd
    reference_price = df['Close'].iloc[-1]

    ax_fut.axvline(reference_date, color='gray', linestyle='--', linewidth=2, alpha=0.7, label='Reference Date')
    ax_fut.axhline(reference_price, color='black', linestyle=':', linewidth=1.5, alpha=0.5, label=f'Ref Price: {reference_price:.2f}')

    first_plot = True
    for (label, res), color in zip(scenario_results.items(), colors):
        fut_df = res['forecast_15']
        if not fut_df.empty and 'ohclv_forecast' in res and not res['ohclv_forecast'].empty:
            ohclv_df = res['ohclv_forecast']
            if len(ohclv_df) > 0 and 'High' in ohclv_df.columns and 'Low' in ohclv_df.columns:
                dates = pd.to_datetime(ohclv_df['Date'])
                ax_fut.fill_between(dates, ohclv_df['Low'].values, ohclv_df['High'].values, color=color, alpha=0.15)
            
        if not fut_df.empty:
            plot_dates = [reference_date] + list(pd.to_datetime(fut_df['Date']))
            plot_prices = [reference_price] + list(fut_df['Close'])
            if first_plot:
                ax_fut.scatter([reference_date], [reference_price], color='red', s=150, zorder=10, marker='s', edgecolors='black', label='Ref Point')
                first_plot = False
            ax_fut.plot(plot_dates, plot_prices, color=color, marker='o', linewidth=2, markersize=5, label=f"{label}")

    ax_fut.set_title('Future 15-Day Forecast Comparison (with Cone)', fontsize=12, fontweight='bold')
    ax_fut.legend(loc='best', fontsize=9)
    ax_fut.grid(True, alpha=0.3)

    # 图5：残差直方图
    ax_res_hist = axes[2, 0]
    all_residuals, residual_labels = [], []
    for (label, res), color in zip(scenario_results.items(), colors):
        if res.get('residual_stats') and res['residual_stats'].get('residuals') is not None:
            all_residuals.append(res['residual_stats']['residuals'])
            residual_labels.append(label)

    if all_residuals:
        for residuals, label, color in zip(all_residuals, residual_labels, colors[:len(all_residuals)]):
            ax_res_hist.hist(residuals, bins=20, alpha=0.6, label=label, color=color, edgecolor='black')
        combined_residuals = np.concatenate(all_residuals)
        x_range = np.linspace(combined_residuals.min(), combined_residuals.max(), 100)
        normal_curve = stats.norm.pdf(x_range, np.mean(combined_residuals), np.std(combined_residuals)) * len(combined_residuals) * (x_range[1] - x_range[0])
        ax_res_hist.plot(x_range, normal_curve, 'k--', linewidth=2, label='Normal Dist')
        ax_res_hist.legend()
        ax_res_hist.axvline(x=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
    else:
        ax_res_hist.text(0.5, 0.5, "No Residual Data", ha='center', va='center', transform=ax_res_hist.transAxes)
    ax_res_hist.set_title('Residual Histogram', fontsize=12, fontweight='bold')

    # 图6：Q-Q 图
    ax_qq = axes[2, 1]
    if all_residuals:
        stats.probplot(np.concatenate(all_residuals), dist="norm", plot=ax_qq)
        ax_qq.set_title('Q-Q Plot (Normality test)', fontsize=12, fontweight='bold')
    else:
        ax_qq.text(0.5, 0.5, "No Residual Data", ha='center', va='center', transform=ax_qq.transAxes)
        ax_qq.set_title('Q-Q Plot', fontsize=12, fontweight='bold')

    # 添加底图水印
    fig.suptitle(f'Scenario Backtest | {STEP9_TICKER_SYMBOL}', fontsize=16,fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0.05, 1, 0.99])
    fig.text(0.01, 0.01, watermark_text, ha='left', va='bottom', fontsize=10, color='dimgray', family='monospace')

    scenario_plot_path = model_run_dir / 'Step9_Scenario_Backtesting.png'
    fig.savefig(scenario_plot_path, dpi=300, bbox_inches='tight')
    print(f"✅ 场景回测可视化图已保存至: {scenario_plot_path}")
    plt.show()

    # ============================================================
    # 输出长文摘要 & 自动 TXT 记录 (落盘至 TODAY_RUN_DIR)
    # ============================================================
    summary_lines = []
    def log_print(msg=""):
        print(msg)
        summary_lines.append(msg)

    log_print("\n" + "=" * 80)
    log_print("=== SCENARIO BACKTESTING SUMMARY ===")
    log_print("=" * 80)

    for label, result in scenario_results.items():
        log_print(f"\n📊 [{label}]")
        log_print(f"   Training Cutoff: {result['train_cutoff'].date()}")
        log_print(f"   Train Sequences: {result['train_sequences']}")
        log_print(f"   Holdout Days: {result['holdout_sequences']}")

        if not np.isnan(result['mae']):
            log_print(f"   📈 验证集评估指标:")
            log_print(f"      MAE: {result['mae']:.2f} HKD")
            if 'rmse' in result and not np.isnan(result.get('rmse', np.nan)):
                log_print(f"      RMSE: {result['rmse']:.2f} HKD")
        else:
            log_print(f"   Holdout MAE: N/A (Future Forecast Only)")

    log_print("\n" + "=" * 80)
    log_print("=== FUTURE 15-DAY OHLCV FORECAST COMPARISON ===")
    log_print("=" * 80)

    for label, result in scenario_results.items():
        ohclv_df = result['ohclv_forecast']
        if not ohclv_df.empty:
            log_print(f"\nScenario: [{label}]")
            display_df = ohclv_df.copy()
            display_df['Date'] = pd.to_datetime(display_df['Date']).dt.strftime('%Y-%m-%d')
            display_df['Open'] = display_df['Open'].map('{:.2f}'.format)
            display_df['High'] = display_df['High'].map('{:.2f}'.format)
            display_df['Low'] = display_df['Low'].map('{:.2f}'.format)
            display_df['Close'] = display_df['Close'].map('{:.2f}'.format)
            display_df['Volume'] = display_df['Volume'].map('{:,.0f}'.format)
            log_print(display_df.to_string(index=False))

    log_print("\n" + "=" * 80)
    log_print("=== FUTURE 15-DAY FORECAST COMPARISON (HKD) ===")
    log_print("=" * 80)

    comparison_15 = None
    for label, result in scenario_results.items():
        if not result['forecast_15'].empty:
            df_slice = result['forecast_15'][['Date', 'Close']].copy()
            df_slice['Date'] = pd.to_datetime(df_slice['Date'])
            df_slice = df_slice[df_slice['Date'] > latest_date_pd].head(15).copy()
            if not df_slice.empty:
                df_slice = df_slice.rename(columns={'Close': f'{label} Close'})
                df_slice['Date'] = df_slice['Date'].dt.strftime('%Y-%m-%d')
                if comparison_15 is None: comparison_15 = df_slice
                else: comparison_15 = pd.merge(comparison_15, df_slice, on='Date', how='outer')

    if comparison_15 is not None:
        log_print(comparison_15.to_string(index=False))
    
        log_print("\n" + "=" * 80)
        log_print("=== ENSEMBLE PREDICTION (Weighted Average) ===")
        log_print("=" * 80)
    
        scenario_weights = {}
        scenario_maes = {}
        for label, result in scenario_results.items():
            if not np.isnan(result.get('mae', np.nan)) and result['mae'] > 0:
                scenario_maes[label] = result['mae']
                scenario_weights[label] = 1.0 / result['mae']
            else:
                avg_mae = np.mean([r.get('mae', np.nan) for r in scenario_results.values() if not np.isnan(r.get('mae', np.nan)) and r['mae'] > 0])
                if not np.isnan(avg_mae) and avg_mae > 0: scenario_weights[label] = 1.0 / avg_mae
                else: scenario_weights[label] = 1.0
            
        total_weight = sum(scenario_weights.values())
        if total_weight > 0: scenario_weights = {k: v / total_weight for k, v in scenario_weights.items()}
    
        ensemble_predictions, ensemble_stds = [], []
        for idx, row in comparison_15.iterrows():
            weighted_sum = 0.0
            preds = []
            for label, weight in scenario_weights.items():
                col_name = f'{label} Close'
                if col_name in row and not pd.isna(row[col_name]):
                    weighted_sum += weight * row[col_name]
                    preds.append(row[col_name])
            ensemble_predictions.append(weighted_sum)
            ensemble_stds.append(np.std(preds) if len(preds) > 1 else 0.0)
        
        ensemble_df = pd.DataFrame({
            'Date': comparison_15['Date'],
            'Ensemble_Close': ensemble_predictions,
            'Std_Deviation': ensemble_stds,
            'Lower_Bound_95%': [p - 1.96*s for p, s in zip(ensemble_predictions, ensemble_stds)],
            'Upper_Bound_95%': [p + 1.96*s for p, s in zip(ensemble_predictions, ensemble_stds)]
        })
    
        log_print("\n场景权重（基于验证集MAE）:")
        for label, weight in scenario_weights.items():
            mae = scenario_maes.get(label, 'N/A')
            log_print(f"   {label:20s}: 权重={weight*100:5.2f}% (MAE={mae if mae != 'N/A' else mae})")
        
        log_print("\n集成预测结果（加权平均）:")
        display_ensemble = ensemble_df.copy()
        display_ensemble['Date'] = pd.to_datetime(display_ensemble['Date']).dt.strftime('%Y-%m-%d')
        display_ensemble['Ensemble_Close'] = display_ensemble['Ensemble_Close'].map('{:.2f}'.format)
        display_ensemble['Std_Deviation'] = display_ensemble['Std_Deviation'].map('{:.2f}'.format)
        display_ensemble['Lower_Bound_95%'] = display_ensemble['Lower_Bound_95%'].map('{:.2f}'.format)
        display_ensemble['Upper_Bound_95%'] = display_ensemble['Upper_Bound_95%'].map('{:.2f}'.format)
        log_print(display_ensemble.to_string(index=False))

    log_print("\n" + "=" * 80)

    # 把文本日志保存到当天目录
    summary_txt_path = model_run_dir / 'Step9_Scenario_Backtesting_Summary.txt'
    with open(summary_txt_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(summary_lines))

    print(f"✅ 回测摘要文本已落盘保存至: {summary_txt_path}")

ℹ️ [STEP 9] 当前为 INFERENCE 模式，跳过训练回测与诊断步骤。
